# PINN for plane Poiseuille flow — self-contained

Everything is in this one notebook. **Runtime → Run all.**

Three stages: verify the code, train a model, use the trained model.

* Accuracy is the **relative L2 error against the analytic solution** — the
  19th-century closed form for this flow. That is the `rel L2 vs exact` column.
* The **loss is a diagnostic, never an accuracy claim.** It can fall while the
  answer gets worse, which is why both are shown side by side.
* Everything runs in **float64**. On a free Colab T4 that may be no faster than
  a laptop (a T4's FP64 rate is 1/32 of its FP32); an A100 is genuinely faster.

## 1. Unpack the code and install dependencies

In [ ]:
import json, pathlib

# The project's source files, verbatim from the tested repository.
SOURCES = json.loads(r'''{
"src/__init__.py": "\"\"\"PINN for branching channel flow \u2014 Phase 1 (plane Poiseuille).\n\nIMPORTANT: double precision is enabled here, at package import, *before* any\nJAX array can be created by a submodule. float32 silently caps achievable\nrelative L2 error near 1e-4 and the symptom is indistinguishable from an\noptimiser failure. Do not move, guard or make this conditional.\n\"\"\"\n\nimport jax as _jax\n\n_jax.config.update(\"jax_enable_x64\", True)\n\nimport jax.numpy as _jnp  # noqa: E402  (must follow the x64 switch)\n\nif _jnp.zeros(1).dtype != _jnp.float64:\n    raise RuntimeError(\n        \"float64 is not active: jnp.zeros(1).dtype == \"\n        f\"{_jnp.zeros(1).dtype}. Something imported jax and created an array \"\n        \"before src/__init__.py ran, or JAX_ENABLE_X64 was overridden.\"\n    )\n\n__all__ = [\"assert_float64\"]\n\n\ndef assert_float64(*arrays, where=\"\"):\n    \"\"\"Raise unless every argument is float64.\n\n    Used at module boundaries where a float32 leak would be silent. `where` is\n    a free-text label included in the error message.\n    \"\"\"\n    for i, a in enumerate(arrays):\n        dtype = _jnp.asarray(a).dtype\n        if dtype != _jnp.float64:\n            raise TypeError(\n                f\"float64 required{' in ' + where if where else ''}: \"\n                f\"argument {i} has dtype {dtype}. See CLAUDE.md, \"\n                \"'Non-negotiable numerical rules'.\"\n            )\n",
"src/config.py": "\"\"\"Strict configuration loading.\n\nTwo rules from CLAUDE.md are enforced structurally here rather than by\nconvention:\n\n* \"Every run is fully determined by its config file plus its seed. No\n  hyperparameters hard-coded in source, no defaults that silently apply.\"\n* \"No silent fallbacks. A missing config key raises rather than defaulting.\"\n\nSo every key is required and every unknown key is an error. Keys that only\napply under a particular choice (e.g. `bc.lifting_length` when the inlet is\nhandled by lifting) are required exactly when that choice is active and\nrejected when it is not -- a config carrying a setting that does nothing is a\nconfig that lies about what was run.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport yaml\n\nfrom .physics.conventions import EntranceLengthCorrelation, entrance_length_over_h\nfrom .physics.scaling import PressureScaling\n\n\nclass ConfigError(ValueError):\n    \"\"\"Raised for any missing, unknown, or invalid configuration key.\"\"\"\n\n\ndef _require(mapping: dict, key: str, path: str, kind=None):\n    if not isinstance(mapping, dict):\n        raise ConfigError(f\"{path or '<root>'} must be a mapping, got {type(mapping).__name__}\")\n    if key not in mapping:\n        raise ConfigError(\n            f\"missing required config key {_join(path, key)!r} \"\n            f\"(present keys: {sorted(mapping)})\"\n        )\n    value = mapping.pop(key)\n    if kind is float and isinstance(value, int) and not isinstance(value, bool):\n        value = float(value)\n    if kind is not None and not isinstance(value, kind):\n        raise ConfigError(\n            f\"config key {_join(path, key)!r} must be {kind.__name__}, \"\n            f\"got {type(value).__name__} ({value!r})\"\n        )\n    return value\n\n\ndef _choice(mapping: dict, key: str, path: str, options):\n    value = _require(mapping, key, path, str)\n    if value not in options:\n        raise ConfigError(\n            f\"config key {_join(path, key)!r} must be one of \"\n            f\"{sorted(options)}, got {value!r}\"\n        )\n    return value\n\n\ndef _reject_unused(mapping: dict, key: str, path: str, because: str):\n    if key in mapping:\n        raise ConfigError(\n            f\"config key {_join(path, key)!r} is set but has no effect {because}. \"\n            \"Remove it: a config must describe exactly what was run.\"\n        )\n\n\ndef _finish(mapping: dict, path: str):\n    if mapping:\n        raise ConfigError(\n            f\"unknown config key(s) under {path or '<root>'}: {sorted(mapping)}\"\n        )\n\n\ndef _join(path: str, key: str) -> str:\n    return f\"{path}.{key}\" if path else key\n\n\ndef _positive(value, name: str):\n    if not (value > 0):\n        raise ConfigError(f\"{name} must be positive, got {value!r}\")\n    return value\n\n\n# --------------------------------------------------------------------------- #\n# Schema\n# --------------------------------------------------------------------------- #\n\n\n@dataclass(frozen=True)\nclass DomainConfig:\n    \"\"\"Streamwise extent of the channel.\n\n    SPEC v2 S3: `length` is DERIVED, not configured. The config supplies the\n    development margin `alpha` and the outlet margin, and\n\n        L* = alpha * (L_e / h)(Re_h) + outlet_margin\n\n    is computed at load from the entrance-length correlation. A config may\n    override `length` explicitly, but then it must still satisfy that bound and\n    the loader raises if it does not -- a truncated domain produces a solution\n    that looks converged and is wrong, which is exactly the failure class this\n    project exists to catch, so the check fires before training rather than\n    after.\n    \"\"\"\n\n    length: float\n    alpha: float\n    outlet_margin: float\n    entrance_length_over_h: float\n    explicitly_overridden: bool\n\n\n@dataclass(frozen=True)\nclass PhysicsConfig:\n    reynolds: float\n    pressure_scaling: PressureScaling\n    formulation: str  # velocity_pressure | stream_function\n\n\n@dataclass(frozen=True)\nclass FourierConfig:\n    enabled: bool\n    num_features: int | None\n    bandwidth: float | None\n\n\n@dataclass(frozen=True)\nclass RWFConfig:\n    enabled: bool\n    mean: float | None\n    stddev: float | None\n\n\n@dataclass(frozen=True)\nclass ModelConfig:\n    arch: str  # mlp | modified_mlp\n    width: int\n    depth: int\n    fourier: FourierConfig\n    rwf: RWFConfig\n    lift: str | None  # cubic | sinusoidal, psi_omega only (SPEC v2 S4)\n\n\n@dataclass(frozen=True)\nclass BCConfig:\n    inlet: str  # soft | lifting\n    lifting_length: float | None\n    outlet_velocity: str  # soft\n    pressure_gauge: str | None  # soft | hard; None for psi_omega\n\n\n@dataclass(frozen=True)\nclass SamplingConfig:\n    n_interior: int\n    n_validation: int\n    n_inlet: int\n    n_outlet: int\n    resample_every: int\n    seed: int\n\n\n@dataclass(frozen=True)\nclass GradNormConfig:\n    update_every: int\n    alpha: float\n\n\n@dataclass(frozen=True)\nclass WeightsConfig:\n    scheme: str  # fixed | grad_norm\n    values: Mapping[str, float]\n    grad_norm: GradNormConfig | None\n\n\n@dataclass(frozen=True)\nclass AdamConfig:\n    steps: int\n    lr_init: float\n    decay_rate: float\n    decay_steps: int\n\n\n@dataclass(frozen=True)\nclass LBFGSConfig:\n    enabled: bool\n    max_iterations: int | None\n    gtol: float | None\n\n\n@dataclass(frozen=True)\nclass TrainingConfig:\n    adam: AdamConfig\n    lbfgs: LBFGSConfig\n    select: str  # final | best_validation\n\n\n@dataclass(frozen=True)\nclass EvaluationConfig:\n    nx: int\n    ny: int\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str\n    seed: int\n    case: str  # A | B\n    inlet_uniform_value: float | None  # Case B only\n    domain: DomainConfig\n    correlation: EntranceLengthCorrelation\n    physics: PhysicsConfig\n    model: ModelConfig\n    bc: BCConfig\n    sampling: SamplingConfig\n    weights: WeightsConfig\n    training: TrainingConfig\n    evaluation: EvaluationConfig\n    raw: Mapping[str, Any] = field(repr=False, default_factory=dict)\n\n    # -- derived, never configurable -----------------------------------------\n\n    @property\n    def inlet_flux(self) -> float:\n        \"\"\"Non-dimensional flux entering at x* = 0.\n\n        Case A: int (1 - y^2) dy = 4/3 over y in [-1, 1].\n        Case B: uniform u* = U over the full height 2, so flux = 2U.\n        \"\"\"\n        if self.case == \"A\":\n            return 4.0 / 3.0\n        return 2.0 * self.inlet_uniform_value\n\n    @property\n    def loss_terms(self) -> tuple[str, ...]:\n        \"\"\"Exactly the residual/BC terms this configuration produces.\n\n        Walls never appear: no-slip is imposed by construction in the ansatz,\n        so there is no wall loss term to weight (SPEC section 3).\n        \"\"\"\n        if self.physics.formulation == \"psi_omega\":\n            # SPEC v2 S4: two coupled second-order residuals. Continuity is\n            # identical, and pressure is not in the system at all.\n            terms = [\"vorticity\", \"vorticity_transport\"]\n        else:\n            terms = [\"momentum_x\", \"momentum_y\", \"continuity\"]\n        if self.bc.inlet == \"soft\":\n            terms.append(\"inlet\")\n        if self.bc.outlet_velocity == \"soft\":\n            terms.append(\"outlet_velocity\")\n        if self.bc.pressure_gauge == \"soft\":\n            terms.append(\"pressure_gauge\")\n        return tuple(terms)\n\n\n# --------------------------------------------------------------------------- #\n# Parsing\n# --------------------------------------------------------------------------- #\n\n\ndef parse_config(document: Mapping[str, Any]) -> Config:\n    \"\"\"Validate a already-loaded YAML mapping into a `Config`.\"\"\"\n    import copy\n\n    raw = copy.deepcopy(dict(document))\n    doc = copy.deepcopy(dict(document))\n\n    name = _require(doc, \"name\", \"\", str)\n    seed = _require(doc, \"seed\", \"\", int)\n    case = _choice(doc, \"case\", \"\", {\"A\", \"B\"})\n\n    # -- correlation (parsed before domain: S3 derives L* from it) --\n    corr = _require(doc, \"correlation\", \"\", dict)\n    if \"citation\" not in corr:\n        raise ConfigError(\n            \"correlation.citation is required (SPEC v2 S1). The entrance-length \"\n            \"constants differ between pipe and plane-channel geometry and must \"\n            \"be read from the source; a wrong constant is invisible in the \"\n            \"output, a missing citation is not.\"\n        )\n    correlation = EntranceLengthCorrelation(\n        c0=_require(corr, \"c0\", \"correlation\", float),\n        c1=_require(corr, \"c1\", \"correlation\", float),\n        n=_require(corr, \"n\", \"correlation\", float),\n        citation=_require(corr, \"citation\", \"correlation\", str),\n    )\n    _finish(corr, \"correlation\")\n\n    # -- physics (parsed before domain: L* depends on Re_h) --\n    p = _require(doc, \"physics\", \"\", dict)\n    reynolds = _positive(_require(p, \"reynolds\", \"physics\", float), \"physics.reynolds\")\n    if reynolds > 200.0:\n        raise ConfigError(\n            f\"physics.reynolds = {reynolds} exceeds the Phase 1 ceiling of 200 \"\n            \"(SPEC section 2: 'Do not attempt Re > 200 in Phase 1').\"\n        )\n    pressure_scaling = _choice(p, \"pressure_scaling\", \"physics\", {\"viscous\", \"inertial\"})\n    formulation = _choice(\n        p, \"formulation\", \"physics\", {\"velocity_pressure\", \"psi_omega\"}\n    )\n    if case == \"B\":\n        inlet_uniform_value = _positive(\n            _require(p, \"inlet_uniform_value\", \"physics\", float),\n            \"physics.inlet_uniform_value\",\n        )\n    else:\n        _reject_unused(\n            p, \"inlet_uniform_value\", \"physics\", \"for case A (its inlet is parabolic)\"\n        )\n        inlet_uniform_value = None\n    physics = PhysicsConfig(reynolds, pressure_scaling, formulation)\n    _finish(p, \"physics\")\n\n    # -- domain (S3: L* derived from Re_h and the correlation) --\n    d = _require(doc, \"domain\", \"\", dict)\n    alpha = _positive(_require(d, \"alpha\", \"domain\", float), \"domain.alpha\")\n    outlet_margin = _positive(\n        _require(d, \"outlet_margin\", \"domain\", float), \"domain.outlet_margin\"\n    )\n    if outlet_margin < 5.0:\n        raise ConfigError(\n            f\"domain.outlet_margin must be at least 5 (SPEC v2 S3), got \"\n            f\"{outlet_margin}. The margin exists so the zero-gradient outflow \"\n            \"condition is applied where the flow is already developed.\"\n        )\n    le_over_h = float(entrance_length_over_h(reynolds, correlation))\n    required_length = alpha * le_over_h + outlet_margin\n    if \"length\" in d:\n        length = _positive(_require(d, \"length\", \"domain\", float), \"domain.length\")\n        if length < required_length:\n            raise ConfigError(\n                f\"domain.length = {length:g} is below the S3 bound \"\n                f\"{required_length:g} = alpha ({alpha:g}) * L_e/h ({le_over_h:.4f}) \"\n                f\"+ outlet_margin ({outlet_margin:g}) at Re_h = {reynolds:g}. \"\n                \"A truncated domain produces a solution that looks converged \"\n                \"and is wrong.\"\n            )\n        overridden = True\n    else:\n        length = required_length\n        overridden = False\n    domain = DomainConfig(\n        length=length,\n        alpha=alpha,\n        outlet_margin=outlet_margin,\n        entrance_length_over_h=le_over_h,\n        explicitly_overridden=overridden,\n    )\n    _finish(d, \"domain\")\n\n    # -- model --\n    m = _require(doc, \"model\", \"\", dict)\n    arch = _choice(m, \"arch\", \"model\", {\"mlp\", \"modified_mlp\"})\n    width = _positive(_require(m, \"width\", \"model\", int), \"model.width\")\n    depth = _positive(_require(m, \"depth\", \"model\", int), \"model.depth\")\n\n    f = _require(m, \"fourier\", \"model\", dict)\n    fourier_enabled = _require(f, \"enabled\", \"model.fourier\", bool)\n    if fourier_enabled:\n        fourier = FourierConfig(\n            True,\n            _positive(_require(f, \"num_features\", \"model.fourier\", int), \"num_features\"),\n            _positive(_require(f, \"bandwidth\", \"model.fourier\", float), \"bandwidth\"),\n        )\n    else:\n        _reject_unused(f, \"num_features\", \"model.fourier\", \"when fourier.enabled is false\")\n        _reject_unused(f, \"bandwidth\", \"model.fourier\", \"when fourier.enabled is false\")\n        fourier = FourierConfig(False, None, None)\n    _finish(f, \"model.fourier\")\n\n    r = _require(m, \"rwf\", \"model\", dict)\n    rwf_enabled = _require(r, \"enabled\", \"model.rwf\", bool)\n    if rwf_enabled:\n        rwf = RWFConfig(\n            True,\n            _require(r, \"mean\", \"model.rwf\", float),\n            _positive(_require(r, \"stddev\", \"model.rwf\", float), \"model.rwf.stddev\"),\n        )\n    else:\n        _reject_unused(r, \"mean\", \"model.rwf\", \"when rwf.enabled is false\")\n        _reject_unused(r, \"stddev\", \"model.rwf\", \"when rwf.enabled is false\")\n        rwf = RWFConfig(False, None, None)\n    _finish(r, \"model.rwf\")\n\n    if formulation == \"psi_omega\":\n        lift = _choice(m, \"lift\", \"model\", {\"cubic\", \"sinusoidal\"})\n    else:\n        _reject_unused(m, \"lift\", \"model\", \"for the velocity_pressure formulation\")\n        lift = None\n    model = ModelConfig(arch, width, depth, fourier, rwf, lift)\n    _finish(m, \"model\")\n\n    # -- bc --\n    b = _require(doc, \"bc\", \"\", dict)\n    inlet = _choice(b, \"inlet\", \"bc\", {\"soft\", \"lifting\"})\n    if inlet == \"lifting\":\n        if case == \"B\":\n            raise ConfigError(\n                \"bc.inlet = 'lifting' is not available for case B: a uniform \"\n                \"inlet u* = 1 is incompatible with the hard no-slip wall \"\n                \"ansatz at the corners (y* = +-1, x* = 0). Use 'soft'.\"\n            )\n        lifting_length = _positive(\n            _require(b, \"lifting_length\", \"bc\", float), \"bc.lifting_length\"\n        )\n    else:\n        _reject_unused(b, \"lifting_length\", \"bc\", \"when bc.inlet is 'soft'\")\n        lifting_length = None\n    outlet_velocity = _choice(b, \"outlet_velocity\", \"bc\", {\"soft\"})\n    if formulation == \"psi_omega\":\n        # Pressure is absent from the psi-omega system, so there is no pressure\n        # ansatz to choose. Recovery is post hoc from a gauge point fixed at the\n        # outlet centreline -- that is the definition of the gauge, not a tunable.\n        _reject_unused(\n            b, \"pressure_gauge\", \"bc\",\n            \"for the psi_omega formulation, which has no pressure in its system \"\n            \"(recovery is post hoc from a gauge point at the outlet centreline)\",\n        )\n        pressure_gauge = None\n    else:\n        pressure_gauge = _choice(b, \"pressure_gauge\", \"bc\", {\"soft\", \"hard\"})\n    bc = BCConfig(inlet, lifting_length, outlet_velocity, pressure_gauge)\n    _finish(b, \"bc\")\n\n    # -- sampling --\n    s = _require(doc, \"sampling\", \"\", dict)\n    sampling = SamplingConfig(\n        n_interior=_positive(_require(s, \"n_interior\", \"sampling\", int), \"n_interior\"),\n        # Held-out collocation points, never trained on. L-BFGS runs on a FROZEN\n        # point set, so it can drive the residual down on those specific points\n        # while the true error rises. Detecting that needs points the optimiser\n        # has never seen -- and unlike a comparison against the analytic\n        # solution, this still works in Phase 2 where no analytic solution\n        # exists.\n        n_validation=_positive(\n            _require(s, \"n_validation\", \"sampling\", int), \"n_validation\"\n        ),\n        n_inlet=_positive(_require(s, \"n_inlet\", \"sampling\", int), \"n_inlet\"),\n        n_outlet=_positive(_require(s, \"n_outlet\", \"sampling\", int), \"n_outlet\"),\n        resample_every=_positive(\n            _require(s, \"resample_every\", \"sampling\", int), \"resample_every\"\n        ),\n        seed=_require(s, \"seed\", \"sampling\", int),\n    )\n    _finish(s, \"sampling\")\n\n    # -- training --\n    t = _require(doc, \"training\", \"\", dict)\n    a = _require(t, \"adam\", \"training\", dict)\n    adam = AdamConfig(\n        steps=_require(a, \"steps\", \"training.adam\", int),\n        lr_init=_positive(_require(a, \"lr_init\", \"training.adam\", float), \"lr_init\"),\n        decay_rate=_positive(\n            _require(a, \"decay_rate\", \"training.adam\", float), \"decay_rate\"\n        ),\n        decay_steps=_positive(\n            _require(a, \"decay_steps\", \"training.adam\", int), \"decay_steps\"\n        ),\n    )\n    if adam.steps < 0:\n        raise ConfigError(\"training.adam.steps must be >= 0\")\n    _finish(a, \"training.adam\")\n\n    lb = _require(t, \"lbfgs\", \"training\", dict)\n    lbfgs_enabled = _require(lb, \"enabled\", \"training.lbfgs\", bool)\n    if lbfgs_enabled:\n        lbfgs = LBFGSConfig(\n            True,\n            _positive(\n                _require(lb, \"max_iterations\", \"training.lbfgs\", int), \"max_iterations\"\n            ),\n            _positive(_require(lb, \"gtol\", \"training.lbfgs\", float), \"gtol\"),\n        )\n    else:\n        _reject_unused(lb, \"max_iterations\", \"training.lbfgs\", \"when lbfgs is disabled\")\n        _reject_unused(lb, \"gtol\", \"training.lbfgs\", \"when lbfgs is disabled\")\n        lbfgs = LBFGSConfig(False, None, None)\n    _finish(lb, \"training.lbfgs\")\n    select = _choice(t, \"select\", \"training\", {\"final\", \"best_validation\"})\n    training = TrainingConfig(adam, lbfgs, select)\n    _finish(t, \"training\")\n\n    # -- evaluation --\n    e = _require(doc, \"evaluation\", \"\", dict)\n    evaluation = EvaluationConfig(\n        nx=_positive(_require(e, \"nx\", \"evaluation\", int), \"evaluation.nx\"),\n        ny=_positive(_require(e, \"ny\", \"evaluation\", int), \"evaluation.ny\"),\n    )\n    _finish(e, \"evaluation\")\n\n    # -- weights (validated last: the active term list depends on everything\n    #    above, and the config must supply a weight for exactly those terms) --\n    w = _require(doc, \"weights\", \"\", dict)\n    scheme = _choice(w, \"scheme\", \"weights\", {\"fixed\", \"grad_norm\"})\n    values = _require(w, \"values\", \"weights\", dict)\n    if scheme == \"grad_norm\":\n        g = _require(w, \"grad_norm\", \"weights\", dict)\n        grad_norm = GradNormConfig(\n            update_every=_positive(\n                _require(g, \"update_every\", \"weights.grad_norm\", int), \"update_every\"\n            ),\n            alpha=_require(g, \"alpha\", \"weights.grad_norm\", float),\n        )\n        if not 0.0 <= grad_norm.alpha <= 1.0:\n            raise ConfigError(\"weights.grad_norm.alpha must lie in [0, 1]\")\n        _finish(g, \"weights.grad_norm\")\n    else:\n        _reject_unused(w, \"grad_norm\", \"weights\", \"when weights.scheme is 'fixed'\")\n        grad_norm = None\n    weights = WeightsConfig(scheme, dict(values), grad_norm)\n    _finish(w, \"weights\")\n\n    _finish(doc, \"\")\n\n    config = Config(\n        name=name,\n        seed=seed,\n        case=case,\n        inlet_uniform_value=inlet_uniform_value,\n        domain=domain,\n        correlation=correlation,\n        physics=physics,\n        model=model,\n        bc=bc,\n        sampling=sampling,\n        weights=weights,\n        training=training,\n        evaluation=evaluation,\n        raw=raw,\n    )\n\n    expected_terms = set(config.loss_terms)\n    given_terms = set(values)\n    if given_terms != expected_terms:\n        missing = sorted(expected_terms - given_terms)\n        extra = sorted(given_terms - expected_terms)\n        raise ConfigError(\n            \"weights.values must name exactly the active loss terms \"\n            f\"{sorted(expected_terms)}.\"\n            + (f\" Missing: {missing}.\" if missing else \"\")\n            + (f\" Not active for this configuration: {extra}.\" if extra else \"\")\n        )\n    for term, value in values.items():\n        if not isinstance(value, (int, float)) or isinstance(value, bool):\n            raise ConfigError(f\"weights.values.{term} must be a number, got {value!r}\")\n        if value < 0:\n            raise ConfigError(f\"weights.values.{term} must be >= 0, got {value!r}\")\n\n    return config\n\n\ndef load_config(path: str | Path) -> Config:\n    \"\"\"Load and validate a YAML experiment file.\"\"\"\n    path = Path(path)\n    if not path.is_file():\n        raise ConfigError(f\"config file not found: {path}\")\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        document = yaml.safe_load(handle)\n    if not isinstance(document, dict):\n        raise ConfigError(f\"{path}: top level of a config must be a mapping\")\n    try:\n        return parse_config(document)\n    except ConfigError as error:\n        raise ConfigError(f\"{path}: {error}\") from None\n",
"src/evaluation/__init__.py": "\"\"\"Accuracy metrics, conservation checks, residual maps and plots.\"\"\"\n",
"src/evaluation/accuracy.py": "\"\"\"Case-specific accuracy evaluation on the fixed evaluation grid.\n\nNever on the training points (SPEC section 9). The grid is rebuilt from the\nconfig, so the same config always evaluates on the same points regardless of\nwhat the sampler did.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax.numpy as jnp\nimport numpy as np\n\nfrom ..config import Config\nfrom ..geometry.domain import Channel\nfrom ..geometry.sampling import evaluation_grid\nfrom ..physics import analytic\nfrom ..physics.conventions import entrance_length_over_h\nfrom ..physics.derivatives import batched_field_derivatives\nfrom ..physics import residuals, streamfunction\nfrom ..physics.derivatives import batched_stream_derivatives\nfrom .drift import mass_drift\nfrom .metrics import (\n    centreline_velocity,\n    cross_channel_flux,\n    entrance_length,\n    evaluate_fields,\n    flux_conservation_error,\n    max_abs,\n    relative_l2,\n)\n\n#: SPEC test 3 asks for ten streamwise stations.\nN_FLUX_STATIONS = 10\n\n#: Stations are trimmed 5% away from the inlet and outlet planes so that a\n#: boundary-condition error is not counted as a mass-conservation error.\nFLUX_STATION_MARGIN = 0.05\n\n\ndef _residual_norms(config, model, params, points):\n    \"\"\"Max and RMS of each unweighted residual on the evaluation grid.\n\n    Reported as a DIAGNOSTIC. A small residual is not an accuracy claim: the\n    failure mode this project exists to catch is exactly a low residual with a\n    wrong solution.\n    \"\"\"\n    out = {}\n    if config.physics.formulation == \"psi_omega\":\n        sd = batched_stream_derivatives(lambda xy: model.stream(params, xy), points)\n        names = (\"vorticity\", \"vorticity_transport\")\n        values = streamfunction.residuals(sd, config.physics.reynolds)\n    else:\n        d = batched_field_derivatives(lambda xy: model.field(params, xy), points)\n        names = (\"momentum_x\", \"momentum_y\", \"continuity\")\n        values = residuals.navier_stokes(\n            d, config.physics.reynolds, config.physics.pressure_scaling\n        )\n    for name, value in zip(names, values):\n        value = np.asarray(value)\n        out[f\"residual_{name}_max\"] = float(np.max(np.abs(value)))\n        out[f\"residual_{name}_rms\"] = float(np.sqrt(np.mean(value**2)))\n    return out\n\n\ndef _flux_metrics(config, model, params):\n    length = config.domain.length\n    stations = np.linspace(\n        FLUX_STATION_MARGIN * length,\n        (1.0 - FLUX_STATION_MARGIN) * length,\n        N_FLUX_STATIONS,\n    )\n    fluxes = cross_channel_flux(model.field, params, stations)\n    return {\n        \"flux_stations\": stations.tolist(),\n        \"flux_values\": np.asarray(fluxes).tolist(),\n        \"flux_mean\": float(np.mean(fluxes)),\n        # SPEC v2 S7 defines D against the FIRST station, so a monotonic\n        # downstream drift shows at full size rather than half of it. The\n        # spread-about-the-mean figure is kept alongside for comparison with\n        # the SPEC v1 numbers.\n        \"flux_conservation_relative_error\": mass_drift(fluxes),\n        \"flux_spread_about_mean\": flux_conservation_error(fluxes),\n        \"flux_relative_error_vs_inlet\": float(\n            abs(np.mean(fluxes) - config.inlet_flux) / config.inlet_flux\n        ),\n    }\n\n\ndef evaluate(config: Config, model, params) -> dict:\n    \"\"\"All reported metrics for a finished run.\"\"\"\n    channel = Channel(config.domain.length)\n    points, x, y = evaluation_grid(channel, config.evaluation.nx, config.evaluation.ny)\n\n    u, v, p = evaluate_fields(model.field, params, points)\n\n    metrics = {\n        \"case\": config.case,\n        \"reynolds\": config.physics.reynolds,\n        \"pressure_scaling\": config.physics.pressure_scaling,\n        \"formulation\": config.physics.formulation,\n        \"evaluation_points\": int(points.shape[0]),\n        \"v_max_abs\": max_abs(v),\n        \"u_max\": float(np.max(u)),\n    }\n    metrics[\"has_pressure_in_system\"] = bool(model.has_pressure)\n    if model.lift is not None:\n        metrics[\"lift\"] = model.lift\n    metrics.update(_flux_metrics(config, model, params))\n    metrics.update(_residual_norms(config, model, params, points))\n\n    if not model.has_pressure:\n        # SPEC v2 S4: pressure is absent from the psi-omega system and is\n        # recovered post hoc by integrating grad p from the outlet gauge point.\n        # Path independence is asserted, not assumed: grad p is only a gradient\n        # if the solution actually satisfies momentum.\n        metrics.update(\n            _recovered_pressure_metrics(config, model, params, np.asarray(y))\n        )\n\n    grid_y = np.asarray(points[:, 1])\n    grid_x = np.asarray(points[:, 0])\n\n    if config.case == \"A\":\n        # The exact solution is known everywhere.\n        u_exact = analytic.u_nd(grid_y)\n        p_exact = analytic.p_nd(\n            grid_x,\n            config.domain.length,\n            config.physics.reynolds,\n            config.physics.pressure_scaling,\n        )\n        metrics[\"u_relative_l2\"] = relative_l2(u, u_exact)\n        metrics[\"u_max_abs_error\"] = max_abs(u - u_exact)\n        if model.has_pressure:\n            metrics[\"p_relative_l2\"] = relative_l2(p, p_exact)\n            metrics[\"p_max_abs_error\"] = max_abs(p - p_exact)\n    else:\n        # No closed-form solution. Two checks apply (SPEC section 2, Case B):\n        # the downstream profile, and the entrance length.\n        #\n        # A uniform inlet u* = 1 is DISCONTINUOUS at the corners against hard\n        # no-slip walls: the ansatz u = phi(y) N_u would need N_u = 1/(1-y^2)\n        # at the inlet, which is unbounded. The network therefore delivers\n        # slightly less than the nominal flux, and that deficit is an inlet\n        # boundary-condition artefact, not a solver error.\n        #\n        # So the developed profile is compared two ways, and both are reported:\n        #   * against the NOMINAL flux from the config -- includes the deficit;\n        #   * against the MEASURED flux -- a pure shape test, which is the\n        #     question Phase 2 actually cares about (\"does the flow redevelop\n        #     into the right profile?\"). Mass conservation is checked separately\n        #     and independently by the flux metrics above.\n        measured_flux = float(metrics[\"flux_mean\"])\n        nominal = analytic.developed_profile_nd(np.asarray(y), config.inlet_flux)\n        shaped = analytic.developed_profile_nd(np.asarray(y), measured_flux)\n\n        outlet_points = jnp.stack(\n            [jnp.full_like(y, config.domain.length), y], axis=-1\n        )\n        u_outlet, _, _ = evaluate_fields(model.field, params, outlet_points)\n        metrics[\"outlet_profile_relative_l2_vs_nominal_flux\"] = relative_l2(\n            u_outlet, nominal\n        )\n        metrics[\"outlet_profile_relative_l2\"] = relative_l2(u_outlet, shaped)\n        metrics[\"developed_centreline_nominal\"] = float(0.75 * config.inlet_flux)\n        metrics[\"developed_centreline_measured\"] = float(0.75 * measured_flux)\n        metrics[\"inlet_flux_deficit_relative\"] = float(\n            (config.inlet_flux - measured_flux) / config.inlet_flux\n        )\n\n        stations = np.linspace(0.0, config.domain.length, 2001)\n        centreline = centreline_velocity(model.field, params, stations)\n        # The 99% threshold is against the flux the solution actually carries.\n        measured = entrance_length(stations, centreline, 0.75 * measured_flux)\n        # SPEC v2 S1: the correlation consumes Re_Dh, not Re_h, and returns a\n        # length in D_h. `entrance_length_over_h` converts both internally, so\n        # this call site cannot get the factor of 8/3 or the factor of 4 wrong.\n        correlation = float(\n            entrance_length_over_h(config.physics.reynolds, config.correlation)\n        )\n        metrics[\"entrance_length_measured\"] = measured\n        metrics[\"entrance_length_correlation\"] = correlation\n        metrics[\"entrance_length_relative_error\"] = (\n            float(abs(measured - correlation) / correlation)\n            if np.isfinite(measured)\n            else float(\"nan\")\n        )\n        # T-S2.2: the coupling between profile error and mass drift stays\n        # visible in every run. A ratio near 1 means the profile error IS the\n        # mass drift wearing an amplitude disguise. Undefined once the drift is\n        # at machine zero (stream function), where it is reported as such.\n        drift = metrics[\"flux_conservation_relative_error\"]\n        metrics[\"profile_error_over_mass_error\"] = (\n            float(metrics[\"outlet_profile_relative_l2\"] / drift)\n            if drift > 1e-12\n            else None\n        )\n        metrics[\"centreline_stations\"] = stations[::100].tolist()\n        metrics[\"centreline_values\"] = np.asarray(centreline)[::100].tolist()\n\n    return metrics\n\n\ndef _recovered_pressure_metrics(config, model, params, y_grid) -> dict:\n    \"\"\"Post-hoc pressure for the psi-omega system, plus the path-independence check.\n\n    Sampled on a modest set of probe points rather than the full evaluation\n    grid: each probe costs a path integral whose integrand needs second\n    derivatives of velocity, i.e. third derivatives of psi.\n    \"\"\"\n    velocity_fn = lambda xy: model.velocity(params, xy)  # noqa: E731\n    length = config.domain.length\n    gauge = (length, 0.0)\n\n    probes = [\n        (x, yy)\n        for x in np.linspace(0.05 * length, 0.95 * length, 7)\n        for yy in (-0.6, 0.0, 0.6)\n    ]\n\n    x_first, y_first = [], []\n    for probe in probes:\n        x_first.append(\n            streamfunction.recover_pressure(\n                velocity_fn, probe, gauge, config.physics.reynolds,\n                config.physics.pressure_scaling, path=\"x_first\",\n            )\n        )\n        y_first.append(\n            streamfunction.recover_pressure(\n                velocity_fn, probe, gauge, config.physics.reynolds,\n                config.physics.pressure_scaling, path=\"y_first\",\n            )\n        )\n    x_first = np.asarray(x_first)\n    y_first = np.asarray(y_first)\n\n    out = {\n        \"pressure_path_max_disagreement\": float(np.max(np.abs(x_first - y_first))),\n        \"pressure_recovery_probes\": len(probes),\n    }\n    if config.case == \"A\":\n        exact = analytic.p_nd(\n            np.array([p[0] for p in probes]), length,\n            config.physics.reynolds, config.physics.pressure_scaling,\n        )\n        out[\"p_relative_l2_recovered\"] = relative_l2(x_first, exact)\n    return out\n",
"src/evaluation/drift.py": "\"\"\"Mass-drift metric and its convergence fit (SPEC v2, S7).\n\nWith `M(x*) = integral u* dy*` on a FIXED evaluation grid at `K >= 10` stations:\n\n    D = max_x | M(x*) - M(0) | / M(0)\n\nTest 3 asserts `D <= 1e-6` and the tolerance stays. Test 1 measures a pointwise\nfield error; test 3 measures the variation of an integral, and integration\nnormally averages errors down, so a tighter bound on the integral is the\ncorrect default. `D` exceeding the pointwise error means the error is\nSYSTEMATIC AND ACCUMULATING DOWNSTREAM rather than cancelling -- which is the\nfinding, and relaxing the bound would erase it.\n\nThe discriminating measurement is the trace of `D` against L-BFGS iteration,\nfitted to\n\n    D(n) = D_inf + a n^-b\n\n* `D_inf` significantly above 1e-6 -> formulation bias. Report it with a\n  confidence interval and close the test as a characterised limitation.\n* `D_inf` consistent with zero -> under-convergence, and a longer run clears it.\n\nEither outcome is a result.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\ndef mass_drift(fluxes) -> float:\n    \"\"\"D, referenced to the FIRST station as SPEC v2 S7 defines it.\n\n    Note this differs from a spread-about-the-mean measure: referencing the\n    inlet station makes a monotonic downstream drift show at its full size\n    rather than half of it.\n    \"\"\"\n    fluxes = np.asarray(fluxes, dtype=np.float64)\n    if fluxes.size < 10:\n        raise ValueError(\n            f\"SPEC v2 S7 requires at least 10 stations, got {fluxes.size}\"\n        )\n    reference = float(fluxes[0])\n    if reference == 0.0:\n        raise ValueError(\"flux at the first station is zero; D is undefined\")\n    return float(np.max(np.abs(fluxes - reference)) / abs(reference))\n\n\n@dataclass(frozen=True)\nclass DriftFit:\n    \"\"\"Result of fitting D(n) = D_inf + a n^-b.\"\"\"\n\n    d_infinity: float\n    amplitude: float\n    exponent: float\n    d_infinity_stderr: float\n    points: int\n    converged: bool\n    message: str\n    at_machine_precision: bool = False\n    tail_at_last_point: float = float(\"nan\")\n    last_iteration: float = float(\"nan\")\n    decay_range: float = float(\"nan\")\n    max_relative_residual: float = float(\"nan\")\n    moved_off_guess: bool = True\n\n    #: Largest allowed |model - D| / D across the trace. A power law fitted to\n    #: a noisy tail does not track it tightly -- the real caseA traces reach\n    #: 0.85 and 1.15 with genuine fits -- so this is a gross-failure detector,\n    #: not a quality bar. The degenerate fits it exists to catch sit at 5.8,\n    #: 14.5 and 6e5, so a cut at 2.0 separates them with room on both sides.\n    MAXIMUM_RELATIVE_RESIDUAL = 2.0\n\n    #: The fitted transient must decay by at least this factor across the\n    #: traced range for D_inf to be a measurement rather than an extrapolation.\n    #: Recalibrated on synthetic traces carrying 10% multiplicative scatter,\n    #: since identifiability is a property of noisy data (the earlier noiseless\n    #: calibration was measuring the unweighted fitter's own error, and became\n    #: meaningless once `fit_drift` weighted its residuals relatively). Median\n    #: relative error in D_inf over 20 seeds, at D_inf = 1e-5: decay 4.5 is\n    #: 801% out and 7.1 is 218% out, against 33% at decay 29 and 1.6% at 197.\n    #: One decade is the conservative cut between them. Note decay range alone\n    #: is not sufficient -- at decay 29 with D_inf = 1e-7 the error is still\n    #: 1040% -- which is what `resolves` catches downstream.\n    MINIMUM_DECAY_RANGE = 10.0\n\n    @property\n    def identifiable(self) -> bool:\n        \"\"\"Whether the trace actually pins D_inf down.\n\n        D_inf is the asymptote of D(n) = D_inf + a n^-b. What determines\n        whether it can be measured is not how big the tail is, but how much the\n        tail DECAYED across the traced range: without that leverage the fit\n        cannot separate the constant from the transient.\n\n        The fitted standard error is not usable for this. On synthetic traces\n        it underestimates the true error by roughly an order of magnitude\n        (0.023 reported against 0.33 actual, 0.089 against 25), which is\n        typical for a weakly constrained parameter in a nonlinear fit. The\n        decay range separates the same cases cleanly.\n        \"\"\"\n        if not self.converged or not np.isfinite(self.decay_range):\n            return False\n        return self.decay_range >= self.MINIMUM_DECAY_RANGE\n\n    @property\n    def tracks_data(self) -> bool:\n        \"\"\"Whether the fitted curve actually follows the trace.\n\n        `curve_fit` reporting success is not evidence that a fit happened. On\n        an unscaled least-squares problem whose residuals are O(1e-7), the cost\n        at the initial guess is already below the default `ftol`, so the\n        optimiser returns `p0` untouched and `converged=True`. That produced\n        \"D_inf = 1.885e-07 +- 9.1e-07, consistent with zero\" for the 30000\n        iteration run -- a verdict with the right conclusion drawn from a\n        model that missed its own final data point by 9x.\n\n        Relative weighting in `fit_drift` removes the cause; this checks the\n        symptom directly, and catches any other way the fit can fail to\n        describe the data.\n        \"\"\"\n        if not self.converged or not np.isfinite(self.max_relative_residual):\n            return False\n        if not self.moved_off_guess:\n            return False\n        return self.max_relative_residual <= self.MAXIMUM_RELATIVE_RESIDUAL\n\n    @property\n    def consistent_with_zero(self) -> bool:\n        \"\"\"True when D_inf is within two standard errors of zero.\"\"\"\n        if not np.isfinite(self.d_infinity_stderr):\n            return False\n        return self.d_infinity <= 2.0 * self.d_infinity_stderr\n\n    def resolves(self, tolerance=1e-6) -> bool:\n        \"\"\"Whether the fit can say anything AT the tolerance scale.\n\n        Separate from `identifiable`, which asks whether the trace has enough\n        decay leverage. This asks whether the resulting uncertainty is small\n        enough to matter: a short trace can fit a clean power law (good decay\n        range) and still leave D_inf uncertain by far more than the tolerance\n        being tested, in which case \"consistent with zero\" is true but says\n        nothing. Measured on a 400-iteration run: D_inf = 2.2e-21 +- 9.8e-3\n        against a 1e-6 tolerance.\n        \"\"\"\n        if not np.isfinite(self.d_infinity_stderr):\n            return False\n        return self.d_infinity_stderr <= tolerance\n\n    def verdict(self, tolerance=1e-6) -> str:\n        if self.at_machine_precision:\n            return (\n                \"D is at machine precision throughout: this formulation \"\n                \"conserves mass by construction, so there is no drift to \"\n                \"characterise and no convergence claim to make. Test 3 is \"\n                \"VACUOUS here (see docs/falsifiability-matrix.md), not passed\"\n            )\n        if not self.converged:\n            return \"fit did not converge; too few points or too little variation\"\n        if not self.tracks_data:\n            if not self.moved_off_guess:\n                return (\n                    \"the fit is degenerate: curve_fit returned the initial \"\n                    \"guess unchanged, so no fit was performed. No verdict can \"\n                    \"be drawn from it\"\n                )\n            return (\n                f\"the fitted curve does not describe the trace: it misses the \"\n                f\"data by up to {self.max_relative_residual:.1f}x its own \"\n                f\"value, against the {self.MAXIMUM_RELATIVE_RESIDUAL:.0f}x \"\n                \"cut. D_inf is not a measurement of anything here\"\n            )\n        if not self.identifiable:\n            return (\n                f\"D_inf is NOT identifiable from this trace: the fitted \"\n                f\"transient decayed by only {self.decay_range:.1f}x across it, \"\n                f\"against the {self.MINIMUM_DECAY_RANGE:.0f}x needed to \"\n                \"separate the constant from the transient. No bias or \"\n                \"under-convergence verdict can be drawn -- extend the L-BFGS \"\n                \"budget and refit\"\n            )\n        if not self.resolves(tolerance):\n            return (\n                f\"the trace cannot resolve D_inf at the {tolerance:.0e} scale: \"\n                f\"the fitted uncertainty is +-{self.d_infinity_stderr:.1e}, \"\n                \"larger than the tolerance being tested. Any verdict here would \"\n                \"be an over-claim -- run longer before concluding anything\"\n            )\n        if self.consistent_with_zero:\n            return (\n                \"D_inf is consistent with zero: the residual drift is \"\n                \"under-convergence, and a longer run should clear it\"\n            )\n        if self.d_infinity > tolerance:\n            return (\n                f\"D_inf = {self.d_infinity:.3e} +- {self.d_infinity_stderr:.1e} \"\n                f\"exceeds {tolerance:.0e}: this is formulation bias, and test 3 \"\n                \"should be closed as a characterised limitation rather than \"\n                \"chased with more iterations\"\n            )\n        return (\n            f\"D_inf = {self.d_infinity:.3e} +- {self.d_infinity_stderr:.1e} \"\n            f\"lies below {tolerance:.0e}: more iterations should reach the target\"\n        )\n\n\ndef fit_drift(iterations, drifts) -> DriftFit:\n    \"\"\"Least-squares fit of D(n) = D_inf + a n^-b.\"\"\"\n    from scipy.optimize import curve_fit\n\n    iterations = np.asarray(iterations, dtype=np.float64)\n    drifts = np.asarray(drifts, dtype=np.float64)\n\n    # A formulation that conserves mass by construction traces D at the\n    # rounding floor. Fitting D(n) = D_inf + a n^-b to that is fitting noise,\n    # and it produced the confident nonsense \"D_inf = 1.0e-10, more iterations\n    # should reach the target\" for a run whose D was already exactly zero.\n    finite = drifts[np.isfinite(drifts)]\n    if finite.size and float(np.max(finite)) < 1e-12:\n        return DriftFit(\n            d_infinity=float(np.max(finite)), amplitude=0.0, exponent=float(\"nan\"),\n            d_infinity_stderr=0.0, points=int(finite.size), converged=False,\n            message=\"D is at machine precision; nothing to fit\",\n            at_machine_precision=True,\n        )\n\n    keep = (iterations > 0) & np.isfinite(drifts) & (drifts > 0)\n    iterations, drifts = iterations[keep], drifts[keep]\n\n    if iterations.size < 5:\n        return DriftFit(np.nan, np.nan, np.nan, np.nan, int(iterations.size), False,\n                        \"fewer than 5 usable trace points\")\n\n    def model(n, d_inf, a, b):\n        return d_inf + a * n ** (-b)\n\n    guess = (float(drifts[-1]), float(max(drifts[0] - drifts[-1], drifts[0])), 0.5)\n    try:\n        # Weight each residual by its own D. Unweighted, this problem spans\n        # three and a half decades with a cost of O(1e-9) at the initial\n        # guess -- already under curve_fit's default ftol, so the optimiser\n        # returns p0 and reports success. Relative residuals are O(1), which\n        # is also the right statistical choice: the trace is meaningful to\n        # constant relative, not constant absolute, precision.\n        popt, pcov = curve_fit(\n            model, iterations, drifts, p0=guess, sigma=drifts, absolute_sigma=False,\n            bounds=([0.0, 0.0, 1e-3], [np.inf, np.inf, 5.0]), maxfev=20000,\n        )\n    except Exception as error:  # noqa: BLE001 - a failed fit is a reportable state\n        return DriftFit(np.nan, np.nan, np.nan, np.nan, int(iterations.size), False,\n                        f\"curve_fit failed: {error}\")\n\n    stderr = float(np.sqrt(np.diag(pcov))[0]) if np.all(np.isfinite(pcov)) else np.nan\n    first, last = float(iterations[0]), float(iterations[-1])\n    tail = float(popt[1] * last ** (-popt[2]))\n    decay = float((last / first) ** popt[2]) if first > 0 else float(\"nan\")\n    residual = float(np.max(np.abs(model(iterations, *popt) - drifts) / drifts))\n    return DriftFit(\n        d_infinity=float(popt[0]), amplitude=float(popt[1]), exponent=float(popt[2]),\n        d_infinity_stderr=stderr, points=int(iterations.size), converged=True,\n        message=\"ok\", tail_at_last_point=tail, last_iteration=last,\n        decay_range=decay, max_relative_residual=residual,\n        moved_off_guess=not bool(np.allclose(popt, guess)),\n    )\n",
"src/evaluation/falsifiability.py": "\"\"\"Falsifiability matrix (SPEC v2, S5).\n\nA test `T` with assertion `A(theta)` over network parameters is VACUOUS under\nansatz `F` if `A(F(theta))` holds for all `theta`. Then `T` reports a property\nof `F`, not of the trained solution.\n\nVacuity is decidable by sampling: if a randomly initialised, UNTRAINED network\npasses a test, that test cannot distinguish a trained solution from noise.\n\n    LIVE      the untrained model fails at every seed -- the test bites\n    WEAKENED  it passes at some seeds and not others\n    VACUOUS   it passes at every seed -- the test measures the ansatz\n\nThis exists because Phase 1 hit the failure mode twice and nothing reported it:\nhard no-slip silently emptied the wall tests, and the stream function silently\nemptied the mass-conservation test. The defect is not the disarming, it is the\nabsence of detection.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\nfrom ..geometry.domain import WALL, Channel\nfrom ..geometry.sampling import evaluation_grid\nfrom ..models.build import build_model\nfrom ..physics import analytic\nfrom .metrics import cross_channel_flux, evaluate_fields, flux_conservation_error, max_abs, relative_l2\n\nLIVE = \"LIVE\"\nWEAKENED = \"WEAKENED\"\nVACUOUS = \"VACUOUS\"\n\n#: A check can be LIVE at random initialisation and still be nearly free,\n#: because the ansatz makes the correct answer a single trivially reachable\n#: point in parameter space. Sampling at the initialisation distribution cannot\n#: see that. `degenerate_targets` probes for it directly by zeroing the output\n#: layer, i.e. setting the network's contribution identically to zero.\nDEGENERATE = \"DEGENERATE-TARGET\"\n\n\n@dataclass(frozen=True)\nclass Check:\n    \"\"\"One assertion, evaluated on a model without training it.\n\n    `passes(config, model, params) -> bool` must mirror the real test's\n    assertion, including its tolerance.\n    \"\"\"\n\n    name: str\n    passes: Callable\n    spec_test: str = \"\"\n\n\ndef _wall_points(length, n=64):\n    y = np.concatenate([np.full(n, WALL), np.full(n, -WALL)])\n    x = np.concatenate([np.linspace(0.0, length, n)] * 2)\n    return jnp.asarray(np.stack([x, y], axis=-1))\n\n\ndef _grid(config):\n    return evaluation_grid(Channel(config.domain.length), 121, 61)\n\n\ndef check_wall_no_slip(config, model, params):\n    values = jax.vmap(lambda xy: model.field(params, xy))(\n        _wall_points(config.domain.length)\n    )\n    return max_abs(values[:, 0]) < 1e-12 and max_abs(values[:, 1]) < 1e-12\n\n\ndef check_test_1_u_relative_l2(config, model, params):\n    points, _, _ = _grid(config)\n    u, _, _ = evaluate_fields(model.field, params, points)\n    exact = analytic.u_nd(np.asarray(points[:, 1]))\n    return relative_l2(u, exact) <= 1e-5\n\n\ndef check_test_2_v_max_abs(config, model, params):\n    points, _, _ = _grid(config)\n    _, v, _ = evaluate_fields(model.field, params, points)\n    return max_abs(v) <= 1e-5\n\n\ndef check_test_3_mass_conservation(config, model, params):\n    length = config.domain.length\n    stations = np.linspace(0.05 * length, 0.95 * length, 10)\n    fluxes = cross_channel_flux(model.field, params, stations)\n    try:\n        return flux_conservation_error(fluxes) <= 1e-6\n    except ValueError:\n        return False\n\n\ndef check_inlet_profile(config, model, params):\n    y = jnp.linspace(-1.0, 1.0, 129)\n    points = jnp.stack([jnp.zeros_like(y), y], axis=-1)\n    u, v, _ = evaluate_fields(model.field, params, points)\n    if config.case == \"A\":\n        target = analytic.u_nd(np.asarray(y))\n    else:\n        target = np.full(len(y), config.inlet_uniform_value)\n    return max_abs(u - target) < 1e-6 and max_abs(v) < 1e-6\n\n\ndef check_pressure_gauge(config, model, params):\n    if not model.has_pressure:\n        return False  # no pressure in the system: the assertion does not apply\n    y = jnp.linspace(-1.0, 1.0, 65)\n    points = jnp.stack([jnp.full_like(y, config.domain.length), y], axis=-1)\n    _, _, p = evaluate_fields(model.field, params, points)\n    return max_abs(p) == 0.0\n\n\ndef check_always_true(config, model, params):\n    \"\"\"T-S5.2: a deliberately vacuous check, to prove the detector detects.\"\"\"\n    return True\n\n\ndef check_always_false(config, model, params):\n    \"\"\"Control in the other direction: must always report LIVE.\"\"\"\n    return False\n\n\nDEFAULT_CHECKS = (\n    Check(\"wall no-slip\", check_wall_no_slip, \"structural\"),\n    Check(\"test 1  u relative L2\", check_test_1_u_relative_l2, \"1\"),\n    Check(\"test 2  max |v*|\", check_test_2_v_max_abs, \"2\"),\n    Check(\"test 3  mass conservation\", check_test_3_mass_conservation, \"3\"),\n    Check(\"inlet profile\", check_inlet_profile, \"structural\"),\n    Check(\"pressure gauge at outlet\", check_pressure_gauge, \"structural\"),\n    Check(\"CONTROL always-true\", check_always_true, \"T-S5.2\"),\n    Check(\"CONTROL always-false\", check_always_false, \"T-S5.2\"),\n)\n\n\ndef classify(config, checks=DEFAULT_CHECKS, seeds=(0, 1, 2, 3, 4)):\n    \"\"\"Classify every check for one configuration, at random initialisation.\"\"\"\n    verdicts = {}\n    for check in checks:\n        outcomes = []\n        for seed in seeds:\n            params, model = build_model(config, jax.random.PRNGKey(seed))\n            outcomes.append(bool(check.passes(config, model, params)))\n        if all(outcomes):\n            verdicts[check.name] = VACUOUS\n        elif any(outcomes):\n            verdicts[check.name] = WEAKENED\n        else:\n            verdicts[check.name] = LIVE\n    return verdicts\n\n\ndef build_matrix(configs, checks=DEFAULT_CHECKS, seeds=(0, 1, 2, 3, 4)):\n    \"\"\"`{check name: {column label: verdict}}` across several configurations.\"\"\"\n    matrix = {check.name: {} for check in checks}\n    for label, config in configs.items():\n        for name, verdict in classify(config, checks, seeds).items():\n            matrix[name][label] = verdict\n    return matrix\n\n\ndef format_matrix(matrix, columns) -> str:\n    width = max(len(name) for name in matrix) + 2\n    header = \"| \" + \"check\".ljust(width) + \"| \" + \" | \".join(\n        c.ljust(10) for c in columns\n    ) + \" |\"\n    rule = \"|\" + \"-\" * (width + 1) + \"|\" + \"|\".join(\"-\" * 12 for _ in columns) + \"|\"\n    lines = [header, rule]\n    for name, row in matrix.items():\n        cells = \" | \".join(row.get(c, \"-\").ljust(10) for c in columns)\n        lines.append(\"| \" + name.ljust(width) + \"| \" + cells + \" |\")\n    return \"\\n\".join(lines)\n\n\ndef all_vacuous_rows(matrix):\n    \"\"\"Checks that are vacuous in EVERY column.\n\n    Such a row means the property is untested by this suite and must be flagged\n    in the report, never silently accepted.\n    \"\"\"\n    return [\n        name\n        for name, row in matrix.items()\n        if row and all(v == VACUOUS for v in row.values())\n    ]\n\n\ndef zero_network_contribution(params):\n    \"\"\"Set the output layer to zero, so the network contributes nothing.\n\n    For a psi-omega model under the cubic lift this makes psi exactly the lift,\n    whose derivative IS the Case A solution -- so any check comparing against\n    that solution passes perfectly. Under the sinusoidal lift it does not.\n    \"\"\"\n    import copy\n\n    params = copy.deepcopy(params)\n    output = params[\"output\"] if \"output\" in params else params[\"layers\"][-1]\n    weight_key = \"W\" if \"W\" in output else \"V\"\n    output[weight_key] = jnp.zeros_like(output[weight_key])\n    output[\"b\"] = jnp.zeros_like(output[\"b\"])\n    return params\n\n\ndef degenerate_targets(config, checks=DEFAULT_CHECKS, seed=0):\n    \"\"\"Checks whose assertion passes when the network contributes nothing.\n\n    A check listed here is not vacuous -- an untrained network fails it -- but\n    the ansatz has placed the correct answer at a single point the optimiser\n    can reach exactly, so passing it demonstrates the lift rather than the\n    network's expressivity. SPEC v2 S4 flags exactly this for the cubic lift;\n    this function decides it by measurement instead of by argument.\n    \"\"\"\n    params, model = build_model(config, jax.random.PRNGKey(seed))\n    zeroed = zero_network_contribution(params)\n    degenerate = []\n    for check in checks:\n        if check.name.startswith(\"CONTROL\"):\n            continue\n        try:\n            if check.passes(config, model, zeroed):\n                degenerate.append(check.name)\n        except Exception:  # a check that cannot even run is not degenerate\n            continue\n    return degenerate\n",
"src/evaluation/metrics.py": "\"\"\"Accuracy metrics.\n\nAccuracy is ALWAYS the relative L2 error against the analytic solution,\ncomputed on the fixed evaluation grid. The loss value is a diagnostic and is\nnever reported as an accuracy claim (CLAUDE.md).\n\nThe published failure mode this guards against: a plain DeepXDE setup on this\nexact problem reaching a total test loss of 8.7e-5 while its relative L2 error\nwas 8.6% (SPEC section 5).\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\nfrom ..geometry.domain import WALL\n\n\ndef relative_l2(predicted, exact):\n    \"\"\"||predicted - exact||_2 / ||exact||_2.\n\n    Raises if the reference field is identically zero: the relative L2 error of\n    a zero field is not defined, and silently returning an absolute error there\n    would be exactly the kind of quiet lie this project is built to avoid. Use\n    `max_abs` for fields whose exact value is zero (e.g. v*).\n    \"\"\"\n    predicted = np.asarray(predicted, dtype=np.float64)\n    exact = np.asarray(exact, dtype=np.float64)\n    denominator = np.linalg.norm(exact)\n    if denominator == 0.0:\n        raise ValueError(\n            \"relative L2 error is undefined against an identically zero \"\n            \"reference field; use max_abs instead\"\n        )\n    return float(np.linalg.norm(predicted - exact) / denominator)\n\n\ndef max_abs(values):\n    return float(np.max(np.abs(np.asarray(values))))\n\n\ndef gauss_legendre_nodes(n: int):\n    \"\"\"Gauss-Legendre nodes and weights mapped onto y* in [-1, 1].\n\n    Used for every cross-channel integral. The channel half-height is exactly\n    the Legendre interval, so no rescaling is needed, and GL is exact for the\n    analytic parabola to machine precision -- which matters for SPEC test 3,\n    whose tolerance is 1e-6 relative.\n    \"\"\"\n    nodes, weights = np.polynomial.legendre.leggauss(n)\n    return nodes * WALL, weights * WALL\n\n\ndef evaluate_fields(field_fn, params, points, batch_size: int = 8192):\n    \"\"\"(u, v, p) at an (N, 2) array of points, batched to bound peak memory.\"\"\"\n    apply = jax.jit(jax.vmap(lambda xy: field_fn(params, xy)))\n    outputs = []\n    for start in range(0, points.shape[0], batch_size):\n        outputs.append(np.asarray(apply(points[start : start + batch_size])))\n    stacked = np.concatenate(outputs, axis=0)\n    return stacked[:, 0], stacked[:, 1], stacked[:, 2]\n\n\ndef cross_channel_flux(field_fn, params, x_stations, n_nodes: int = 200):\n    \"\"\"int_{-1}^{1} u*(x, y) dy at each x station, by Gauss-Legendre quadrature.\"\"\"\n    nodes, weights = gauss_legendre_nodes(n_nodes)\n    x_stations = np.asarray(x_stations, dtype=np.float64)\n    grid_x, grid_y = np.meshgrid(x_stations, nodes, indexing=\"ij\")\n    points = jnp.asarray(np.stack([grid_x.reshape(-1), grid_y.reshape(-1)], axis=-1))\n    u, _, _ = evaluate_fields(field_fn, params, points)\n    u = u.reshape(len(x_stations), n_nodes)\n    return u @ weights\n\n\ndef flux_conservation_error(fluxes):\n    \"\"\"Relative spread of the flux across stations.\n\n    Station-to-station AGREEMENT, not agreement with the analytic flux: this is\n    the mass-conservation check (SPEC test 3), and a solution that conserves\n    mass at the wrong flux is a different failure from one that does not\n    conserve it at all. Both are reported.\n    \"\"\"\n    fluxes = np.asarray(fluxes, dtype=np.float64)\n    mean = float(np.mean(fluxes))\n    if mean == 0.0:\n        raise ValueError(\"mean flux is zero; conservation error is undefined\")\n    return float(np.max(np.abs(fluxes - mean)) / abs(mean))\n\n\ndef centreline_velocity(field_fn, params, x_stations):\n    \"\"\"u*(x, 0) at each station -- the entrance-length diagnostic.\"\"\"\n    x_stations = np.asarray(x_stations, dtype=np.float64)\n    points = jnp.asarray(np.stack([x_stations, np.zeros_like(x_stations)], axis=-1))\n    u, _, _ = evaluate_fields(field_fn, params, points)\n    return u\n\n\ndef entrance_length(x_stations, centreline, developed_value, threshold=0.99):\n    \"\"\"First x* at which the centreline reaches `threshold` of its developed value.\n\n    Linear interpolation between the two bracketing stations. Returns NaN if the\n    threshold is never reached inside the domain, which is a real result (the\n    domain is too short) and must not be silently clipped to L*.\n    \"\"\"\n    x_stations = np.asarray(x_stations, dtype=np.float64)\n    centreline = np.asarray(centreline, dtype=np.float64)\n    target = threshold * developed_value\n    reached = centreline >= target\n    if not np.any(reached):\n        return float(\"nan\")\n    index = int(np.argmax(reached))\n    if index == 0:\n        return float(x_stations[0])\n    x0, x1 = x_stations[index - 1], x_stations[index]\n    c0, c1 = centreline[index - 1], centreline[index]\n    if c1 == c0:\n        return float(x1)\n    return float(x0 + (target - c0) * (x1 - x0) / (c1 - c0))\n",
"src/evaluation/plots.py": "\"\"\"Figures for a finished run. Written into the run directory.\n\nMatplotlib only, no seaborn, one figure per file, default colours. These are\ndiagnostic figures for the Phase 1 write-up, not publication graphics.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nfrom pathlib import Path\n\nimport jax.numpy as jnp\nimport matplotlib\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport numpy as np  # noqa: E402\n\nfrom ..config import Config  # noqa: E402\nfrom ..geometry.domain import Channel  # noqa: E402\nfrom ..geometry.sampling import evaluation_grid  # noqa: E402\nfrom ..physics import analytic  # noqa: E402\nfrom ..physics import residuals  # noqa: E402\nfrom ..physics.derivatives import batched_field_derivatives  # noqa: E402\nfrom .metrics import evaluate_fields  # noqa: E402\n\n\ndef _grid_fields(config: Config, model, params):\n    channel = Channel(config.domain.length)\n    points, x, y = evaluation_grid(channel, config.evaluation.nx, config.evaluation.ny)\n    u, v, p = evaluate_fields(model.field, params, points)\n    shape = (config.evaluation.nx, config.evaluation.ny)\n    return np.asarray(x), np.asarray(y), u.reshape(shape), v.reshape(shape), p.reshape(shape), points\n\n\ndef plot_velocity_profiles(config, model, params, path: Path):\n    \"\"\"Predicted vs analytic profile at several stations.\"\"\"\n    x, y, u, _, _, _ = _grid_fields(config, model, params)\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))\n\n    stations = np.linspace(0, len(x) - 1, 5).astype(int)\n    for index in stations:\n        axes[0].plot(u[index], y, label=f\"x* = {x[index]:.1f}\")\n    if config.case == \"A\":\n        reference = analytic.u_nd(y)\n        label = \"analytic  1 - y*\u00b2\"\n    else:\n        reference = analytic.developed_profile_nd(y, config.inlet_flux)\n        label = f\"developed (flux {config.inlet_flux:.3f})\"\n    axes[0].plot(reference, y, \"k--\", linewidth=2, label=label)\n    axes[0].set_xlabel(\"u*\")\n    axes[0].set_ylabel(\"y*\")\n    axes[0].set_title(f\"{config.name}: streamwise profiles\")\n    axes[0].legend(fontsize=8)\n    axes[0].grid(alpha=0.3)\n\n    if config.case == \"A\":\n        error = np.abs(u - analytic.u_nd(y)[None, :])\n        image = axes[1].pcolormesh(x, y, error.T, shading=\"auto\")\n        fig.colorbar(image, ax=axes[1], label=\"|u* - u*_exact|\")\n        axes[1].set_title(\"pointwise error in u*\")\n    else:\n        axes[1].plot(x, u[:, u.shape[1] // 2])\n        axes[1].axhline(0.75 * config.inlet_flux, color=\"k\", linestyle=\"--\",\n                        label=\"developed centreline\")\n        axes[1].set_ylabel(\"u*(x*, 0)\")\n        axes[1].set_title(\"centreline development\")\n        axes[1].legend(fontsize=8)\n    axes[1].set_xlabel(\"x*\")\n    axes[1].grid(alpha=0.3)\n\n    fig.tight_layout()\n    fig.savefig(path, dpi=140)\n    plt.close(fig)\n\n\ndef plot_residual_map(config, model, params, path: Path):\n    \"\"\"Where the momentum residual lives. A diagnostic, never an accuracy claim.\"\"\"\n    channel = Channel(config.domain.length)\n    points, x, y = evaluation_grid(channel, config.evaluation.nx, config.evaluation.ny)\n    shape = (config.evaluation.nx, config.evaluation.ny)\n    if config.physics.formulation == \"psi_omega\":\n        from ..physics import streamfunction\n        from ..physics.derivatives import batched_stream_derivatives\n\n        sd = batched_stream_derivatives(lambda xy: model.stream(params, xy), points)\n        values_and_names = list(\n            zip(\n                streamfunction.residuals(sd, config.physics.reynolds),\n                (\"r1  omega + lap psi\", \"r2  vorticity transport\"),\n            )\n        )\n    else:\n        d = batched_field_derivatives(lambda xy: model.field(params, xy), points)\n        values_and_names = list(\n            zip(\n                residuals.navier_stokes(\n                    d, config.physics.reynolds, config.physics.pressure_scaling\n                ),\n                (\"momentum x\", \"momentum y\", \"continuity\"),\n            )\n        )\n\n    fig, axes = plt.subplots(len(values_and_names), 1, figsize=(9, 8), sharex=True)\n    axes = np.atleast_1d(axes)\n    for axis, (values, name) in zip(axes, values_and_names):\n        magnitude = np.abs(np.asarray(values)).reshape(shape).T\n        floor = max(magnitude.max(), 1e-300) * 1e-8\n        image = axis.pcolormesh(\n            np.asarray(x), np.asarray(y), np.log10(np.maximum(magnitude, floor)),\n            shading=\"auto\",\n        )\n        fig.colorbar(image, ax=axis, label=\"log10 |residual|\")\n        axis.set_ylabel(\"y*\")\n        axis.set_title(f\"{name} residual (unweighted, diagnostic only)\")\n    axes[-1].set_xlabel(\"x*\")\n    fig.tight_layout()\n    fig.savefig(path, dpi=140)\n    plt.close(fig)\n\n\ndef plot_loss_history(history_path: Path, path: Path, title: str):\n    \"\"\"Per-term loss history from the Adam phase.\"\"\"\n    if not history_path.is_file():\n        return\n    with history_path.open(encoding=\"utf-8\") as handle:\n        rows = list(csv.DictReader(handle))\n    if not rows:\n        return\n\n    steps = np.array([float(row[\"step\"]) for row in rows])\n    term_columns = sorted(c for c in rows[0] if c.startswith(\"term_\"))\n\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))\n    axes[0].semilogy(steps, [float(row[\"loss\"]) for row in rows], \"k\", label=\"total\")\n    for column in term_columns:\n        values = np.array([float(row[column]) for row in rows])\n        axes[0].semilogy(steps, np.maximum(values, 1e-300), label=column[5:], alpha=0.8)\n    axes[0].set_xlabel(\"Adam step\")\n    axes[0].set_ylabel(\"loss (diagnostic, not accuracy)\")\n    axes[0].set_title(title)\n    axes[0].legend(fontsize=8)\n    axes[0].grid(alpha=0.3)\n\n    weight_columns = sorted(c for c in rows[0] if c.startswith(\"weight_\"))\n    for column in weight_columns:\n        axes[1].semilogy(steps, [float(row[column]) for row in rows], label=column[7:])\n    axes[1].set_xlabel(\"Adam step\")\n    axes[1].set_ylabel(\"loss weight\")\n    axes[1].set_title(\"loss weights\")\n    axes[1].legend(fontsize=8)\n    axes[1].grid(alpha=0.3)\n\n    fig.tight_layout()\n    fig.savefig(path, dpi=140)\n    plt.close(fig)\n\n\ndef write_all(config: Config, model, params, directory: Path):\n    plot_velocity_profiles(config, model, params, directory / \"profiles.png\")\n    plot_residual_map(config, model, params, directory / \"residuals.png\")\n    plot_loss_history(\n        directory / \"history.csv\", directory / \"loss_history.png\", config.name\n    )\n",
"src/geometry/__init__.py": "\"\"\"Domain definitions, wall distance functions, and collocation sampling.\"\"\"\n",
"src/geometry/domain.py": "\"\"\"The Phase 1 domain: a straight channel x* in [0, L*], y* in [-1, 1].\n\nEverything is non-dimensional. The half-height is 1 by construction -- that is\nwhat \"lengths are scaled by h\" means -- so it is not a free parameter and is\nnot configurable.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport jax.numpy as jnp\n\n#: Non-dimensional wall position. Walls are at y* = -WALL and y* = +WALL.\nWALL = 1.0\n\n\n@dataclass(frozen=True)\nclass Channel:\n    \"\"\"Straight channel of non-dimensional length `length`.\"\"\"\n\n    length: float\n\n    def __post_init__(self):\n        if not (self.length > 0):\n            raise ValueError(f\"channel length must be positive, got {self.length!r}\")\n\n    @property\n    def bounds_lower(self):\n        return jnp.array([0.0, -WALL])\n\n    @property\n    def bounds_upper(self):\n        return jnp.array([self.length, WALL])\n\n\ndef phi(y):\n    \"\"\"Wall distance function: phi(y*) = 1 - y*^2.\n\n    Vanishes on both walls and is strictly positive between them, so\n    multiplying a network output by it enforces no-slip EXACTLY rather than\n    through a penalty. This is the single largest accuracy lever in the project\n    (SPEC section 3), which is why it is a named function with its own tests\n    rather than an inline expression.\n    \"\"\"\n    return 1.0 - y**2\n\n\ndef dphi_dy(y):\n    \"\"\"d(phi)/dy = -2y. Written out so tests can check phi against it.\"\"\"\n    return -2.0 * y\n\n\ndef normalise(xy, length):\n    \"\"\"Map (x*, y*) from [0, L*] x [-1, 1] onto [-1, 1]^2.\n\n    A tanh network fed x* in [0, 20] and y* in [-1, 1] sees two inputs whose\n    scales differ by a factor 20; the first hidden layer then has to undo that\n    with its weights. This is a structural property of the domain, not a tuned\n    hyperparameter, so it is derived from the config's domain length rather\n    than being configurable.\n    \"\"\"\n    x, y = xy[..., 0], xy[..., 1]\n    return jnp.stack([2.0 * x / length - 1.0, y / WALL], axis=-1)\n",
"src/geometry/lifts.py": "\"\"\"Stream-function lifts (SPEC v2, S4).\n\nA lift `psi_lift(y*)` must satisfy\n\n    psi_lift(+-1) = +- Q*/2      (the wall values differ by the flux)\n    psi_lift'(+-1) = 0           (no-slip: u = dpsi/dy vanishes at the wall)\n\nso that with `psi = psi_lift + phi^2 N_psi` and `phi = 1 - y*^2` both wall\nconditions hold exactly for any `N_psi`.\n\nTwo lifts are provided because ONE OF THEM DISARMS THE GATE TEST.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax.numpy as jnp\n\n\ndef cubic(y, flux):\n    \"\"\"psi = (3Q/4)(y - y^3/3).\n\n    Check: psi(1) = (3Q/4)(2/3) = Q/2, and psi' = (3Q/4)(1 - y^2) = 0 at y = +-1.\n\n    WARNING -- this lift IS the exact Case A solution. Its derivative is\n    (3Q/4)(1 - y^2), which for Q = 4/3 is exactly 1 - y^2. Under this lift the\n    correct Case A answer is `N_psi = 0`, so Case A stops testing the network's\n    expressivity and tests only that the lift and training machinery are wired\n    up. Use it as the MACHINERY check; never quote its accuracy as a result.\n    \"\"\"\n    return 0.75 * flux * (y - y**3 / 3.0)\n\n\ndef sinusoidal(y, flux):\n    \"\"\"psi = (Q/2) sin(pi y / 2).\n\n    Check: psi(+-1) = +-Q/2, and psi' = (Q pi/4) cos(pi y/2) = 0 at y = +-1.\n\n    Satisfies the same boundary conditions as `cubic` but its derivative is NOT\n    the Poiseuille profile, so `N_psi` must learn a non-zero correction. This is\n    the EXPRESSIVITY check and the only one whose accuracy may be quoted.\n    \"\"\"\n    return 0.5 * flux * jnp.sin(0.5 * jnp.pi * y)\n\n\nLIFTS = {\"cubic\": cubic, \"sinusoidal\": sinusoidal}\n\n#: Lifts whose derivative is the exact fully developed profile, so that the\n#: correct answer is N_psi = 0. Reported by the sentinel matrix (S5).\nDEGENERATE_LIFTS = frozenset({\"cubic\"})\n\n\ndef get(name):\n    if name not in LIFTS:\n        raise ValueError(f\"unknown lift {name!r}; expected one of {sorted(LIFTS)}\")\n    return LIFTS[name]\n",
"src/geometry/sampling.py": "\"\"\"Collocation sampling.\n\nTwo rules from the SPEC drive the design here:\n\n* the sampling seed is logged and every draw is a pure function of an explicit\n  PRNG key, so a run is reproducible from (config, seed) alone;\n* the fixed EVALUATION grid is built by a different function from the training\n  points, because reported error must never be measured on training points\n  (SPEC section 9, \"Resampling collocation points\").\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import NamedTuple\n\nimport jax\nimport jax.numpy as jnp\n\nfrom .domain import WALL, Channel\n\n\nclass CollocationPoints(NamedTuple):\n    \"\"\"One draw of training points. All arrays are (N, 2) of (x*, y*).\"\"\"\n\n    interior: jnp.ndarray\n    inlet: jnp.ndarray\n    outlet: jnp.ndarray\n\n\ndef sample_interior(key, channel: Channel, n: int):\n    \"\"\"Uniform random points strictly inside the channel.\n\n    Wall points are excluded: no-slip is exact by construction, so a\n    collocation point on the wall contributes nothing but a degenerate residual\n    (phi = 0 makes u and v identically zero there for any parameters).\n    \"\"\"\n    return jax.random.uniform(\n        key,\n        shape=(n, 2),\n        minval=channel.bounds_lower,\n        maxval=channel.bounds_upper,\n        dtype=jnp.float64,\n    )\n\n\ndef sample_inlet(key, channel: Channel, n: int):\n    \"\"\"Points on x* = 0, y* uniform in (-1, 1).\"\"\"\n    y = jax.random.uniform(key, shape=(n,), minval=-WALL, maxval=WALL, dtype=jnp.float64)\n    return jnp.stack([jnp.zeros_like(y), y], axis=-1)\n\n\ndef sample_outlet(key, channel: Channel, n: int):\n    \"\"\"Points on x* = L*, y* uniform in (-1, 1).\"\"\"\n    y = jax.random.uniform(key, shape=(n,), minval=-WALL, maxval=WALL, dtype=jnp.float64)\n    return jnp.stack([jnp.full_like(y, channel.length), y], axis=-1)\n\n\ndef sample_all(key, channel: Channel, n_interior: int, n_inlet: int, n_outlet: int):\n    \"\"\"One complete draw. Pure function of `key`.\"\"\"\n    k_int, k_in, k_out = jax.random.split(key, 3)\n    return CollocationPoints(\n        interior=sample_interior(k_int, channel, n_interior),\n        inlet=sample_inlet(k_in, channel, n_inlet),\n        outlet=sample_outlet(k_out, channel, n_outlet),\n    )\n\n\ndef evaluation_grid(channel: Channel, nx: int, ny: int):\n    \"\"\"Fixed tensor-product grid for reporting error. Never used for training.\n\n    Endpoints are included: the walls are where the hard BC is exact and the\n    inlet/outlet are where the boundary conditions live, so an error metric\n    that skipped them would flatter the model.\n\n    Returns (points (nx*ny, 2), x (nx,), y (ny,)).\n    \"\"\"\n    x = jnp.linspace(0.0, channel.length, nx, dtype=jnp.float64)\n    y = jnp.linspace(-WALL, WALL, ny, dtype=jnp.float64)\n    xx, yy = jnp.meshgrid(x, y, indexing=\"ij\")\n    points = jnp.stack([xx.reshape(-1), yy.reshape(-1)], axis=-1)\n    return points, x, y\n\n\ndef station_grid(channel: Channel, n_stations: int, ny: int, *, margin: float = 0.0):\n    \"\"\"Streamwise stations for the mass-conservation check (SPEC test 3).\n\n    `margin` trims the stations away from the inlet and outlet planes; the\n    conservation check is about the interior, and a station sitting exactly on\n    a boundary would fold a BC error into a conservation error.\n\n    Returns (points (n_stations, ny, 2), x_stations (n_stations,), y (ny,)).\n    \"\"\"\n    x = jnp.linspace(\n        margin * channel.length,\n        (1.0 - margin) * channel.length,\n        n_stations,\n        dtype=jnp.float64,\n    )\n    y = jnp.linspace(-WALL, WALL, ny, dtype=jnp.float64)\n    xx, yy = jnp.meshgrid(x, y, indexing=\"ij\")\n    return jnp.stack([xx, yy], axis=-1), x, y\n",
"src/models/__init__.py": "\"\"\"Network architectures and the hard boundary-condition ansatz.\"\"\"\n",
"src/models/build.py": "\"\"\"Assemble a model from a validated `Config`.\n\nReturns `(params, field_fn)` where `field_fn(params, xy) -> [u, v, p]` at a\nsingle non-dimensional point. Everything that varies between experiments comes\nfrom the config; nothing here has a default.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport jax\nimport jax.numpy as jnp\n\nfrom ..config import Config\nfrom .fourier import apply_fourier, init_fourier_matrix\nfrom .hard_bc import (\n    make_psi_omega_field,\n    make_velocity_from_stream,\n    make_velocity_pressure_field,\n)\nfrom .mlp import apply_mlp, apply_modified_mlp, init_mlp, init_modified_mlp\n\n#: Raw network outputs per formulation.\n#: velocity_pressure -> (N_u, N_v, N_p);  psi_omega -> (N_psi, N_omega).\n_N_OUTPUTS = {\"velocity_pressure\": 3, \"psi_omega\": 2}\n\n\n@dataclass(frozen=True)\nclass Model:\n    \"\"\"What a formulation exposes.\n\n    `has_pressure` is False for the psi-omega system: pressure is absent from\n    that system and is recovered post hoc by path integration, so any pressure\n    metric taken from `field` there would be meaningless. Callers must check\n    the flag rather than trusting the third slot of `field`.\n    \"\"\"\n\n    formulation: str\n    field: object          # (params, xy) -> [u, v, p]; p is 0 when has_pressure is False\n    velocity: object       # (params, xy) -> [u, v]\n    stream: object         # (params, xy) -> [psi, omega], or None\n    has_pressure: bool\n    lift: object = None\n\n\ndef build_model(config: Config, key):\n    \"\"\"Build parameters and the field function for `config`.\"\"\"\n    fourier_key, net_key = jax.random.split(key)\n\n    n_out = _N_OUTPUTS[config.physics.formulation]\n\n    if config.model.fourier.enabled:\n        matrix = init_fourier_matrix(\n            fourier_key,\n            config.model.fourier.num_features,\n            config.model.fourier.bandwidth,\n        )\n        n_in = 2 * config.model.fourier.num_features\n        embed = lambda xi: apply_fourier(matrix, xi)  # noqa: E731\n    else:\n        n_in = 2\n        embed = lambda xi: xi  # noqa: E731\n\n    rwf = dict(\n        rwf_enabled=config.model.rwf.enabled,\n        rwf_mean=config.model.rwf.mean,\n        rwf_stddev=config.model.rwf.stddev,\n    )\n\n    if config.model.arch == \"mlp\":\n        params = init_mlp(net_key, n_in, n_out, config.model.width, config.model.depth, **rwf)\n        core = apply_mlp\n    elif config.model.arch == \"modified_mlp\":\n        params = init_modified_mlp(\n            net_key, n_in, n_out, config.model.width, config.model.depth, **rwf\n        )\n        core = apply_modified_mlp\n    else:\n        raise ValueError(f\"unknown model.arch {config.model.arch!r}\")\n\n    def net_apply(p, xi):\n        return core(p, embed(xi))\n\n    if config.physics.formulation == \"velocity_pressure\":\n        field_fn = make_velocity_pressure_field(\n            net_apply,\n            length=config.domain.length,\n            inlet_mode=config.bc.inlet,\n            lifting_length=config.bc.lifting_length,\n            gauge_mode=config.bc.pressure_gauge,\n        )\n        model = Model(\n            formulation=\"velocity_pressure\",\n            field=field_fn,\n            velocity=lambda p, xy: field_fn(p, xy)[:2],\n            stream=None,\n            has_pressure=True,\n        )\n    elif config.physics.formulation == \"psi_omega\":\n        if config.bc.inlet == \"lifting\":\n            raise ValueError(\n                \"bc.inlet = 'lifting' is not implemented for the psi_omega \"\n                \"formulation: the inlet condition there is a condition on \"\n                \"dpsi/dy, and a lifting construction exact for both u and v at \"\n                \"x* = 0 needs lambda'(0) = 0 as well. Use bc.inlet = 'soft'.\"\n            )\n        stream_fn = make_psi_omega_field(\n            net_apply,\n            length=config.domain.length,\n            flux=config.inlet_flux,\n            lift=config.model.lift,\n        )\n        velocity_fn = make_velocity_from_stream(stream_fn)\n        model = Model(\n            formulation=\"psi_omega\",\n            # Pressure is absent from the psi-omega system. The zero here is a\n            # placeholder that `has_pressure=False` forbids anyone reporting.\n            field=lambda p, xy: jnp.concatenate(\n                [velocity_fn(p, xy), jnp.zeros((1,), dtype=jnp.float64)]\n            ),\n            velocity=velocity_fn,\n            stream=stream_fn,\n            has_pressure=False,\n            lift=config.model.lift,\n        )\n    else:\n        raise ValueError(f\"unknown physics.formulation {config.physics.formulation!r}\")\n\n    return params, model\n\n\ndef count_parameters(params) -> int:\n    return sum(int(leaf.size) for leaf in jax.tree.leaves(params))\n",
"src/models/dense.py": "\"\"\"Dense layers, with optional random weight factorisation (RWF).\n\nRWF (SPEC section 3 item 4) reparameterises each weight matrix as\n\n    W = diag(exp(s)) V,      s ~ N(mean, stddev)\n\nso every output neuron carries its own learnable scale. Initialisation draws W\nby Glorot normal first and then factors it, so switching RWF on does not change\nthe initial function the network represents -- only how the optimiser moves\nthrough parameter space.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\n\ndef glorot_normal(key, n_out: int, n_in: int):\n    \"\"\"Glorot (Xavier) normal initialisation, float64.\"\"\"\n    stddev = jnp.sqrt(2.0 / (n_in + n_out))\n    return stddev * jax.random.normal(key, (n_out, n_in), dtype=jnp.float64)\n\n\ndef init_dense(key, n_in: int, n_out: int, *, rwf_enabled: bool, rwf_mean=None, rwf_stddev=None):\n    \"\"\"Initialise one dense layer as a parameter pytree.\"\"\"\n    w_key, s_key = jax.random.split(key)\n    weight = glorot_normal(w_key, n_out, n_in)\n    bias = jnp.zeros((n_out,), dtype=jnp.float64)\n\n    if not rwf_enabled:\n        return {\"W\": weight, \"b\": bias}\n\n    if rwf_mean is None or rwf_stddev is None:\n        raise ValueError(\"RWF requires both rwf_mean and rwf_stddev\")\n    scale = rwf_mean + rwf_stddev * jax.random.normal(s_key, (n_out,), dtype=jnp.float64)\n    # V = W / exp(s), so that diag(exp(s)) V == W exactly at initialisation.\n    factor = weight / jnp.exp(scale)[:, None]\n    return {\"s\": scale, \"V\": factor, \"b\": bias}\n\n\ndef apply_dense(layer, x):\n    \"\"\"Apply one dense layer to a single input vector.\"\"\"\n    if \"W\" in layer:\n        return layer[\"W\"] @ x + layer[\"b\"]\n    return (jnp.exp(layer[\"s\"])[:, None] * layer[\"V\"]) @ x + layer[\"b\"]\n\n\ndef layer_weight(layer):\n    \"\"\"The effective weight matrix of a layer, factorised or not.\"\"\"\n    if \"W\" in layer:\n        return layer[\"W\"]\n    return jnp.exp(layer[\"s\"])[:, None] * layer[\"V\"]\n",
"src/models/fourier.py": "\"\"\"Random Fourier feature embedding of the input coordinates.\n\n    gamma(xi) = [cos(B xi), sin(B xi)],   B ~ N(0, bandwidth^2)\n\nwith `xi` the coordinates already normalised to [-1, 1]^2. Addresses the\nspectral bias of a plain MLP (SPEC section 3 item 2).\n\nB is drawn once from the run's PRNG key and then held FIXED -- it is not a\ntrainable parameter, so it lives outside the parameter pytree and is never\ntouched by the optimiser. That also keeps the L-BFGS parameter vector free of\nconstants.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\n\ndef init_fourier_matrix(key, num_features: int, bandwidth: float, n_inputs: int = 2):\n    \"\"\"Draw the fixed frequency matrix B of shape (num_features, n_inputs).\"\"\"\n    return bandwidth * jax.random.normal(\n        key, (num_features, n_inputs), dtype=jnp.float64\n    )\n\n\ndef apply_fourier(matrix, xi):\n    \"\"\"Embed a single normalised coordinate vector. Output dim = 2 * num_features.\"\"\"\n    projection = matrix @ xi\n    return jnp.concatenate([jnp.cos(projection), jnp.sin(projection)])\n",
"src/models/hard_bc.py": "\"\"\"The boundary-condition ansatz: what the network is allowed to represent.\n\nNo-slip is NEVER a penalty term in this project (SPEC section 3). The velocity\nis constructed so that it vanishes on both walls for any parameter values:\n\n    u*(x*, y*) = phi(y*) N_u(x*, y*)\n    v*(x*, y*) = phi(y*) N_v(x*, y*)      phi(y*) = 1 - y*^2\n\nInlet handling\n--------------\nTwo options, both implemented, chosen by config (`bc.inlet`):\n\n* \"soft\"    -- an inlet loss term. The PINN has to DISCOVER the parabolic\n               profile from the PDE plus a soft nudge. This is the default for\n               Case A because it is the honest test of the harness: with\n               lifting, u(0, y) = phi(y) = 1 - y^2 is already the exact Case A\n               inlet condition and the inlet is no longer being solved for.\n* \"lifting\" -- u = phi(y) [1 + lambda(x) N_u], v = phi(y) lambda(x) N_v with\n               lambda(x) = 1 - exp(-x / l), lambda(0) = 0. This makes the\n               Case A inlet exact by construction and removes the inlet loss\n               term entirely. Only available for Case A: a uniform Case B inlet\n               is incompatible with hard no-slip at the corners.\n\nPressure gauge\n--------------\nPressure is unique only up to a constant. `bc.pressure_gauge`:\n\n* \"soft\" -- a loss term pinning p*(L*, y*) = 0.\n* \"hard\" -- p* = (1 - x*/L*) N_p, which is exactly zero on the outlet plane.\n\nWithout one of these the pressure relative L2 error is meaningless (SPEC\nsection 9).\n\nStream function\n---------------\nu = dpsi/dy, v = -dpsi/dx satisfies continuity identically. The wall condition\nneeds psi_y = psi_x = 0 at y* = +-1, which phi^2 gives:\n\n    psi = psi_base(y*) + phi(y*)^2 N_psi(x*, y*)\n\nphi^2 alone is NOT enough, though: psi(+1) - psi(-1) equals the flux, so a psi\nthat is constant on both walls can only represent zero net throughflow. Hence\n`psi_base`, which carries the flux and has zero slope at both walls:\n\n    \"cubic\"  psi_base = (Q/4)(3y - y^3)   -> psi_base' = (3Q/4)(1 - y^2)\n    \"sine\"   psi_base = (Q/2) sin(pi y/2) -> psi_base' = (Q pi/4) cos(pi y/2)\n\nThe cubic base IS the exact Case A solution, so with it the network only has to\nlearn N_psi = 0 and the Case A comparison against the velocity-pressure\nformulation would be meaningless. \"sine\" carries the same flux with the wrong\nshape, so the network still has to do work. Use \"sine\" for any reported\ncomparison; \"cubic\" exists to isolate optimiser behaviour.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\nfrom ..geometry import lifts\nfrom ..geometry.domain import normalise, phi\n\n\ndef _lifting(x, lifting_length):\n    \"\"\"lambda(x) = 1 - exp(-x/l): zero at the inlet, ~1 within a few l.\"\"\"\n    return 1.0 - jnp.exp(-x / lifting_length)\n\n\ndef _pressure_gauge_factor(x, length, gauge_mode):\n    if gauge_mode == \"soft\":\n        return 1.0\n    if gauge_mode == \"hard\":\n        return 1.0 - x / length\n    raise ValueError(f\"unknown pressure gauge mode {gauge_mode!r}\")\n\n\ndef psi_base(y, flux, base):\n    \"\"\"Stream function carrying `flux` with zero slope at both walls.\"\"\"\n    if base == \"cubic\":\n        return 0.25 * flux * (3.0 * y - y**3)\n    if base == \"sine\":\n        return 0.5 * flux * jnp.sin(0.5 * jnp.pi * y)\n    raise ValueError(f\"unknown stream_base {base!r}\")\n\n\ndef make_velocity_pressure_field(\n    net_apply, *, length, inlet_mode, lifting_length, gauge_mode\n):\n    \"\"\"Build field_fn(params, xy) -> [u, v, p] for the u-v-p formulation.\"\"\"\n\n    def field_fn(params, xy):\n        x, y = xy[0], xy[1]\n        raw = net_apply(params, normalise(xy, length))\n        wall = phi(y)\n\n        if inlet_mode == \"soft\":\n            u = wall * raw[0]\n            v = wall * raw[1]\n        elif inlet_mode == \"lifting\":\n            lift = _lifting(x, lifting_length)\n            # u(0, y) = phi(y) = 1 - y^2 exactly; v(0, y) = 0 exactly.\n            u = wall * (1.0 + lift * raw[0])\n            v = wall * lift * raw[1]\n        else:\n            raise ValueError(f\"unknown inlet mode {inlet_mode!r}\")\n\n        p = _pressure_gauge_factor(x, length, gauge_mode) * raw[2]\n        return jnp.array([u, v, p])\n\n    return field_fn\n\n\ndef make_stream_function_field(net_apply, *, length, flux, base, gauge_mode):\n    \"\"\"Build field_fn(params, xy) -> [u, v, p] for the stream-function form.\n\n    Continuity is satisfied identically, so there is no continuity loss term.\n    The cost is one extra order of differentiation in the momentum residual:\n    u_xx here is a third derivative of the network.\n    \"\"\"\n\n    def stream(params, xy):\n        y = xy[1]\n        raw = net_apply(params, normalise(xy, length))\n        return psi_base(y, flux, base) + phi(y) ** 2 * raw[0]\n\n    def field_fn(params, xy):\n        gradient = jax.grad(stream, argnums=1)(params, xy)\n        u = gradient[1]\n        v = -gradient[0]\n        raw = net_apply(params, normalise(xy, length))\n        p = _pressure_gauge_factor(xy[0], length, gauge_mode) * raw[1]\n        return jnp.array([u, v, p])\n\n    return field_fn\n\n\ndef make_psi_omega_field(net_apply, *, length, flux, lift):\n    \"\"\"Build the psi-omega pair for SPEC v2 S4.\n\n        psi   = psi_lift(y*) + phi(y*)^2 N_psi(x*, y*)\n        omega = N_omega(x*, y*)                          -- NO hard constraint\n\n    At the walls phi = 0, so psi = psi_lift there and\n    d_y[phi^2 N] = 2 phi phi' N + phi^2 d_y N = 0. Both wall conditions hold\n    exactly for any parameters, which is what pins each wall to its own\n    streamline constant and hence fixes the flux.\n\n    `omega` is deliberately left free. It is coupled only through\n    r_1 = omega + lap psi, which the network can violate -- that residual is\n    what partially restores the falsifiability the pure-psi form destroys.\n    Hard-constraining omega would remove it (SPEC v2 S4, and see S5).\n    \"\"\"\n    lift_fn = lifts.get(lift)\n\n    def stream_fn(params, xy):\n        y = xy[1]\n        raw = net_apply(params, normalise(xy, length))\n        psi = lift_fn(y, flux) + phi(y) ** 2 * raw[0]\n        omega = raw[1]\n        return jnp.array([psi, omega])\n\n    return stream_fn\n\n\ndef make_velocity_from_stream(stream_fn):\n    \"\"\"(u, v) = (dpsi/dy, -dpsi/dx) from a psi-omega stream function.\"\"\"\n\n    def velocity_fn(params, xy):\n        gradient = jax.grad(lambda p, z: stream_fn(p, z)[0], argnums=1)(params, xy)\n        return jnp.array([gradient[1], -gradient[0]])\n\n    return velocity_fn\n",
"src/models/mlp.py": "\"\"\"Baseline tanh MLP and the modified (two-encoder) MLP.\n\nBoth take an already-embedded input vector and return `n_out` raw outputs; the\nhard boundary-condition ansatz is applied outside, in `hard_bc.py`.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\nfrom .dense import apply_dense, init_dense\n\n\ndef init_mlp(key, n_in: int, n_out: int, width: int, depth: int, **rwf):\n    \"\"\"Depth `depth` counts HIDDEN layers, so there are depth+1 dense layers.\"\"\"\n    sizes = [n_in] + [width] * depth + [n_out]\n    keys = jax.random.split(key, len(sizes) - 1)\n    return {\n        \"layers\": [\n            init_dense(k, a, b, **rwf) for k, a, b in zip(keys, sizes[:-1], sizes[1:])\n        ]\n    }\n\n\ndef apply_mlp(params, x):\n    *hidden, last = params[\"layers\"]\n    for layer in hidden:\n        x = jnp.tanh(apply_dense(layer, x))\n    return apply_dense(last, x)\n\n\ndef init_modified_mlp(key, n_in: int, n_out: int, width: int, depth: int, **rwf):\n    \"\"\"Wang et al.'s modified MLP: two input encoders mixed multiplicatively.\n\n        U = tanh(W_u xi + b_u),  V = tanh(W_v xi + b_v)\n        H_1 = tanh(W_1 xi + b_1)\n        Z_k = tanh(W_k H_k + b_k)\n        H_{k+1} = (1 - Z_k) * U + Z_k * V\n\n    The encoders let every hidden layer see the input again, which is what\n    fixes the vanishing input sensitivity of a deep plain MLP.\n    \"\"\"\n    keys = jax.random.split(key, depth + 3)\n    encoder_u = init_dense(keys[0], n_in, width, **rwf)\n    encoder_v = init_dense(keys[1], n_in, width, **rwf)\n    hidden = [init_dense(keys[2], n_in, width, **rwf)]\n    hidden += [init_dense(keys[3 + i], width, width, **rwf) for i in range(depth - 1)]\n    output = init_dense(keys[depth + 2], width, n_out, **rwf)\n    return {\"encoder_u\": encoder_u, \"encoder_v\": encoder_v, \"hidden\": hidden, \"output\": output}\n\n\ndef apply_modified_mlp(params, x):\n    encoded_u = jnp.tanh(apply_dense(params[\"encoder_u\"], x))\n    encoded_v = jnp.tanh(apply_dense(params[\"encoder_v\"], x))\n    h = x\n    for layer in params[\"hidden\"]:\n        z = jnp.tanh(apply_dense(layer, h))\n        h = (1.0 - z) * encoded_u + z * encoded_v\n    return apply_dense(params[\"output\"], h)\n",
"src/physics/__init__.py": "\"\"\"Physics layer: scaling, analytic solutions, residual operators.\n\nThis package must not import from `src.models` or `src.training` (see\nCLAUDE.md, \"Module layout\"). Everything here is a pure function of\nnon-dimensional inputs, with no module-level state.\n\"\"\"\n",
"src/physics/analytic.py": "\"\"\"Analytic solutions for plane Poiseuille flow \u2014 the oracle for Phase 1.\n\nTwo coordinate worlds are kept strictly apart:\n\n* `*_si` functions take SI arguments and are used only by the scaling tests.\n* everything else is non-dimensional (lengths in units of the half-height h,\n  velocities in units of U_max, pressure per `Scales.pressure_scale`).\n\nSign convention: the flow is in +x, driven by a favourable pressure gradient\nG = -dp/dx > 0.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .scaling import PressureScaling\n\n#: Non-dimensional flux of the reference profile u* = 1 - y*^2 over y* in\n#: [-1, 1]:  int_{-1}^{1} (1 - y^2) dy = 4/3.\nFLUX_ND = 4.0 / 3.0\n\n#: Non-dimensional full channel height (walls at y* = +-1).\nCHANNEL_HEIGHT_ND = 2.0\n\n\n# --------------------------------------------------------------------------- #\n# SI forms\n# --------------------------------------------------------------------------- #\n\n\ndef u_si(y, pressure_gradient, h, mu):\n    \"\"\"u(y) = (G / 2 mu) (h^2 - y^2) for G = -dp/dx > 0.\"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    return (pressure_gradient / (2.0 * mu)) * (h**2 - y**2)\n\n\ndef u_max_si(pressure_gradient, h, mu):\n    \"\"\"U_max = G h^2 / (2 mu).\"\"\"\n    return pressure_gradient * h**2 / (2.0 * mu)\n\n\ndef flow_rate_si(pressure_gradient, h, mu):\n    \"\"\"Q = 2 G h^3 / (3 mu) per unit depth.\n\n    NOTE THE EXPONENT. In 2D the flux scales as h^3, not as r^4 (Hagen-\n    Poiseuille, circular pipe). Phase 3 geometry optimisation depends on this\n    exponent and the two must never be mixed. See SPEC section 1.\n    \"\"\"\n    return 2.0 * pressure_gradient * h**3 / (3.0 * mu)\n\n\ndef pressure_gradient_si(u_max, h, mu):\n    \"\"\"Inverse of `u_max_si`: G = 2 mu U_max / h^2.\"\"\"\n    return 2.0 * mu * u_max / h**2\n\n\n# --------------------------------------------------------------------------- #\n# Non-dimensional forms\n# --------------------------------------------------------------------------- #\n\n\ndef u_nd(y):\n    \"\"\"u*(y*) = 1 - y*^2.\"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    return 1.0 - y**2\n\n\ndef v_nd(y):\n    \"\"\"v* = 0 everywhere. The PINN has to discover this, not be told it.\"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    return np.zeros_like(y)\n\n\ndef developed_profile_nd(y, flux):\n    \"\"\"Fully developed parabolic profile carrying a prescribed flux.\n\n    Mass conservation fixes the amplitude:  int_{-1}^{1} u* dy* = flux, and for\n    u* = A (1 - y*^2) that integral is 4A/3, so A = 3*flux/4.\n\n    For Case A (flux = 4/3) this is exactly `u_nd`. For Case B, whose uniform\n    inlet u* = 1 carries flux 2, the developed centreline velocity is 3/2 --\n    NOT 1. See docs/spec-deviations.md.\n    \"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    return 0.75 * flux * (1.0 - y**2)\n\n\ndef pressure_gradient_nd(reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"Constant dp*/dx* of the fully developed non-dimensional solution.\n\n    Substituting u* = 1 - y*^2, v* = 0 into the momentum equation:\n\n    viscous scaling   Re (u.grad)u = -grad p* + lap u*  ->  dp*/dx* = -2\n    inertial scaling  (u.grad)u = -grad p* + lap(u*)/Re ->  dp*/dx* = -2/Re\n\n    The ratio Re between the two is exactly the ratio of the pressure scales,\n    which is checked in the scaling tests.\n    \"\"\"\n    if pressure_scaling == \"viscous\":\n        return -2.0\n    if pressure_scaling == \"inertial\":\n        return -2.0 / reynolds\n    raise ValueError(f\"unknown pressure_scaling {pressure_scaling!r}\")\n\n\ndef p_nd(x, domain_length: float, reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"p*(x*) with the gauge p*(x* = L*) = 0.\n\n    Pressure is unique only up to a constant. Pinning it at the outlet is the\n    gauge fix used throughout Phase 1; without it the pressure L2 error is\n    meaningless (SPEC section 9).\n    \"\"\"\n    x = np.asarray(x, dtype=np.float64)\n    dpdx = pressure_gradient_nd(reynolds, pressure_scaling)\n    return dpdx * (x - domain_length)\n\n\ndef stream_function_nd(y, flux=FLUX_ND):\n    \"\"\"psi*(y*) with u* = dpsi/dy, for the fully developed profile.\n\n    psi = A (y - y^3/3) with A = 3*flux/4, gauged so psi(0) = 0. The wall values\n    differ by exactly `flux`, which is what makes a stream-function ansatz that\n    is constant on both walls unable to represent a channel with net throughflow\n    (see src/models/hard_bc.py).\n    \"\"\"\n    y = np.asarray(y, dtype=np.float64)\n    return 0.75 * flux * (y - y**3 / 3.0)\n\n\n# Entrance length now lives in `conventions.py` (SPEC v2 S1).\n#\n# The previous `entrance_length_correlation_nd` here took a bare\n# `bulk_velocity_nd` argument, and Case B called it with 1.0 (the SPEC v1 inlet\n# amplitude) where the parabolic mean 2/3 was required. That inflated the\n# correlation by 25% at Re_h = 10 and is exactly the unsuffixed-Reynolds bug S1\n# is written to eliminate. Use `conventions.entrance_length_over_h(re_h, corr)`,\n# which converts internally and cannot be called with the wrong convention.\n",
"src/physics/conventions.py": "\"\"\"Reynolds-number and length-scale conventions (SPEC v2, S1).\n\nTwo Reynolds numbers live in this project and they are NOT equal.\n\nSolver convention -- half-height `h`, centreline velocity `U_max`:\n\n    Re_h = rho * U_max * h / mu\n\nCorrelation convention -- hydraulic diameter `D_h`, mean velocity `U_mean`:\n\n    Re_Dh = rho * U_mean * D_h / mu\n\nFor a wide plane channel, D_h = 4A/P = 4(2hw)/(2w) = 4h, and for the parabolic\nprofile U_mean = (2/3) U_max, so\n\n    Re_Dh = (8/3) Re_h\n\nand the reference LENGTHS differ by a factor of four (D_h = 4h). Chained\nthrough a correlation that is linear in Re, that is roughly an order of\nmagnitude -- which is the signature that showed up as a 289% entrance-length\nmiss while every other quantity from the same solution was within a few\npercent.\n\nRule enforced by naming throughout this module: every function that consumes a\nReynolds number says WHICH ONE in its name. A parameter called plain `re` is a\nbug.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n#: Re_Dh / Re_h for a wide plane channel. Derivation in the module docstring.\nRE_DH_OVER_RE_H = 8.0 / 3.0\n\n#: D_h / h for a wide plane channel.\nD_H_OVER_H = 4.0\n\n#: U_mean / U_max for the parabolic profile.\nU_MEAN_OVER_U_MAX = 2.0 / 3.0\n\n\nclass MissingCitationError(ValueError):\n    \"\"\"Raised when a correlation is declared without a source citation.\"\"\"\n\n\n@dataclass(frozen=True)\nclass EntranceLengthCorrelation:\n    \"\"\"Blended Durst-type correlation, in HYDRAULIC-DIAMETER units.\n\n        L_e / D_h = [ c0^n + (c1 * Re_Dh)^n ]^(1/n)\n\n    The constants differ between pipe and plane-channel geometry and must be\n    read from the source, never assumed. `citation` is mandatory: a wrong\n    constant here is invisible in the output, whereas a missing citation is\n    not.\n    \"\"\"\n\n    c0: float\n    c1: float\n    n: float\n    citation: str\n\n    def __post_init__(self):\n        if not isinstance(self.citation, str) or not self.citation.strip():\n            raise MissingCitationError(\n                \"an entrance-length correlation must carry a non-empty \"\n                \"'citation'. The constants differ between pipe and plane-\"\n                \"channel geometry and a wrong one is invisible in the output.\"\n            )\n        for name in (\"c0\", \"c1\", \"n\"):\n            value = getattr(self, name)\n            if not np.isfinite(value) or value <= 0.0:\n                raise ValueError(\n                    f\"correlation.{name} must be finite and positive, got {value!r}\"\n                )\n\n    def length_over_dh(self, reynolds_dh):\n        \"\"\"L_e / D_h.\"\"\"\n        return (self.c0**self.n + (self.c1 * reynolds_dh) ** self.n) ** (1.0 / self.n)\n\n\n# --------------------------------------------------------------------------- #\n# Reynolds numbers\n# --------------------------------------------------------------------------- #\n\n\ndef reynolds_h(rho, u_max, h, mu):\n    \"\"\"Solver convention: Re_h = rho U_max h / mu.\"\"\"\n    return rho * u_max * h / mu\n\n\ndef reynolds_dh(rho, u_mean, d_h, mu):\n    \"\"\"Correlation convention: Re_Dh = rho U_mean D_h / mu.\"\"\"\n    return rho * u_mean * d_h / mu\n\n\ndef re_h_to_re_dh(re_h):\n    \"\"\"Re_Dh = (8/3) Re_h. Exact for a wide plane channel.\"\"\"\n    return RE_DH_OVER_RE_H * re_h\n\n\ndef re_dh_to_re_h(re_dh):\n    \"\"\"Inverse of `re_h_to_re_dh`.\"\"\"\n    return re_dh / RE_DH_OVER_RE_H\n\n\n# --------------------------------------------------------------------------- #\n# Entrance length\n# --------------------------------------------------------------------------- #\n\n\ndef entrance_length_over_h(re_h, correlation: EntranceLengthCorrelation):\n    \"\"\"L_e / h, converting conventions internally.\n\n    Takes the SOLVER Reynolds number, converts to the correlation's convention,\n    evaluates, and converts the length back:  L_e/h = 4 * (L_e/D_h).\n\n    Returning in units of `h` means call sites never touch D_h and cannot get\n    the factor of four wrong.\n    \"\"\"\n    return D_H_OVER_H * correlation.length_over_dh(re_h_to_re_dh(re_h))\n",
"src/physics/derivatives.py": "\"\"\"Autodiff assembly of the derivative bundle the residual operator consumes.\n\nKept separate from `residuals.py` so that the residual algebra can be unit\ntested against hand-derived expressions with no autodiff in the loop, and the\nautodiff itself can be checked against finite differences independently\n(SPEC test 5).\n\nA \"field function\" here is any callable `f(xy) -> array of shape (3,)` giving\n(u, v, p) at the single point `xy = [x, y]`. It may internally be a network, a\nhard-BC-wrapped network, an analytic expression, or a stream-function\nconstruction that differentiates a scalar potential -- this module does not\ncare, and takes second derivatives of whatever it is handed.\n\nTwo implementations of the same mathematics live here:\n\n`field_derivatives`\n    One point, nested `jacfwd`. Short, obviously correct, used as the reference.\n\n`batched_field_derivatives`\n    A whole (N, 2) array at once. Instead of vmapping the single-point version,\n    it differentiates the ALREADY-BATCHED field function along a constant\n    tangent direction, twice. Each output row depends only on its own input\n    row, so a batch-wide directional derivative gives every point's derivative\n    in that direction in one pass -- and every pass stays a full-size matrix\n    product rather than a vmapped matrix-vector product.\n\n    Measured on the baseline config (width 128, depth 4, 4096 points): the\n    batched form costs 195 ms per loss-and-gradient against 460 ms for the\n    vmapped one, for bitwise-comparable values. The reverse-mode pass over the\n    vmapped nested `jacfwd` is the expensive part -- it stores a tangent copy\n    of every activation for every point -- and the batched form cuts the number\n    of those copies. `test_batched_derivatives_match_single_point` pins the two\n    together so the fast path cannot drift from the reference.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\nfrom .residuals import Derivatives\n\n#: Unit tangent directions in (x, y).\n_EX = jnp.array([1.0, 0.0])\n_EY = jnp.array([0.0, 1.0])\n\n\ndef field_derivatives(field_fn, xy) -> Derivatives:\n    \"\"\"Value, first and second derivatives of (u, v, p) at one point.\n\n    Forward-mode throughout: the input is 2-dimensional and the output is\n    3-dimensional, so forward mode is the cheaper direction for both the\n    Jacobian and the Hessian.\n    \"\"\"\n\n    def value_and_jacobian(point):\n        value = field_fn(point)\n        jacobian = jax.jacfwd(field_fn)(point)\n        # `has_aux` lets the outer jacfwd differentiate the Jacobian while\n        # handing back the value and Jacobian it already computed, instead of\n        # recomputing them in a separate pass.\n        return jacobian, (value, jacobian)\n\n    hessian, (value, jacobian) = jax.jacfwd(value_and_jacobian, has_aux=True)(xy)\n\n    return _assemble(value, jacobian[:, 0], jacobian[:, 1], hessian[:, 0, 0], hessian[:, 1, 1])\n\n\ndef _assemble(value, d_dx, d_dy, d2_dx2, d2_dy2) -> Derivatives:\n    \"\"\"Pack directional derivatives into the bundle. Index 0=u, 1=v, 2=p.\"\"\"\n    return Derivatives(\n        u=value[..., 0], v=value[..., 1], p=value[..., 2],\n        u_x=d_dx[..., 0], v_x=d_dx[..., 1], p_x=d_dx[..., 2],\n        u_y=d_dy[..., 0], v_y=d_dy[..., 1], p_y=d_dy[..., 2],\n        u_xx=d2_dx2[..., 0], v_xx=d2_dx2[..., 1],\n        u_yy=d2_dy2[..., 0], v_yy=d2_dy2[..., 1],\n    )\n\n\ndef _directional(batched_fn, points, direction):\n    \"\"\"(value, first, second) derivatives along a constant `direction`.\n\n    The inner `jvp` returns the value alongside the first derivative, and the\n    outer one differentiates that pair, so a single call yields all three\n    without a separate primal evaluation.\n    \"\"\"\n    tangents = jnp.broadcast_to(direction, points.shape)\n\n    def value_and_first(pts):\n        return jax.jvp(batched_fn, (pts,), (tangents,))\n\n    (value, first), (_, second) = jax.jvp(value_and_first, (points,), (tangents,))\n    return value, first, second\n\n\ndef batched_directional_derivatives(vector_fn, points):\n    \"\"\"(value, d/dx, d/dy, d2/dx2, d2/dy2) for any vector-valued `f(xy)`.\n\n    Shared by the velocity-pressure bundle and the psi-omega bundle: both need\n    exactly the value, both first derivatives and both unmixed second\n    derivatives of a small vector of outputs, and neither needs the cross\n    derivative.\n    \"\"\"\n    batched_fn = jax.vmap(vector_fn)\n    value, d_dx, d2_dx2 = _directional(batched_fn, points, _EX)\n    _, d_dy, d2_dy2 = _directional(batched_fn, points, _EY)\n    return value, d_dx, d_dy, d2_dx2, d2_dy2\n\n\ndef batched_field_derivatives(field_fn, points) -> Derivatives:\n    \"\"\"Derivative bundle for an (N, 2) array of points.\"\"\"\n    return _assemble(*batched_directional_derivatives(field_fn, points))\n\n\ndef batched_stream_derivatives(stream_fn, points):\n    \"\"\"Derivative bundle for the psi-omega system (SPEC v2 S4).\n\n    `stream_fn(xy) -> [psi, omega]`. Only SECOND derivatives are taken, of both\n    outputs -- the whole point of solving the coupled second-order system\n    rather than substituting omega = -lap psi is to avoid fourth-order autodiff\n    through tanh.\n    \"\"\"\n    from .streamfunction import StreamDerivatives\n\n    value, d_dx, d_dy, d2_dx2, d2_dy2 = batched_directional_derivatives(\n        stream_fn, points\n    )\n    return StreamDerivatives(\n        psi_x=d_dx[..., 0], psi_y=d_dy[..., 0],\n        psi_xx=d2_dx2[..., 0], psi_yy=d2_dy2[..., 0],\n        omega=value[..., 1],\n        omega_x=d_dx[..., 1], omega_y=d_dy[..., 1],\n        omega_xx=d2_dx2[..., 1], omega_yy=d2_dy2[..., 1],\n    )\n\n\ndef field_and_jacobian(field_fn, xy):\n    \"\"\"(value, jacobian) at one point, without the Hessian.\n\n    Boundary terms (outlet du/dx, dv/dx) need only first derivatives; computing\n    the full bundle there would triple the cost for nothing.\n    \"\"\"\n    return field_fn(xy), jax.jacfwd(field_fn)(xy)\n\n\ndef batched_field_and_jacobian(field_fn, points):\n    \"\"\"(values (N, 3), jacobians (N, 3, 2)) for an (N, 2) array of points.\"\"\"\n    batched_fn = jax.vmap(field_fn)\n    value, d_dx = jax.jvp(batched_fn, (points,), (jnp.broadcast_to(_EX, points.shape),))\n    _, d_dy = jax.jvp(batched_fn, (points,), (jnp.broadcast_to(_EY, points.shape),))\n    return value, jnp.stack([d_dx, d_dy], axis=-1)\n",
"src/physics/optimum.py": "\"\"\"Analytic optimum for the Phase 3 objective (SPEC v2, S8).\n\nObjective: minimise total hydraulic resistance at fixed total channel area.\nThis needs no metabolic-cost coefficient, which is why it is preferable to a\nliteral Murray formulation here.\n\nPlane channel, per unit depth, half-height `h`, length `L`:\n\n    Q = 2 G h^3 / (3 mu)      =>      R = dp/Q = 3 mu L / (2 h^3)\n\nNOTE THE EXPONENT. R ~ h^-3 and area ~ h for a 2D slot; the circular pipe has\nR ~ r^-4 and volume ~ r^2. Mixing them silently is the Phase 3 hazard the SPEC\ncalls out, and it is what makes the answer below an exponent of 2 rather than\nMurray's 3.\n\nTwo independent derivation routes are implemented and must agree; that\nagreement is asserted as a test rather than assumed.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n#: Exponent pairs (n, m) for `R ~ h^-n` and `area or volume ~ h^m`.\nPLANE_CHANNEL = (3, 1)\nCIRCULAR_PIPE = (4, 2)\n\n\ndef resistance_plane_channel(mu, length, half_height):\n    \"\"\"R = 3 mu L / (2 h^3), per unit depth.\"\"\"\n    return 3.0 * mu * length / (2.0 * half_height**3)\n\n\ndef flow_rate_plane_channel(pressure_gradient, mu, half_height):\n    \"\"\"Q = 2 G h^3 / (3 mu), per unit depth. Inverse of `resistance_plane_channel`.\"\"\"\n    return 2.0 * pressure_gradient * half_height**3 / (3.0 * mu)\n\n\ndef optimum_ratio_by_lagrange(mu=1.0, length_parent=1.0, length_daughter=1.0):\n    \"\"\"h_0/h_1 from stationarity of R_tot + lambda (A - A_0).\n\n    Parent `(h_0, L_0)` feeding two identical daughters `(h_1, L_1)` in\n    parallel:\n\n        R_tot = (3 mu / 2) [ L_0/h_0^3 + L_1/(2 h_1^3) ]\n        A     = 2 h_0 L_0 + 4 h_1 L_1\n\n    Stationarity gives\n\n        dL/dh_0 = -9 mu L_0 / (2 h_0^4) + 2 lambda L_0 = 0  =>  h_0^4 = 9 mu / (4 lambda)\n        dL/dh_1 = -9 mu L_1 / (4 h_1^4) + 4 lambda L_1 = 0  =>  h_1^4 = 9 mu / (16 lambda)\n\n    so h_0^4 / h_1^4 = 4 and h_0/h_1 = 4^(1/4) = sqrt(2), independent of mu,\n    lambda and both lengths. The lengths are accepted as arguments precisely so\n    a test can confirm that independence.\n    \"\"\"\n    # Both stationarity conditions carry a factor L_i that cancels, so the\n    # ratio is computed from the exponents alone. Written out rather than\n    # returned as a literal, so the derivation is executable.\n    lam = 1.0  # any positive value; it cancels\n    h0_fourth = 9.0 * mu / (4.0 * lam)\n    h1_fourth = 9.0 * mu / (16.0 * lam)\n    return float((h0_fourth / h1_fourth) ** 0.25)\n\n\ndef flow_exponent(n, m):\n    \"\"\"Q ~ h^((n+m)/2) from minimising A Q^2 h^-n + B h^m.\n\n    The cost is dissipation plus a maintenance term. Stationarity in `h`:\n\n        -n A Q^2 h^(-n-1) + m B h^(m-1) = 0   =>   Q^2 ~ h^(n+m)\n\n    Plane channel (n, m) = (3, 1) gives Q ~ h^2; circular pipe (4, 2) gives\n    Q ~ r^3, which is Murray's law and the check that this route is right.\n    \"\"\"\n    return (n + m) / 2.0\n\n\ndef optimum_ratio_by_flow_exponent(n=3, m=1):\n    \"\"\"h_0/h_1 for a symmetric bifurcation, from the flow exponent.\n\n    With `Q ~ h^k` and the parent flux split equally between two daughters,\n    `h_0^k = 2 h_1^k`, so `h_0/h_1 = 2^(1/k)`. For the plane channel k = 2 and\n    the ratio is sqrt(2).\n    \"\"\"\n    k = flow_exponent(n, m)\n    return float(2.0 ** (1.0 / k))\n\n\ndef area_law_exponent(n=3, m=1):\n    \"\"\"The exponent `k` in the conserved quantity `h_0^k = h_1^k + h_2^k`.\n\n    Plane channel: k = 2, i.e. h_0^2 = h_1^2 + h_2^2 -- the 2D analogue of\n    Murray's law. The cube law is specific to circular cross-sections.\n    \"\"\"\n    return flow_exponent(n, m)\n\n\n#: The Phase 3 oracle in the fully developed limit. The DEVIATION of the PINN\n#: optimum from this value is the Phase 3 result; the law itself is not.\nOPTIMUM_HEIGHT_RATIO = np.sqrt(2.0)\n",
"src/physics/residuals.py": "\"\"\"Unweighted residuals of the 2D steady incompressible Navier-Stokes system.\n\nOne small function per governing-equation term, so each term can be unit-tested\non its own (CLAUDE.md, \"Code style\"). Every function here is pure, takes\nnon-dimensional inputs, and returns an UNWEIGHTED residual. All loss weighting\nlives in src/training/weights.py and nowhere else.\n\nNon-dimensional forms\n---------------------\ninertial pressure scaling (p* = p / (rho U^2)):\n\n    R_x = u u_x + v u_y + p_x - (1/Re) (u_xx + u_yy)\n    R_y = u v_x + v v_y + p_y - (1/Re) (v_xx + v_yy)\n\nviscous pressure scaling (p* = p h / (mu U)):\n\n    R_x = Re (u u_x + v u_y) + p_x - (u_xx + u_yy)\n    R_y = Re (u v_x + v v_y) + p_y - (v_xx + v_yy)\n\ncontinuity (both):\n\n    R_c = u_x + v_y\n\nThe two momentum forms are the same equation multiplied by Re (because\np*_viscous = Re p*_inertial), so they have identical solutions and different\nconditioning. That is exactly why the SPEC makes the choice a config option.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import NamedTuple\n\nfrom .scaling import PressureScaling\n\n\nclass Derivatives(NamedTuple):\n    \"\"\"Field values and the derivatives the residual operator needs.\n\n    A NamedTuple so it is a JAX pytree and can be vmapped/jitted directly.\n    All entries are non-dimensional.\n    \"\"\"\n\n    u: object\n    v: object\n    p: object\n    u_x: object\n    u_y: object\n    v_x: object\n    v_y: object\n    p_x: object\n    p_y: object\n    u_xx: object\n    u_yy: object\n    v_xx: object\n    v_yy: object\n\n\n# --------------------------------------------------------------------------- #\n# Individual terms\n# --------------------------------------------------------------------------- #\n\n\ndef convective_x(u, v, u_x, u_y):\n    \"\"\"(u . grad) u, x-component.\"\"\"\n    return u * u_x + v * u_y\n\n\ndef convective_y(u, v, v_x, v_y):\n    \"\"\"(u . grad) u, y-component.\"\"\"\n    return u * v_x + v * v_y\n\n\ndef laplacian(f_xx, f_yy):\n    \"\"\"grad^2 f.\"\"\"\n    return f_xx + f_yy\n\n\ndef divergence(u_x, v_y):\n    \"\"\"div u = u_x + v_y. This is the continuity residual, unweighted.\"\"\"\n    return u_x + v_y\n\n\n# --------------------------------------------------------------------------- #\n# Assembled momentum residuals\n# --------------------------------------------------------------------------- #\n\n\ndef _scaling_coefficients(reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"(convective coefficient, viscous coefficient) for the chosen scaling.\"\"\"\n    if pressure_scaling == \"inertial\":\n        return 1.0, 1.0 / reynolds\n    if pressure_scaling == \"viscous\":\n        return reynolds, 1.0\n    raise ValueError(\n        f\"unknown pressure_scaling {pressure_scaling!r}; \"\n        \"expected 'inertial' or 'viscous'\"\n    )\n\n\ndef momentum_x(d: Derivatives, reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"x-momentum residual, unweighted.\"\"\"\n    a_conv, a_visc = _scaling_coefficients(reynolds, pressure_scaling)\n    return (\n        a_conv * convective_x(d.u, d.v, d.u_x, d.u_y)\n        + d.p_x\n        - a_visc * laplacian(d.u_xx, d.u_yy)\n    )\n\n\ndef momentum_y(d: Derivatives, reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"y-momentum residual, unweighted.\"\"\"\n    a_conv, a_visc = _scaling_coefficients(reynolds, pressure_scaling)\n    return (\n        a_conv * convective_y(d.u, d.v, d.v_x, d.v_y)\n        + d.p_y\n        - a_visc * laplacian(d.v_xx, d.v_yy)\n    )\n\n\ndef continuity(d: Derivatives):\n    \"\"\"Continuity residual, unweighted.\"\"\"\n    return divergence(d.u_x, d.v_y)\n\n\ndef navier_stokes(d: Derivatives, reynolds: float, pressure_scaling: PressureScaling):\n    \"\"\"All three residuals as a tuple (R_x, R_y, R_c), unweighted.\"\"\"\n    return (\n        momentum_x(d, reynolds, pressure_scaling),\n        momentum_y(d, reynolds, pressure_scaling),\n        continuity(d),\n    )\n",
"src/physics/scaling.py": "\"\"\"Explicit, invertible SI <-> non-dimensional transform.\n\nEvery physics quantity in this project lives in one of two worlds and the\ntransform between them is written down here once. Nothing else in the codebase\nis allowed to hold an SI-valued viscosity, density or channel height.\n\nReference quantities\n--------------------\nlength    h        channel half-height\nvelocity  U_max    centreline velocity of the fully developed profile\npressure  depends on the chosen scaling:\n            viscous   p_ref = mu * U_max / h        (use for Re <~ 1)\n            inertial  p_ref = rho * U_max**2        (use for Re > 1)\n\nReynolds number\n---------------\n    Re = rho * U_max * h / mu\n\nThe two pressure scalings are related by exactly a factor Re:\n\n    p*_viscous = p*_inertial * Re\n\nwhich is a useful consistency check and is tested.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Literal\n\nimport numpy as np\n\nPressureScaling = Literal[\"viscous\", \"inertial\"]\n\n_VALID_PRESSURE_SCALINGS = (\"viscous\", \"inertial\")\n\n#: Relative tolerance for a \"machine precision\" SI -> non-dim -> SI round trip.\n#: A round trip is one division followed by one multiplication, each correctly\n#: rounded, so the relative error is bounded by ~2 eps. 4 eps is that bound with\n#: one bit of headroom. This is a derived bound, not a fitted tolerance: it must\n#: never be widened to make a test pass.\nROUNDTRIP_RTOL = 4.0 * float(np.finfo(np.float64).eps)\n\n\n@dataclass(frozen=True)\nclass Scales:\n    \"\"\"Reference quantities defining the non-dimensionalisation.\n\n    Parameters are SI. This is the only object in the codebase that is allowed\n    to carry SI values; it converts them and hands non-dimensional numbers to\n    everything else.\n\n    Attributes\n    ----------\n    h : channel half-height [m]\n    u_max : centreline velocity of the fully developed profile [m/s]\n    rho : density [kg/m^3]\n    mu : dynamic viscosity [Pa s]\n    pressure_scaling : \"viscous\" or \"inertial\"\n    \"\"\"\n\n    h: float\n    u_max: float\n    rho: float\n    mu: float\n    pressure_scaling: PressureScaling\n\n    def __post_init__(self) -> None:\n        for name in (\"h\", \"u_max\", \"rho\", \"mu\"):\n            value = getattr(self, name)\n            if not np.isfinite(value):\n                raise ValueError(f\"Scales.{name} must be finite, got {value!r}\")\n            if value <= 0.0:\n                raise ValueError(f\"Scales.{name} must be positive, got {value!r}\")\n        if self.pressure_scaling not in _VALID_PRESSURE_SCALINGS:\n            raise ValueError(\n                f\"pressure_scaling must be one of {_VALID_PRESSURE_SCALINGS}, \"\n                f\"got {self.pressure_scaling!r}\"\n            )\n\n    # -- reference quantities -------------------------------------------------\n\n    @property\n    def reynolds(self) -> float:\n        \"\"\"Re = rho * U_max * h / mu.\"\"\"\n        return self.rho * self.u_max * self.h / self.mu\n\n    @property\n    def length_scale(self) -> float:\n        return self.h\n\n    @property\n    def velocity_scale(self) -> float:\n        return self.u_max\n\n    @property\n    def pressure_scale(self) -> float:\n        \"\"\"Pressure reference p_ref such that p* = p / p_ref.\"\"\"\n        if self.pressure_scaling == \"viscous\":\n            return self.mu * self.u_max / self.h\n        return self.rho * self.u_max**2\n\n    # -- SI -> non-dimensional ------------------------------------------------\n\n    def length_to_nondim(self, x):\n        return np.asarray(x, dtype=np.float64) / self.length_scale\n\n    def velocity_to_nondim(self, u):\n        return np.asarray(u, dtype=np.float64) / self.velocity_scale\n\n    def pressure_to_nondim(self, p):\n        return np.asarray(p, dtype=np.float64) / self.pressure_scale\n\n    def pressure_gradient_to_nondim(self, dpdx):\n        \"\"\"dp*/dx* = (dp/dx) * h / p_ref.\"\"\"\n        return (\n            np.asarray(dpdx, dtype=np.float64)\n            * self.length_scale\n            / self.pressure_scale\n        )\n\n    # -- non-dimensional -> SI ------------------------------------------------\n\n    def length_to_si(self, x_nd):\n        return np.asarray(x_nd, dtype=np.float64) * self.length_scale\n\n    def velocity_to_si(self, u_nd):\n        return np.asarray(u_nd, dtype=np.float64) * self.velocity_scale\n\n    def pressure_to_si(self, p_nd):\n        return np.asarray(p_nd, dtype=np.float64) * self.pressure_scale\n\n    def pressure_gradient_to_si(self, dpdx_nd):\n        return (\n            np.asarray(dpdx_nd, dtype=np.float64)\n            * self.pressure_scale\n            / self.length_scale\n        )\n\n\ndef scales_from_reynolds(\n    reynolds: float,\n    pressure_scaling: PressureScaling,\n    *,\n    h: float = 1.0,\n    u_max: float = 1.0,\n    rho: float = 1.0,\n) -> Scales:\n    \"\"\"Build `Scales` with a prescribed Reynolds number.\n\n    mu is solved for rather than defaulted, so the returned object satisfies\n    `scales.reynolds == reynolds` up to rounding. Used by configs that specify\n    Re directly instead of four SI quantities.\n    \"\"\"\n    if not np.isfinite(reynolds) or reynolds <= 0.0:\n        raise ValueError(f\"reynolds must be finite and positive, got {reynolds!r}\")\n    mu = rho * u_max * h / reynolds\n    return Scales(\n        h=h, u_max=u_max, rho=rho, mu=mu, pressure_scaling=pressure_scaling\n    )\n\n\ndef recommended_pressure_scaling(reynolds: float) -> PressureScaling:\n    \"\"\"The SPEC's recommendation: viscous for Re <= 1, inertial above.\n\n    Advisory only. The active scaling always comes from the config; this\n    function exists so a run can *report* that it deviates from the\n    recommendation, never so it can silently override the config.\n    \"\"\"\n    return \"viscous\" if reynolds <= 1.0 else \"inertial\"\n",
"src/physics/streamfunction.py": "\"\"\"Coupled psi-omega system and post-hoc pressure recovery (SPEC v2, S4).\n\n    u = dpsi/dy,   v = -dpsi/dx      (continuity satisfied identically)\n    omega = -lap psi                 (vorticity)\n\nTaking the curl of the momentum equation eliminates pressure. The system is\nsolved as TWO COUPLED SECOND-ORDER equations in two network outputs\n`(psi, omega)`:\n\n    r_1 = omega + lap psi                                             = 0\n    r_2 = psi_y omega_x - psi_x omega_y - lap(omega)/Re               = 0\n\nrather than by substituting `omega = -lap psi` into r_2, which would give a\nFOURTH-ORDER equation in psi. Fourth-order autodiff through tanh would then set\nthe precision floor -- and Phase 1 already measured second derivatives through\ntanh costing an order of magnitude of achievable accuracy.\n\n`omega` carries no hard constraint and is coupled only through r_1. That is\ndeliberate: it partially restores the falsifiability that the pure-psi form\ndestroys, because r_1 is a real residual the network can violate.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import NamedTuple\n\n\nclass StreamDerivatives(NamedTuple):\n    \"\"\"Derivative bundle for the psi-omega system. All non-dimensional.\"\"\"\n\n    psi_x: object\n    psi_y: object\n    psi_xx: object\n    psi_yy: object\n    omega: object\n    omega_x: object\n    omega_y: object\n    omega_xx: object\n    omega_yy: object\n\n\ndef laplacian(f_xx, f_yy):\n    return f_xx + f_yy\n\n\ndef vorticity_residual(d: StreamDerivatives):\n    \"\"\"r_1 = omega + lap psi. Unweighted.\"\"\"\n    return d.omega + laplacian(d.psi_xx, d.psi_yy)\n\n\ndef vorticity_transport_residual(d: StreamDerivatives, reynolds: float):\n    \"\"\"r_2 = psi_y omega_x - psi_x omega_y - lap(omega)/Re. Unweighted.\n\n    The convective part is (u . grad) omega with u = psi_y, v = -psi_x:\n\n        u omega_x + v omega_y = psi_y omega_x - psi_x omega_y\n    \"\"\"\n    return (\n        d.psi_y * d.omega_x\n        - d.psi_x * d.omega_y\n        - laplacian(d.omega_xx, d.omega_yy) / reynolds\n    )\n\n\ndef residuals(d: StreamDerivatives, reynolds: float):\n    \"\"\"(r_1, r_2), unweighted.\"\"\"\n    return vorticity_residual(d), vorticity_transport_residual(d, reynolds)\n\n\ndef velocity_from_stream(psi_x, psi_y):\n    \"\"\"(u, v) = (psi_y, -psi_x).\"\"\"\n    return psi_y, -psi_x\n\n\n# --------------------------------------------------------------------------- #\n# Post-hoc pressure recovery (SPEC v2, S4)\n# --------------------------------------------------------------------------- #\n\nimport jax  # noqa: E402\nimport jax.numpy as jnp  # noqa: E402\nimport numpy as np  # noqa: E402\n\nfrom .derivatives import batched_directional_derivatives  # noqa: E402\nfrom .residuals import _scaling_coefficients  # noqa: E402\n\n\ndef pressure_gradient(velocity_fn, points, reynolds, pressure_scaling):\n    \"\"\"(p_x, p_y) from the momentum equation, at an (N, 2) array of points.\n\n    Rearranging the momentum residual to zero:\n\n        p_x = a_visc (u_xx + u_yy) - a_conv (u u_x + v u_y)\n        p_y = a_visc (v_xx + v_yy) - a_conv (u v_x + v v_y)\n\n    with (a_conv, a_visc) the scaling coefficients, so this works unchanged\n    under either pressure scaling. `velocity_fn(xy) -> [u, v]`.\n    \"\"\"\n    a_conv, a_visc = _scaling_coefficients(reynolds, pressure_scaling)\n    value, d_dx, d_dy, d2_dx2, d2_dy2 = batched_directional_derivatives(\n        velocity_fn, points\n    )\n    u, v = value[..., 0], value[..., 1]\n    u_x, v_x = d_dx[..., 0], d_dx[..., 1]\n    u_y, v_y = d_dy[..., 0], d_dy[..., 1]\n    lap_u = d2_dx2[..., 0] + d2_dy2[..., 0]\n    lap_v = d2_dx2[..., 1] + d2_dy2[..., 1]\n\n    p_x = a_visc * lap_u - a_conv * (u * u_x + v * u_y)\n    p_y = a_visc * lap_v - a_conv * (u * v_x + v * v_y)\n    return p_x, p_y\n\n\ndef _simpson(values, spacing):\n    \"\"\"Composite Simpson on an odd number of equally spaced samples.\"\"\"\n    n = len(values) - 1\n    if n % 2 != 0:\n        raise ValueError(\"Simpson's rule needs an even number of intervals\")\n    weights = np.ones(n + 1)\n    weights[1:-1:2] = 4.0\n    weights[2:-1:2] = 2.0\n    return float(spacing / 3.0 * np.dot(weights, values))\n\n\ndef _segment(velocity_fn, start, end, reynolds, pressure_scaling, nodes):\n    \"\"\"Integral of grad p . dl along the straight segment start -> end.\"\"\"\n    if nodes % 2 == 0:\n        nodes += 1\n    t = np.linspace(0.0, 1.0, nodes)\n    path = np.outer(1.0 - t, np.asarray(start)) + np.outer(t, np.asarray(end))\n    p_x, p_y = pressure_gradient(\n        velocity_fn, jnp.asarray(path), reynolds, pressure_scaling\n    )\n    delta = np.asarray(end, dtype=np.float64) - np.asarray(start, dtype=np.float64)\n    integrand = np.asarray(p_x) * delta[0] + np.asarray(p_y) * delta[1]\n    return _simpson(integrand, 1.0 / (nodes - 1))\n\n\ndef recover_pressure(\n    velocity_fn, target, gauge_point, reynolds, pressure_scaling, *, path=\"x_first\",\n    nodes=201,\n):\n    \"\"\"p at `target` (a single (2,) point), gauged to zero at `gauge_point`.\n\n    Integrates grad p along an L-shaped path. Two orderings are available and\n    they must agree: path independence is a real consistency check on the\n    recovered field, since grad p is only a gradient if the flow actually\n    satisfies momentum (SPEC v2 S4, T-S4.2).\n\n    \"x_first\": gauge -> (target_x, gauge_y) -> target\n    \"y_first\": gauge -> (gauge_x, target_y) -> target\n    \"\"\"\n    gauge = np.asarray(gauge_point, dtype=np.float64)\n    target = np.asarray(target, dtype=np.float64)\n    if path == \"x_first\":\n        corner = np.array([target[0], gauge[1]])\n    elif path == \"y_first\":\n        corner = np.array([gauge[0], target[1]])\n    else:\n        raise ValueError(f\"unknown path {path!r}\")\n\n    total = 0.0\n    for start, end in ((gauge, corner), (corner, target)):\n        if not np.allclose(start, end):\n            total += _segment(\n                velocity_fn, start, end, reynolds, pressure_scaling, nodes\n            )\n    return total\n",
"src/predict.py": "\"\"\"Use a trained model.\n\nA finished run in `runs/<id>/` is a reusable solver, not just a folder of\nnumbers. This module loads one back and evaluates it anywhere in the channel.\n\n    from src.predict import load_run\n    solution = load_run(\"runs/caseA_re10\")\n    u, v = solution.velocity(3.0, 0.5)\n    y, u = solution.profile(x=3.0)\n\nor from a shell:\n\n    python -m src.predict --run runs/caseA_re10 --at 3.0,0.5\n    python -m src.predict --run runs/caseA_re10 --profile-at 3.0 --out profile.csv\n    python -m src.predict --run runs/caseA_re10 --grid 201 101 --out field.csv\n\nTwo things this deliberately refuses to do quietly:\n\n* It will not evaluate outside the trained domain. A PINN is fitted, not\n  solved, so outside `x* in [0, L*]` and `y* in [-1, 1]` it is extrapolating\n  and the residual it was trained on says nothing there. Pass\n  `allow_extrapolation=True` if you mean it.\n* It will not report a pressure the model does not have. The psi-omega system\n  has no pressure in it, so `pressure()` recovers one post hoc by path\n  integration and says so, rather than returning the placeholder zero that\n  lives in the field's third slot.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom jax.flatten_util import ravel_pytree\n\nfrom .config import Config, load_config\nfrom .geometry.domain import WALL\nfrom .models.build import build_model\nfrom .physics import streamfunction\nfrom .evaluation.metrics import cross_channel_flux\n\n\n@dataclass\nclass TrainedModel:\n    \"\"\"A trained solution, ready to evaluate.\"\"\"\n\n    config: Config\n    params: object\n    model: object\n    run_directory: Path\n\n    # -- introspection --------------------------------------------------------\n\n    @property\n    def domain_length(self) -> float:\n        return self.config.domain.length\n\n    def summary(self) -> str:\n        c = self.config\n        return (\n            f\"{c.name}: case {c.case}, Re_h = {c.physics.reynolds:g}, \"\n            f\"{c.physics.formulation}, L* = {c.domain.length:.3f}, \"\n            f\"pressure in system: {self.model.has_pressure}\"\n        )\n\n    # -- point evaluation -----------------------------------------------------\n\n    def _check_domain(self, x, y, allow_extrapolation):\n        if allow_extrapolation:\n            return\n        x = np.atleast_1d(np.asarray(x, dtype=np.float64))\n        y = np.atleast_1d(np.asarray(y, dtype=np.float64))\n        outside = (\n            (x < 0.0) | (x > self.domain_length) | (y < -WALL) | (y > WALL)\n        )\n        if np.any(outside):\n            first = int(np.argmax(outside))\n            raise ValueError(\n                f\"point (x*={x[first]:g}, y*={y[first]:g}) lies outside the \"\n                f\"trained domain x* in [0, {self.domain_length:g}], \"\n                f\"y* in [-1, 1]. A PINN is fitted, not solved: outside the \"\n                \"region where its residual was enforced it is extrapolating \"\n                \"and its output is not a solution of anything. Pass \"\n                \"allow_extrapolation=True if that is what you want.\"\n            )\n\n    def velocity(self, x, y, *, allow_extrapolation=False):\n        \"\"\"(u*, v*) at the given non-dimensional coordinates. Broadcasts.\"\"\"\n        self._check_domain(x, y, allow_extrapolation)\n        points = self._as_points(x, y)\n        out = jax.vmap(lambda p: self.model.velocity(self.params, p))(points)\n        u = np.asarray(out[:, 0]).reshape(np.shape(x) if np.ndim(x) else ())\n        v = np.asarray(out[:, 1]).reshape(np.shape(x) if np.ndim(x) else ())\n        return (float(u), float(v)) if u.ndim == 0 else (u, v)\n\n    def pressure(self, x, y, *, allow_extrapolation=False):\n        \"\"\"p* at the given coordinates, gauged to zero at the outlet centreline.\n\n        For the velocity-pressure formulation this is a network output. For the\n        psi-omega system pressure is absent from the solved system and is\n        recovered by integrating grad p along a path from the gauge point,\n        which is slower and carries its own consistency error -- the run's\n        `pressure_path_max_disagreement` metric measures it.\n        \"\"\"\n        self._check_domain(x, y, allow_extrapolation)\n        if self.model.has_pressure:\n            points = self._as_points(x, y)\n            out = jax.vmap(lambda p: self.model.field(self.params, p))(points)\n            p = np.asarray(out[:, 2]).reshape(np.shape(x) if np.ndim(x) else ())\n            return float(p) if p.ndim == 0 else p\n\n        velocity_fn = lambda xy: self.model.velocity(self.params, xy)  # noqa: E731\n        gauge = (self.domain_length, 0.0)\n        xs = np.atleast_1d(np.asarray(x, dtype=np.float64))\n        ys = np.atleast_1d(np.asarray(y, dtype=np.float64))\n        xs, ys = np.broadcast_arrays(xs, ys)\n        values = np.array([\n            streamfunction.recover_pressure(\n                velocity_fn, (float(a), float(b)), gauge,\n                self.config.physics.reynolds, self.config.physics.pressure_scaling,\n            )\n            for a, b in zip(xs.ravel(), ys.ravel())\n        ]).reshape(xs.shape)\n        return float(values) if np.ndim(x) == 0 and np.ndim(y) == 0 else values\n\n    def fields(self, x, y, *, allow_extrapolation=False) -> dict:\n        u, v = self.velocity(x, y, allow_extrapolation=allow_extrapolation)\n        return {\n            \"u\": u, \"v\": v,\n            \"p\": self.pressure(x, y, allow_extrapolation=allow_extrapolation),\n        }\n\n    # -- derived quantities ---------------------------------------------------\n\n    def profile(self, x, n=201):\n        \"\"\"Cross-channel velocity profile at one streamwise station.\"\"\"\n        y = np.linspace(-WALL, WALL, n)\n        u, v = self.velocity(np.full_like(y, float(x)), y)\n        return y, u, v\n\n    def flux(self, x):\n        \"\"\"Volumetric flux per unit depth through the station(s) x*.\"\"\"\n        stations = np.atleast_1d(np.asarray(x, dtype=np.float64))\n        values = cross_channel_flux(self.model.field, self.params, stations)\n        return float(values[0]) if np.ndim(x) == 0 else np.asarray(values)\n\n    def grid(self, nx=201, ny=101):\n        \"\"\"(x, y, u, v) on a regular grid covering the whole channel.\"\"\"\n        x = np.linspace(0.0, self.domain_length, nx)\n        y = np.linspace(-WALL, WALL, ny)\n        xx, yy = np.meshgrid(x, y, indexing=\"ij\")\n        u, v = self.velocity(xx.ravel(), yy.ravel())\n        return x, y, u.reshape(nx, ny), v.reshape(nx, ny)\n\n    # -- internals ------------------------------------------------------------\n\n    @staticmethod\n    def _as_points(x, y):\n        xs = np.atleast_1d(np.asarray(x, dtype=np.float64))\n        ys = np.atleast_1d(np.asarray(y, dtype=np.float64))\n        xs, ys = np.broadcast_arrays(xs, ys)\n        return jnp.asarray(np.stack([xs.ravel(), ys.ravel()], axis=-1))\n\n\ndef load_run(run_directory) -> TrainedModel:\n    \"\"\"Load a finished run back into an evaluable model.\n\n    The parameter pytree's STRUCTURE is rebuilt from the run's own config\n    rather than stored, so a run directory stays readable and version\n    independent. That makes it essential that the config travels with the\n    parameters -- which is why `run.py` copies it into the run directory.\n    \"\"\"\n    directory = Path(run_directory)\n    config_path = directory / \"config.yaml\"\n    params_path = directory / \"params.npz\"\n    for path in (config_path, params_path):\n        if not path.is_file():\n            raise FileNotFoundError(\n                f\"{path} not found. A usable run directory needs both \"\n                \"config.yaml and params.npz; has this run finished?\"\n            )\n\n    config = load_config(config_path)\n    template, model = build_model(config, jax.random.PRNGKey(config.seed))\n    flat_template, unflatten = ravel_pytree(template)\n\n    flat = np.load(params_path)[\"flat\"]\n    if flat.shape != flat_template.shape:\n        raise ValueError(\n            f\"{params_path} holds {flat.shape[0]} parameters but the config in \"\n            f\"{config_path} builds a model with {flat_template.shape[0]}. The \"\n            \"config and the saved weights are from different models.\"\n        )\n    return TrainedModel(config, unflatten(jnp.asarray(flat)), model, directory)\n\n\n# --------------------------------------------------------------------------- #\n# CLI\n# --------------------------------------------------------------------------- #\n\n\ndef _write_csv(path, header, rows):\n    with open(path, \"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(header)\n        writer.writerows(rows)\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=\"Evaluate a trained run.\")\n    parser.add_argument(\"--run\", required=True, help=\"a finished runs/<id> directory\")\n    parser.add_argument(\"--at\", help=\"a single point as 'x,y'\")\n    parser.add_argument(\"--profile-at\", type=float, help=\"cross-channel profile at x*\")\n    parser.add_argument(\"--flux-at\", type=float, help=\"flux through the station x*\")\n    parser.add_argument(\"--grid\", nargs=2, type=int, metavar=(\"NX\", \"NY\"),\n                        help=\"evaluate on an NX by NY grid\")\n    parser.add_argument(\"--out\", help=\"write CSV here instead of printing\")\n    parser.add_argument(\"--allow-extrapolation\", action=\"store_true\")\n    args = parser.parse_args(argv)\n\n    solution = load_run(args.run)\n    print(solution.summary())\n\n    if args.at:\n        x, y = (float(v) for v in args.at.split(\",\"))\n        f = solution.fields(x, y, allow_extrapolation=args.allow_extrapolation)\n        print(f\"  at (x*={x:g}, y*={y:g}):  u* = {f['u']:+.9f}  \"\n              f\"v* = {f['v']:+.9f}  p* = {f['p']:+.9f}\")\n\n    if args.profile_at is not None:\n        y, u, v = solution.profile(args.profile_at)\n        rows = list(zip(y, u, v))\n        if args.out:\n            _write_csv(args.out, [\"y\", \"u\", \"v\"], rows)\n            print(f\"  wrote {len(rows)} profile points to {args.out}\")\n        else:\n            print(f\"  profile at x* = {args.profile_at:g} (every 20th point):\")\n            for yy, uu, vv in rows[::20]:\n                print(f\"    y*={yy:+.4f}  u*={uu:+.9f}  v*={vv:+.9f}\")\n\n    if args.flux_at is not None:\n        print(f\"  flux at x* = {args.flux_at:g}: {solution.flux(args.flux_at):.12f}\")\n\n    if args.grid:\n        nx, ny = args.grid\n        x, y, u, v = solution.grid(nx, ny)\n        rows = [\n            (x[i], y[j], u[i, j], v[i, j]) for i in range(nx) for j in range(ny)\n        ]\n        if args.out:\n            _write_csv(args.out, [\"x\", \"y\", \"u\", \"v\"], rows)\n            print(f\"  wrote {len(rows)} grid points to {args.out}\")\n        else:\n            print(f\"  grid {nx}x{ny}: u* in [{u.min():.6f}, {u.max():.6f}], \"\n                  f\"max |v*| = {np.abs(v).max():.3e}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
"src/refit_drift.py": "\"\"\"Recompute the stored drift fit of a finished run from its traced D(n).\n\n`fit_drift` returned an unweighted least-squares fit until the residual\nweighting fix. On a problem whose residuals are O(1e-7) the cost at the\ninitial guess already sits below `curve_fit`'s default `ftol`, so on some\ntraces the optimiser returned `p0` untouched while reporting convergence, and\nthe run recorded a verdict drawn from a curve that never fitted anything.\n\nThe traced D(n) itself is unaffected -- it is measured, not fitted -- so the\ncorrection needs no retraining: refit the stored `drift_trace.csv` and rewrite\nonly the `diag_drift_*` keys. Everything else in `metrics.json` stands.\n\n    python -m src.refit_drift --run runs/caseA_re10\n    python -m src.refit_drift --all --write\n\nWithout `--write` this reports what would change and touches nothing.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport pathlib\n\nfrom .evaluation.drift import fit_drift\n\n#: Column holding D. Early runs wrote the generic `value` header.\nDRIFT_COLUMNS = (\"drift\", \"value\")\n\n\ndef read_trace(path):\n    \"\"\"Return (iterations, drifts) from a stored drift trace.\"\"\"\n    with path.open(encoding=\"utf-8\") as handle:\n        rows = list(csv.DictReader(handle))\n    if not rows:\n        raise ValueError(f\"{path} holds no rows\")\n    columns = rows[0].keys()\n    for candidate in DRIFT_COLUMNS:\n        if candidate in columns:\n            key = candidate\n            break\n    else:\n        raise ValueError(\n            f\"{path} has no drift column; looked for {DRIFT_COLUMNS}, got \"\n            f\"{sorted(columns)}\"\n        )\n    return (\n        [float(row[\"iteration\"]) for row in rows],\n        [float(row[key]) for row in rows],\n    )\n\n\ndef refit(directory, write=False):\n    \"\"\"Refit one run. Returns (before, after) verdict strings.\"\"\"\n    directory = pathlib.Path(directory)\n    trace_path = directory / \"drift_trace.csv\"\n    metrics_path = directory / \"metrics.json\"\n    if not trace_path.exists() or not metrics_path.exists():\n        return None\n    iterations, drifts = read_trace(trace_path)\n    if len(iterations) < 5:\n        return None\n\n    with metrics_path.open(encoding=\"utf-8\") as handle:\n        metrics = json.load(handle)\n    before = metrics.get(\"diag_drift_verdict\", \"(none recorded)\")\n\n    fit = fit_drift(iterations, drifts)\n    updated = {\n        \"diag_drift_d_infinity\": fit.d_infinity,\n        \"diag_drift_d_infinity_stderr\": fit.d_infinity_stderr,\n        \"diag_drift_exponent\": fit.exponent,\n        \"diag_drift_fit_points\": fit.points,\n        \"diag_drift_fit_converged\": fit.converged,\n        \"diag_drift_max_relative_residual\": fit.max_relative_residual,\n        \"diag_drift_moved_off_guess\": fit.moved_off_guess,\n        \"diag_drift_verdict\": fit.verdict(),\n        \"diag_drift_refitted\": (\n            \"diag_drift_* recomputed by src.refit_drift after the residual \"\n            \"weighting fix in evaluation.drift; the traced D(n) and every \"\n            \"other metric are from the original run\"\n        ),\n    }\n    if write:\n        metrics.update(updated)\n        with metrics_path.open(\"w\", encoding=\"utf-8\") as handle:\n            json.dump(metrics, handle, indent=2)\n    return before, updated[\"diag_drift_verdict\"], fit\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    group = parser.add_mutually_exclusive_group(required=True)\n    group.add_argument(\"--run\", help=\"a single run directory\")\n    group.add_argument(\"--all\", action=\"store_true\", help=\"every run under runs/\")\n    parser.add_argument(\n        \"--write\", action=\"store_true\",\n        help=\"rewrite metrics.json; without it nothing is touched\",\n    )\n    args = parser.parse_args(argv)\n\n    directories = (\n        sorted(pathlib.Path(\"runs\").iterdir()) if args.all\n        else [pathlib.Path(args.run)]\n    )\n    for directory in directories:\n        if not directory.is_dir():\n            continue\n        outcome = refit(directory, write=args.write)\n        if outcome is None:\n            print(f\"{directory.name}: no usable drift trace, skipped\")\n            continue\n        before, after, fit = outcome\n        print(f\"\\n{directory.name}\")\n        print(f\"  D_inf   {fit.d_infinity:.4e} +- {fit.d_infinity_stderr:.2e}  \"\n              f\"b={fit.exponent:.4f}  max rel residual {fit.max_relative_residual:.3f}\")\n        print(f\"  before: {before}\")\n        print(f\"  after : {after}\")\n    if not args.write:\n        print(\"\\n(dry run: pass --write to rewrite metrics.json)\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
"src/report.py": "\"\"\"Run directory -> accuracy table.\n\n    python -m src.report --run runs/<id>\n    python -m src.report --all              # every run under runs/\n\nReports relative L2 error against the analytic solution as the accuracy\nmeasure, and the loss separately and explicitly labelled as a diagnostic\n(CLAUDE.md: \"Loss value is a diagnostic, never an accuracy claim\").\n\nEach accuracy number is printed next to its SPEC target and a PASS/FAIL, so a\ntable cannot be read as a success without the reader seeing the target.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nRUNS_ROOT = Path(\"runs\")\n\n#: SPEC section 5 targets, duplicated here for reporting only. The tests are\n#: the gate; this module never decides whether something passed on its own\n#: terms, it just displays the same numbers.\nTARGETS = {\n    \"u_relative_l2\": (\"test 1  u relative L2\", 1e-5, \"<=\", \"A\"),\n    # SPEC test 2 is \"Transverse velocity, Case A\". Developing flow has a\n    # genuine non-zero v*, so applying this target to Case B would report a\n    # false failure for correct physics.\n    \"v_max_abs\": (\"test 2  max |v*|\", 1e-5, \"<=\", \"A\"),\n    \"flux_conservation_relative_error\": (\"test 3  flux conservation\", 1e-6, \"<=\", None),\n    \"outlet_profile_relative_l2\": (\"test 9  outlet profile L2\", 1e-4, \"<=\", \"B\"),\n    \"entrance_length_relative_error\": (\"test 9  entrance length\", 0.15, \"<=\", \"B\"),\n}\n\n\ndef load_metrics(directory: Path) -> dict:\n    path = directory / \"metrics.json\"\n    if not path.is_file():\n        raise FileNotFoundError(f\"{path} not found; has this run finished?\")\n    with path.open(encoding=\"utf-8\") as handle:\n        return json.load(handle)\n\n\ndef format_run(metrics: dict) -> str:\n    lines = []\n    name = metrics.get(\"name\", \"?\")\n    lines.append(f\"=== {name} ===\")\n    lines.append(\n        f\"  case {metrics['case']}   Re = {metrics['reynolds']:g}   \"\n        f\"{metrics['formulation']}   {metrics['pressure_scaling']} pressure   \"\n        f\"seed {metrics.get('seed')}\"\n    )\n\n    lines.append(\"\")\n    lines.append(\"  ACCURACY (relative L2 against the analytic solution)\")\n    any_target = False\n    for key, (label, target, _, only_case) in TARGETS.items():\n        if key not in metrics:\n            continue\n        if only_case is not None and metrics[\"case\"] != only_case:\n            continue\n        any_target = True\n        value = metrics[key]\n        verdict = \"PASS\" if value <= target else \"FAIL\"\n        lines.append(f\"    {label:<32} {value:10.3e}   target <= {target:<8.1e} {verdict}\")\n    if not any_target:\n        lines.append(\"    (no targeted metrics present)\")\n\n    if metrics[\"case\"] != \"A\":\n        lines.append(\n            f\"    {'max |v*| (no target: case B has':<32} {metrics['v_max_abs']:10.3e}\"\n        )\n        lines.append(f\"    {'  genuine transverse velocity)':<32}\")\n\n    for key, label in (\n        (\"p_relative_l2\", \"p relative L2 (gauged at outlet)\"),\n        (\"flux_mean\", \"mean flux\"),\n        (\"flux_relative_error_vs_inlet\", \"flux error vs inlet flux\"),\n        (\"outlet_profile_relative_l2_vs_nominal_flux\", \"outlet profile vs NOMINAL flux\"),\n        (\"inlet_flux_deficit_relative\", \"inlet flux deficit (corner artefact)\"),\n        (\"entrance_length_measured\", \"entrance length measured\"),\n        (\"entrance_length_correlation\", \"entrance length correlation\"),\n    ):\n        if key in metrics:\n            lines.append(f\"    {label:<32} {metrics[key]:10.3e}\")\n\n    lines.append(\"\")\n    lines.append(\"  DIAGNOSTICS -- these are NOT accuracy claims\")\n    for key, label in (\n        (\"diag_loss_after_adam\", \"loss after Adam\"),\n        (\"diag_loss_after_lbfgs\", \"loss after L-BFGS\"),\n        (\"diag_handoff_relative_gap\", \"Adam->L-BFGS loss discontinuity\"),\n        (\"residual_momentum_x_max\", \"max |momentum-x residual|\"),\n        (\"residual_momentum_y_max\", \"max |momentum-y residual|\"),\n        (\"residual_continuity_max\", \"max |continuity residual|\"),\n        (\"diag_parameter_count\", \"parameters\"),\n        (\"diag_lbfgs_iterations\", \"L-BFGS iterations\"),\n        (\"wall_seconds\", \"wall time (s)\"),\n    ):\n        if key in metrics:\n            value = metrics[key]\n            formatted = f\"{value:10.3e}\" if isinstance(value, float) else f\"{value:>10}\"\n            lines.append(f\"    {label:<32} {formatted}\")\n    if \"diag_lbfgs_message\" in metrics:\n        lines.append(f\"    {'L-BFGS termination':<32} {metrics['diag_lbfgs_message']}\")\n    return \"\\n\".join(lines)\n\n\ndef format_table(all_metrics: list) -> str:\n    header = (\n        f\"{'run':<28}{'case':>5}{'Re':>7}{'form':>18}\"\n        f\"{'u relL2':>11}{'max|v|':>11}{'flux cons':>11}{'loss':>11}\"\n    )\n    lines = [header, \"-\" * len(header)]\n    for metrics in all_metrics:\n        loss = metrics.get(\"diag_loss_after_lbfgs\", metrics.get(\"diag_loss_after_adam\"))\n        accuracy = metrics.get(\"u_relative_l2\", metrics.get(\"outlet_profile_relative_l2\"))\n        lines.append(\n            f\"{metrics.get('name', '?'):<28}{metrics['case']:>5}\"\n            f\"{metrics['reynolds']:>7g}{metrics['formulation']:>18}\"\n            f\"{accuracy:>11.2e}{metrics['v_max_abs']:>11.2e}\"\n            f\"{metrics['flux_conservation_relative_error']:>11.2e}\"\n            f\"{(loss if loss is not None else float('nan')):>11.2e}\"\n        )\n    lines.append(\"\")\n    lines.append(\n        \"u relL2 is against the analytic solution; for case B it is the outlet \"\n        \"profile against the flux-consistent parabola.\"\n    )\n    lines.append(\"loss is a DIAGNOSTIC and is never an accuracy claim.\")\n    return \"\\n\".join(lines)\n\n\ndef _write_falsifiability_matrix() -> int:\n    \"\"\"Regenerate the S5 matrix. Must be re-run on any ansatz change.\"\"\"\n    import yaml\n\n    from .config import parse_config\n    from .evaluation.falsifiability import (\n        all_vacuous_rows, build_matrix, degenerate_targets, format_matrix,\n    )\n\n    columns = {\n        \"vp-soft\": \"experiments/caseA_re10_baseline.yaml\",\n        \"vp-hard\": \"experiments/caseA_re10.yaml\",\n        \"psi-cubic\": \"experiments/caseA_re10_psiomega_cubic.yaml\",\n        \"psi-sin\": \"experiments/caseA_re10_psiomega_sinusoidal.yaml\",\n    }\n    configs = {\n        label: parse_config(yaml.safe_load(open(path, encoding=\"utf-8\")))\n        for label, path in columns.items()\n    }\n    matrix = build_matrix(configs)\n    table = format_matrix(matrix, list(columns))\n\n    document = [\n        \"# Falsifiability matrix (SPEC v2, S5)\",\n        \"\",\n        \"Generated by `python -m src.report --falsifiability`.\",\n        \"**Regenerate on every ansatz change** -- a stale matrix is the silent\",\n        \"disarming this exists to prevent, and `tests/test_sentinels.py` fails\",\n        \"if this file does not match the code.\",\n        \"\",\n        \"A check is classified by running it on an UNTRAINED, randomly\",\n        \"initialised network at five seeds:\",\n        \"\",\n        \"* `LIVE` - fails at every seed, so the check bites.\",\n        \"* `WEAKENED` - passes at some seeds, so it partly measures the ansatz.\",\n        \"* `VACUOUS` - passes at every seed, so it measures the ansatz and not\",\n        \"  the trained solution.\",\n        \"\",\n        \"Columns: `vp-*` velocity-pressure with the soft/hard pressure gauge;\",\n        \"`psi-*` the psi-omega system under the cubic/sinusoidal lift.\",\n        \"\",\n        table,\n        \"\",\n    ]\n    degenerate = {label: degenerate_targets(cfg) for label, cfg in configs.items()}\n    document += [\n        \"## Degenerate targets\",\n        \"\",\n        \"A check can be `LIVE` -- an untrained network fails it -- and still be\",\n        \"nearly free, because the ansatz places the correct answer at a single\",\n        \"point the optimiser can reach exactly. Sampling at the initialisation\",\n        \"distribution cannot see that, so this probes it directly by zeroing the\",\n        \"output layer, i.e. removing the network's contribution entirely.\",\n        \"\",\n        \"Checks that PASS with no network contribution at all:\",\n        \"\",\n    ]\n    for label in columns:\n        names = degenerate[label]\n        document.append(f\"* `{label}`: \" + (\", \".join(names) if names else \"none\"))\n    document += [\n        \"\",\n        \"Read `psi-cubic` carefully: essentially the whole Case A suite passes\",\n        \"with the network removed, because the cubic lift's derivative IS the\",\n        \"Case A solution. Its accuracy must never be quoted as a result.\",\n        \"\",\n        \"`test 2  max |v*|` is degenerate in EVERY column: the zero field passes\",\n        \"it. It cannot distinguish a correct solution on its own and is only\",\n        \"meaningful conjoined with test 1.\",\n        \"\",\n    ]\n\n    vacuous = all_vacuous_rows(matrix)\n    if vacuous:\n        document += [\n            \"## Untested properties\",\n            \"\",\n            \"These are `VACUOUS` in every column, so the suite does not test\",\n            \"them at all:\",\n            \"\",\n        ] + [f\"* {name}\" for name in vacuous] + [\"\"]\n    Path(\"docs/falsifiability-matrix.md\").write_text(\n        \"\\n\".join(document), encoding=\"utf-8\"\n    )\n    print(\"\\n\".join(document))\n    return 0\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    group = parser.add_mutually_exclusive_group(required=True)\n    group.add_argument(\"--run\", help=\"a single run directory, e.g. runs/caseA_re10\")\n    group.add_argument(\"--all\", action=\"store_true\", help=\"tabulate every run\")\n    group.add_argument(\n        \"--falsifiability\", action=\"store_true\",\n        help=\"regenerate docs/falsifiability-matrix.md (SPEC v2 S5)\",\n    )\n    parser.add_argument(\"--table\", action=\"store_true\", help=\"one-line-per-run table\")\n    args = parser.parse_args(argv)\n\n    if args.falsifiability:\n        return _write_falsifiability_matrix()\n\n    if args.run:\n        metrics = load_metrics(Path(args.run))\n        print(format_table([metrics]) if args.table else format_run(metrics))\n        return 0\n\n    directories = sorted(d for d in RUNS_ROOT.iterdir() if (d / \"metrics.json\").is_file())\n    if not directories:\n        print(f\"no finished runs found under {RUNS_ROOT}/\")\n        return 1\n    all_metrics = [load_metrics(d) for d in directories]\n    if args.table:\n        print(format_table(all_metrics))\n    else:\n        for metrics in all_metrics:\n            print(format_run(metrics))\n            print()\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
"src/run.py": "\"\"\"Config -> experiment.\n\n    python -m src.run --config experiments/<name>.yaml\n\nWrites everything a run produced into runs/<name>/. That directory is written\nby the code and never edited by hand (CLAUDE.md, \"Module layout\").\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport platform\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport jax\nimport numpy as np\nfrom jax.flatten_util import ravel_pytree\n\nfrom .config import Config, load_config\nfrom .evaluation.accuracy import evaluate\nfrom .evaluation import plots\nfrom .models.build import build_model\nfrom .training.loop import train\n\nRUNS_ROOT = Path(\"runs\")\n\n\ndef run_directory(config: Config, override: str | None) -> Path:\n    return Path(override) if override else RUNS_ROOT / config.name\n\n\ndef save_parameters(path: Path, params) -> None:\n    \"\"\"Store the flattened parameter vector.\n\n    The pytree structure is fully determined by the config, so `load_parameters`\n    rebuilds it rather than pickling it. That keeps run directories readable and\n    version-independent.\n    \"\"\"\n    flat, _ = ravel_pytree(params)\n    np.savez_compressed(path, flat=np.asarray(flat, dtype=np.float64))\n\n\ndef load_parameters(path: Path, config: Config):\n    template, model = build_model(config, jax.random.PRNGKey(config.seed))\n    _, unflatten = ravel_pytree(template)\n    flat = np.load(path)[\"flat\"]\n    return unflatten(flat), model\n\n\ndef write_history(path: Path, rows: list) -> None:\n    if not rows:\n        return\n    columns = sorted({key for row in rows for key in row})\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=columns)\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef execute(config_path: str, out: str | None = None, verbose: bool = True) -> Path:\n    config = load_config(config_path)\n    directory = run_directory(config, out)\n    directory.mkdir(parents=True, exist_ok=True)\n\n    if verbose:\n        print(f\"[{config.name}] case {config.case}, Re = {config.physics.reynolds}, \"\n              f\"{config.physics.formulation}, {config.physics.pressure_scaling} pressure\")\n        print(f\"[{config.name}] active loss terms: {', '.join(config.loss_terms)}\")\n\n    started = time.time()\n    result = train(config, verbose=verbose,\n                   checkpoint_path=directory / 'checkpoint.npz')\n    metrics = evaluate(config, result.model, result.params)\n    metrics[\"wall_seconds\"] = time.time() - started\n    metrics.update({f\"diag_{k}\": v for k, v in result.diagnostics.items()})\n    metrics[\"name\"] = config.name\n    metrics[\"seed\"] = config.seed\n\n    shutil.copyfile(config_path, directory / \"config.yaml\")\n    save_parameters(directory / \"params.npz\", result.params)\n    write_history(directory / \"history.csv\", result.history)\n    # T-S7.2: the D(n) trace is written for every run.\n    write_history(directory / \"drift_trace.csv\", result.drift_trace)\n    with (directory / \"lbfgs.json\").open(\"w\", encoding=\"utf-8\") as handle:\n        json.dump(result.lbfgs_history, handle, indent=2)\n    with (directory / \"metrics.json\").open(\"w\", encoding=\"utf-8\") as handle:\n        json.dump(metrics, handle, indent=2)\n    plots.write_all(config, result.model, result.params, directory)\n    with (directory / \"environment.json\").open(\"w\", encoding=\"utf-8\") as handle:\n        json.dump(\n            {\n                \"python\": sys.version,\n                \"platform\": platform.platform(),\n                \"jax\": jax.__version__,\n                \"jax_devices\": [str(d) for d in jax.devices()],\n                \"x64_enabled\": bool(jax.config.jax_enable_x64),\n            },\n            handle,\n            indent=2,\n        )\n\n    if verbose:\n        print(f\"[{config.name}] wrote {directory}\")\n        _print_summary(metrics)\n    return directory\n\n\ndef _show(metrics: dict, key: str, label: str, suffix: str = \"\") -> None:\n    \"\"\"Print one metric if present. Absence is normal, not an error.\n\n    Which metrics exist depends on the formulation and the case: the psi-omega\n    system has no pressure in it, and Case A has no entrance length. Printing\n    unconditionally is how this crashed every psi-omega run after its results\n    were already safely written.\n    \"\"\"\n    if key in metrics and metrics[key] is not None:\n        print(f\"  {label:<21}: {metrics[key]:.3e}{suffix}\")\n\n\ndef _print_summary(metrics: dict) -> None:\n    print(\"  --- accuracy (relative L2 against the analytic solution) ---\")\n    if metrics[\"case\"] == \"A\":\n        _show(metrics, \"u_relative_l2\", \"u relative L2\", \"   (target <= 1e-5)\")\n        _show(metrics, \"p_relative_l2\", \"p relative L2\")\n        _show(metrics, \"p_relative_l2_recovered\", \"p relative L2 (recovered)\")\n    else:\n        _show(metrics, \"outlet_profile_relative_l2\", \"outlet profile L2\",\n              \"  (target <= 1e-4)\")\n        if \"entrance_length_measured\" in metrics:\n            print(f\"  {'entrance length':<21}: {metrics['entrance_length_measured']:.4f} vs \"\n                  f\"{metrics['entrance_length_correlation']:.4f} correlation \"\n                  f\"({metrics['entrance_length_relative_error'] * 100:.1f}%, target <= 15%)\")\n    _show(metrics, \"v_max_abs\", \"max |v*|\",\n          \"   (target <= 1e-5)\" if metrics[\"case\"] == \"A\"\n          else \"   (no target: case B has genuine transverse velocity)\")\n    _show(metrics, \"flux_conservation_relative_error\", \"mass drift D\",\n          \"   (target <= 1e-6)\")\n    _show(metrics, \"flux_relative_error_vs_inlet\", \"flux vs inlet flux\")\n\n    print(\"  --- diagnostics (NOT accuracy claims) ---\")\n    final = metrics.get(\"diag_loss_after_lbfgs\", metrics.get(\"diag_loss_after_adam\"))\n    if final is not None:\n        print(f\"  {'final loss':<21}: {final:.3e}\")\n    _show(metrics, \"residual_momentum_x_max\", \"max momentum-x resid\")\n    _show(metrics, \"residual_vorticity_max\", \"max r1 resid\")\n    _show(metrics, \"residual_vorticity_transport_max\", \"max r2 resid\")\n    _show(metrics, \"pressure_path_max_disagreement\", \"pressure path disagree\")\n    # Guard at display time as well as at fit time: metrics.json is a snapshot\n    # written by whatever code produced the run, so a stored verdict may predate\n    # the machine-precision guard in drift.py.\n    drift = metrics.get(\"flux_conservation_relative_error\")\n    if drift is not None and drift < 1e-12:\n        print(f\"  {'drift':<21}: D is at machine precision -- this formulation \"\n              \"conserves mass by\")\n        print(f\"  {'':<21}  construction, so test 3 is VACUOUS here, not passed\")\n    elif metrics.get(\"diag_drift_verdict\"):\n        print(f\"  {'drift D_inf':<21}: {metrics['diag_drift_d_infinity']:.3e} \"\n              f\"+- {metrics['diag_drift_d_infinity_stderr']:.1e}\")\n        print(f\"  {'drift verdict':<21}: {metrics['diag_drift_verdict']}\")\n    print(f\"  {'wall time':<21}: {metrics['wall_seconds']:.1f} s\")\n\n\ndef main(argv=None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--config\", required=True, help=\"path to an experiments/*.yaml\")\n    parser.add_argument(\"--out\", default=None, help=\"override the run directory\")\n    parser.add_argument(\"--quiet\", action=\"store_true\")\n    args = parser.parse_args(argv)\n    execute(args.config, args.out, verbose=not args.quiet)\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
"src/training/__init__.py": "\"\"\"Optimisers, loss assembly, weighting, and the training loop.\"\"\"\n",
"src/training/loop.py": "\"\"\"The training loop: Adam with resampling, then L-BFGS on frozen points.\n\nDeterminism (SPEC test 8) is structural here: every random draw comes from a\nPRNG key derived from `config.seed` or `config.sampling.seed`, there is no\nwall-clock or hash-order dependence, and the L-BFGS phase is SciPy's\ndeterministic implementation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport time\nfrom pathlib import Path\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nimport optax\n\nfrom ..config import Config\nfrom ..geometry.domain import Channel\nfrom ..geometry.sampling import sample_all\nfrom ..models.build import build_model, count_parameters\nfrom .losses import make_loss_terms\nfrom .optimisers import adam_schedule, make_adam, run_lbfgs\nfrom ..evaluation.drift import fit_drift, mass_drift\nfrom ..evaluation.metrics import cross_channel_flux\nfrom .weights import initial_weights, make_grad_norm_update, grad_norm_weights, total_loss\n\n#: Target number of logged rows across the Adam phase. Logging cadence is\n#: derived from this rather than configured, because it changes nothing about\n#: the result -- only how densely the history is recorded.\nTARGET_LOG_ROWS = 200\n\n\n#: Points across the channel used by the live accuracy probe. Small on purpose:\n#: it runs at every log interval, and the reported accuracy at the END of a run\n#: is always recomputed on the full evaluation grid.\nPROBE_NX, PROBE_NY = 81, 41\n\n\ndef _make_accuracy_probe(config, model):\n    \"\"\"Relative L2 against the analytic solution, as a function of parameters.\n\n    This is the number that actually means something during training. The loss\n    is a diagnostic and can fall while this rises -- that is the failure mode\n    the whole project is built around -- so both are logged at every interval\n    and printed side by side.\n\n    Case A has an analytic solution everywhere. Case B does not, so the probe\n    reports the outlet profile against the flux-consistent developed parabola,\n    which is what SPEC test 9 measures.\n    \"\"\"\n    from ..geometry.sampling import evaluation_grid\n    from ..physics import analytic\n\n    channel = Channel(config.domain.length)\n    if config.case == \"A\":\n        points, _, _ = evaluation_grid(channel, PROBE_NX, PROBE_NY)\n        reference = analytic.u_nd(np.asarray(points[:, 1]))\n    else:\n        y = np.linspace(-1.0, 1.0, PROBE_NY * 4)\n        points = jnp.stack(\n            [jnp.full(len(y), config.domain.length), jnp.asarray(y)], axis=-1\n        )\n        reference = analytic.developed_profile_nd(y, config.inlet_flux)\n\n    norm = float(np.linalg.norm(reference))\n\n    def probe(params):\n        u = np.asarray(\n            jax.vmap(lambda xy: model.velocity(params, xy))(points)[:, 0]\n        )\n        return float(np.linalg.norm(u - reference) / norm)\n\n    return probe\n\n\ndef _make_drift_probe(config, model):\n    \"\"\"D on the fixed evaluation stations, as a function of parameters alone.\"\"\"\n    length = config.domain.length\n    stations = np.linspace(0.05 * length, 0.95 * length, 10)\n\n    def probe(params):\n        try:\n            return mass_drift(cross_channel_flux(model.field, params, stations, 64))\n        except (ValueError, FloatingPointError):\n            return float(\"nan\")\n\n    return probe\n\n\n@dataclass\nclass TrainingResult:\n    params: Any\n    model: Any\n    history: list = field(default_factory=list)\n    lbfgs_history: list = field(default_factory=list)\n    drift_trace: list = field(default_factory=list)\n    diagnostics: dict = field(default_factory=dict)\n\n\ndef train(config: Config, *, verbose: bool = True, checkpoint_path=None) -> TrainingResult:\n    \"\"\"Train a model.\n\n    `checkpoint_path`, when given, receives the current parameters at every\n    trace point. A long L-BFGS run is hours of compute whose result is\n    otherwise written only at the very end, so an interrupted machine loses all\n    of it; with a checkpoint the loss is bounded by one trace interval.\n    \"\"\"\n    channel = Channel(config.domain.length)\n\n    def save_checkpoint(p, label):\n        if checkpoint_path is None:\n            return\n        try:\n            from jax.flatten_util import ravel_pytree\n\n            flat, _ = ravel_pytree(p)\n            path = Path(checkpoint_path)\n            path.parent.mkdir(parents=True, exist_ok=True)\n            # Write beside the target then rename: a machine that dies during\n            # the write must not leave a truncated checkpoint behind.\n            temporary = path.with_suffix(\".tmp.npz\")\n            np.savez_compressed(temporary, flat=np.asarray(flat, dtype=np.float64),\n                                label=np.asarray(label))\n            temporary.replace(path)\n        except Exception:  # noqa: BLE001 - a failed checkpoint must not kill training\n            pass\n\n    model_key = jax.random.PRNGKey(config.seed)\n    params, model = build_model(config, model_key)\n\n    loss_terms_fn = make_loss_terms(config, model)\n    weights = initial_weights(config)\n\n    def weighted_loss(p, points, w):\n        return total_loss(loss_terms_fn(p, points), w)\n\n    optimiser = make_adam(config)\n    opt_state = optimiser.init(params)\n    schedule = adam_schedule(config)\n\n    @jax.jit\n    def adam_step(p, state, points, w):\n        def loss_and_terms(q):\n            # `has_aux` returns the per-term losses from the SAME trace that\n            # produced the gradient, instead of a second forward pass just to\n            # log them.\n            terms = loss_terms_fn(q, points)\n            return total_loss(terms, w), terms\n\n        (loss, terms), grads = jax.value_and_grad(loss_and_terms, has_aux=True)(p)\n        updates, state = optimiser.update(grads, state, p)\n        return optax.apply_updates(p, updates), state, loss, terms\n\n    term_grad_norms = None\n    if config.weights.scheme == \"grad_norm\":\n        term_grad_norms = jax.jit(make_grad_norm_update(loss_terms_fn))\n\n    # Held-out validation points. Drawn from a PRNG stream that the training\n    # sampler never reaches, so they are disjoint from every training draw.\n    validation_points = sample_all(\n        jax.random.PRNGKey(config.sampling.seed + 1_000_003),\n        channel,\n        config.sampling.n_validation,\n        config.sampling.n_inlet,\n        config.sampling.n_outlet,\n    )\n    validation_loss = jax.jit(\n        lambda p: total_loss(loss_terms_fn(p, validation_points), weights)\n    )\n\n    best = {\"validation\": float(\"inf\"), \"params\": params, \"where\": \"init\"}\n\n    def consider(p, where):\n        \"\"\"Keep the parameters with the lowest HELD-OUT loss seen so far.\"\"\"\n        value = float(validation_loss(p))\n        if value < best[\"validation\"]:\n            best.update(validation=value, params=jax.tree.map(lambda a: a, p), where=where)\n        return value\n\n    sample_key = jax.random.PRNGKey(config.sampling.seed)\n    points = sample_all(\n        sample_key,\n        channel,\n        config.sampling.n_interior,\n        config.sampling.n_inlet,\n        config.sampling.n_outlet,\n    )\n\n    history = []\n    log_every = max(1, config.training.adam.steps // TARGET_LOG_ROWS)\n    accuracy_probe = _make_accuracy_probe(config, model)\n    resample_every = config.sampling.resample_every\n    started = time.time()\n    if verbose:\n        print(\n            f\"  [adam] {config.training.adam.steps} steps, resampling every \"\n            f\"{resample_every} (one 'epoch' = one resampling interval, \"\n            f\"{config.training.adam.steps // max(resample_every, 1)} of them)\",\n            flush=True,\n        )\n        print(f\"  {'step':>8} {'epoch':>6} {'train loss':>12} {'held-out':>12} \"\n              f\"{'rel L2 vs exact':>16} {'lr':>9}\", flush=True)\n\n    for step in range(config.training.adam.steps):\n        if step > 0 and step % config.sampling.resample_every == 0:\n            sample_key, draw_key = jax.random.split(sample_key)\n            points = sample_all(\n                draw_key,\n                channel,\n                config.sampling.n_interior,\n                config.sampling.n_inlet,\n                config.sampling.n_outlet,\n            )\n\n        if (\n            term_grad_norms is not None\n            and step % config.weights.grad_norm.update_every == 0\n        ):\n            weights = grad_norm_weights(\n                term_grad_norms(params, points), weights, config.weights.grad_norm.alpha\n            )\n\n        params, opt_state, loss, terms = adam_step(params, opt_state, points, weights)\n\n        if step % log_every == 0 or step == config.training.adam.steps - 1:\n            accuracy = accuracy_probe(params)\n            held_out = consider(params, f\"adam:{step}\")\n            row = {\n                \"phase\": \"adam\",\n                \"step\": step,\n                \"epoch\": step // max(resample_every, 1),\n                \"loss\": float(loss),\n                \"validation_loss\": held_out,\n                \"relative_l2_vs_exact\": accuracy,\n                \"lr\": float(schedule(step)),\n                **{f\"term_{k}\": float(v) for k, v in terms.items()},\n                **{f\"weight_{k}\": float(v) for k, v in weights.items()},\n            }\n            history.append(row)\n            if verbose and (step % (log_every * 10) == 0):\n                print(\n                    f\"  {step:>8d} {step // max(resample_every, 1):>6d} \"\n                    f\"{float(loss):>12.4e} {held_out:>12.4e} {accuracy:>16.4e} \"\n                    f\"{float(schedule(step)):>9.2e}\",\n                    flush=True,\n                )\n\n    adam_seconds = time.time() - started\n\n    diagnostics = {\n        \"parameter_count\": count_parameters(params),\n        \"adam_seconds\": adam_seconds,\n        \"final_weights\": {k: float(v) for k, v in weights.items()},\n    }\n\n    # -- L-BFGS on FROZEN points -------------------------------------------- #\n    lbfgs_history = []\n    drift_trace = []\n    if config.training.lbfgs.enabled:\n        frozen = points\n        loss_before = float(weighted_loss(params, frozen, weights))\n\n        def objective(p):\n            return weighted_loss(p, frozen, weights)\n\n        # SPEC v2 S7: trace the mass drift D against L-BFGS iteration. Measured\n        # on the FIXED evaluation stations, never on the training collocation\n        # points (T-S7.1) -- the frozen set above is the training set.\n        drift_fn = _make_drift_probe(config, model)\n\n        def trace_fn(p):\n            values = {\n                \"drift\": drift_fn(p),\n                \"relative_l2_vs_exact\": accuracy_probe(p),\n                \"validation_loss\": consider(p, \"lbfgs\"),\n            }\n            save_checkpoint(p, \"lbfgs\")\n            return values\n\n        if verbose:\n            print(\n                f\"  [lbfgs] up to {config.training.lbfgs.max_iterations} iterations\",\n                flush=True,\n            )\n            print(f\"  {'iter':>8} {'':>6} {'train loss':>12} {'held-out':>12} \"\n                  f\"{'rel L2 vs exact':>16} {'drift D':>9}\", flush=True)\n\n        def on_trace(n, values, loss):\n            if verbose:\n                print(\n                    f\"  {n:>8d} {'':>6} {loss:>12.4e} \"\n                    f\"{values['validation_loss']:>12.4e} \"\n                    f\"{values['relative_l2_vs_exact']:>16.4e} {values['drift']:>9.2e}\",\n                    flush=True,\n                )\n\n        started = time.time()\n        result = run_lbfgs(\n            objective,\n            params,\n            config.training.lbfgs.max_iterations,\n            config.training.lbfgs.gtol,\n            trace_fn=trace_fn,\n            on_trace=on_trace,\n        )\n        lbfgs_seconds = time.time() - started\n\n        # SPEC section 9: verify the loss is continuous across the handoff. The\n        # first L-BFGS evaluation is at the Adam parameters on the same frozen\n        # points, so any difference is a bug in the handoff, not optimisation.\n        handoff_gap = abs(result.initial_loss - loss_before)\n        handoff_relative = handoff_gap / max(abs(loss_before), 1e-300)\n\n        final_validation = consider(result.params, \"lbfgs:final\")\n        params = result.params\n        diagnostics[\"validation_loss_final\"] = final_validation\n        lbfgs_history.append(\n            {\n                \"phase\": \"lbfgs\",\n                \"loss_at_handoff_adam_side\": loss_before,\n                \"loss_at_handoff_lbfgs_side\": result.initial_loss,\n                \"handoff_absolute_gap\": handoff_gap,\n                \"handoff_relative_gap\": handoff_relative,\n                \"final_loss\": result.final_loss,\n                \"iterations\": result.iterations,\n                \"function_evaluations\": result.function_evaluations,\n                \"status\": result.status,\n                \"message\": result.message,\n            }\n        )\n        drift_trace = result.trace\n        if drift_trace:\n            fit = fit_drift(\n                [row[\"iteration\"] for row in drift_trace],\n                [row[\"drift\"] for row in drift_trace],\n            )\n            diagnostics.update(\n                {\n                    \"drift_d_infinity\": fit.d_infinity,\n                    \"drift_d_infinity_stderr\": fit.d_infinity_stderr,\n                    \"drift_exponent\": fit.exponent,\n                    \"drift_fit_points\": fit.points,\n                    \"drift_fit_converged\": fit.converged,\n                    \"drift_verdict\": fit.verdict(),\n                }\n            )\n\n        diagnostics.update(\n            {\n                \"lbfgs_seconds\": lbfgs_seconds,\n                \"lbfgs_iterations\": result.iterations,\n                \"lbfgs_status\": result.status,\n                \"lbfgs_message\": result.message,\n                \"handoff_relative_gap\": handoff_relative,\n                \"loss_after_adam\": loss_before,\n                \"loss_after_lbfgs\": result.final_loss,\n            }\n        )\n        if verbose:\n            print(\n                f\"  lbfgs {result.iterations} iters, \"\n                f\"loss {loss_before:.6e} -> {result.final_loss:.6e}, \"\n                f\"handoff gap {handoff_relative:.2e} ({result.message})\",\n                flush=True,\n            )\n    else:\n        diagnostics[\"loss_after_adam\"] = float(weighted_loss(params, points, weights))\n\n    # SPEC v2 does not cover this; it is the fix for the loss/accuracy\n    # divergence. Selection uses the HELD-OUT loss, never the analytic\n    # solution, so the same mechanism works in Phase 2.\n    diagnostics[\"validation_loss_best\"] = best[\"validation\"]\n    diagnostics[\"selected_from\"] = best[\"where\"]\n    diagnostics[\"selection_rule\"] = config.training.select\n    if config.training.select == \"best_validation\":\n        final_value = diagnostics.get(\"validation_loss_final\")\n        improved = final_value is None or best[\"validation\"] < final_value\n        if verbose and best[\"where\"] != \"lbfgs:final\" and improved:\n            print(\n                f\"  selected parameters from {best['where']} \"\n                f\"(held-out loss {best['validation']:.4e}) rather than the final \"\n                f\"iterate ({diagnostics.get('validation_loss_final', float('nan')):.4e})\",\n                flush=True,\n            )\n        params = best[\"params\"]\n\n    return TrainingResult(\n        drift_trace=drift_trace if config.training.lbfgs.enabled else [],\n        params=params,\n        model=model,\n        history=history,\n        lbfgs_history=lbfgs_history,\n        diagnostics=diagnostics,\n    )\n",
"src/training/losses.py": "\"\"\"Per-term losses. Unweighted.\n\nEach term is the mean squared residual of one governing equation or one\nboundary condition, evaluated on the collocation points for that term. Nothing\nhere multiplies by a weight -- that happens in `weights.py` and nowhere else\n(CLAUDE.md, \"Code style\").\n\nThere is deliberately no wall term. No-slip is exact by construction in the\nansatz, so a wall loss would be identically zero and would only dilute the\ngradient of the terms that matter.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax.numpy as jnp\n\nfrom ..config import Config\nfrom ..physics import residuals, streamfunction\nfrom ..physics.derivatives import (\n    batched_field_and_jacobian,\n    batched_field_derivatives,\n    batched_stream_derivatives,\n)\n\n\ndef inlet_profile(config: Config, y):\n    \"\"\"Prescribed streamwise velocity at x* = 0.\n\n    Case A: the fully developed parabola u* = 1 - y*^2.\n    Case B: the uniform profile u* = `physics.inlet_uniform_value`.\n    \"\"\"\n    if config.case == \"A\":\n        return 1.0 - y**2\n    return jnp.full_like(y, config.inlet_uniform_value)\n\n\ndef make_loss_terms(config: Config, model):\n    \"\"\"Build `loss_terms(params, points) -> dict[str, scalar]`.\n\n    The returned dict has exactly the keys in `config.loss_terms`, so a term\n    can never be silently dropped or silently added.\n    \"\"\"\n    reynolds = config.physics.reynolds\n    pressure_scaling = config.physics.pressure_scaling\n    formulation = config.physics.formulation\n\n    def loss_terms(params, points):\n        point_fn = lambda xy: model.field(params, xy)  # noqa: E731\n\n        if formulation == \"psi_omega\":\n            stream_point_fn = lambda xy: model.stream(params, xy)  # noqa: E731\n            sd = batched_stream_derivatives(stream_point_fn, points.interior)\n            r1, r2 = streamfunction.residuals(sd, reynolds)\n            terms = {\n                \"vorticity\": jnp.mean(r1**2),\n                \"vorticity_transport\": jnp.mean(r2**2),\n            }\n        else:\n            d = batched_field_derivatives(point_fn, points.interior)\n            terms = {\n                \"momentum_x\": jnp.mean(\n                    residuals.momentum_x(d, reynolds, pressure_scaling) ** 2\n                ),\n                \"momentum_y\": jnp.mean(\n                    residuals.momentum_y(d, reynolds, pressure_scaling) ** 2\n                ),\n                \"continuity\": jnp.mean(residuals.continuity(d) ** 2),\n            }\n\n        if config.bc.inlet == \"soft\":\n            inlet_values, _ = batched_field_and_jacobian(point_fn, points.inlet)\n            target = inlet_profile(config, points.inlet[:, 1])\n            terms[\"inlet\"] = jnp.mean((inlet_values[:, 0] - target) ** 2) + jnp.mean(\n                inlet_values[:, 1] ** 2\n            )\n\n        needs_outlet = (\n            config.bc.outlet_velocity == \"soft\"\n            or config.bc.pressure_gauge == \"soft\"\n        )\n        if needs_outlet:\n            outlet_values, outlet_jac = batched_field_and_jacobian(\n                point_fn, points.outlet\n            )\n            if config.bc.outlet_velocity == \"soft\":\n                # Zero streamwise gradient of velocity: du/dx = dv/dx = 0.\n                terms[\"outlet_velocity\"] = jnp.mean(\n                    outlet_jac[:, 0, 0] ** 2\n                ) + jnp.mean(outlet_jac[:, 1, 0] ** 2)\n            if config.bc.pressure_gauge == \"soft\":\n                # The gauge fix. Without it the pressure error is meaningless.\n                terms[\"pressure_gauge\"] = jnp.mean(outlet_values[:, 2] ** 2)\n\n        return terms\n\n    return loss_terms\n",
"src/training/optimisers.py": "\"\"\"Optimiser schedule: Adam with exponential decay, then L-BFGS.\n\nThe handoff between the two is the delicate part. SPEC section 9 lists it as a\nknown trap: \"Adam -> L-BFGS handoff frequently resets or mishandles state.\nVerify the loss is continuous across the switch.\" Two things are done about it\nhere:\n\n1. The collocation points are FROZEN for the whole L-BFGS phase. L-BFGS builds\n   a curvature estimate from a sequence of gradients; resampling underneath it\n   changes the objective between evaluations and the curvature estimate becomes\n   meaningless. (This also means the loss values reported during L-BFGS are on\n   a different point set from the Adam ones, which is why accuracy is always\n   measured on the fixed evaluation grid instead.)\n\n2. `run_lbfgs` returns the loss at its own first function evaluation, so the\n   caller can compare it against the last Adam loss ON THE SAME POINTS and\n   record the discontinuity. It should be zero to rounding.\n\nL-BFGS itself is SciPy's, driven on a flattened float64 parameter vector.\nSciPy's implementation is the well-tested Nocedal Fortran code and it is\ndeterministic, which SPEC test 8 depends on.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Callable, NamedTuple\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nimport optax\nfrom jax.flatten_util import ravel_pytree\nfrom scipy.optimize import minimize\n\n\ndef adam_schedule(config):\n    \"\"\"Exponentially decaying learning rate, as configured.\"\"\"\n    return optax.exponential_decay(\n        init_value=config.training.adam.lr_init,\n        transition_steps=config.training.adam.decay_steps,\n        decay_rate=config.training.adam.decay_rate,\n        staircase=False,\n    )\n\n\ndef make_adam(config):\n    return optax.adam(learning_rate=adam_schedule(config))\n\n\nclass LBFGSResult(NamedTuple):\n    params: object\n    initial_loss: float\n    final_loss: float\n    iterations: int\n    function_evaluations: int\n    status: int\n    message: str\n    trace: list = []\n\n\ndef run_lbfgs(\n    loss_fn: Callable, params, max_iterations: int, gtol: float,\n    *, trace_fn=None, trace_every: int | None = None, on_trace=None,\n):\n    \"\"\"Minimise `loss_fn(params)` with SciPy L-BFGS-B on a flat float64 vector.\n\n    `loss_fn` must be a pure function of the parameters alone; bind the\n    collocation points before calling.\n\n    `trace_fn(params) -> float` is sampled every `trace_every` iterations and\n    recorded against the iteration count. SPEC v2 S7 uses this to trace the\n    mass drift D against L-BFGS iteration and fit D(n) = D_inf + a n^-b, which\n    is what discriminates formulation bias from under-convergence.\n    \"\"\"\n    flat_initial, unflatten = ravel_pytree(params)\n\n    value_and_grad = jax.jit(jax.value_and_grad(loss_fn))\n\n    evaluations = {\"count\": 0, \"first_loss\": None}\n\n    def objective(flat):\n        value, grad = value_and_grad(unflatten(jnp.asarray(flat)))\n        evaluations[\"count\"] += 1\n        if evaluations[\"first_loss\"] is None:\n            evaluations[\"first_loss\"] = float(value)\n        flat_grad, _ = ravel_pytree(grad)\n        # SciPy needs contiguous float64 numpy arrays, not JAX arrays.\n        return float(value), np.asarray(flat_grad, dtype=np.float64)\n\n    trace = []\n    if trace_fn is not None:\n        if trace_every is None:\n            trace_every = max(1, max_iterations // 40)\n        state = {\"iteration\": 0}\n\n        def callback(flat):\n            state[\"iteration\"] += 1\n            n = state[\"iteration\"]\n            if n % trace_every == 0 or n == 1:\n                values = trace_fn(unflatten(jnp.asarray(flat)))\n                if not isinstance(values, dict):\n                    values = {\"value\": float(values)}\n                trace.append({\"iteration\": n, **values})\n                if on_trace is not None:\n                    on_trace(n, values, float(value_and_grad(unflatten(jnp.asarray(flat)))[0]))\n    else:\n        callback = None\n\n    result = minimize(\n        objective,\n        np.asarray(flat_initial, dtype=np.float64),\n        jac=True,\n        method=\"L-BFGS-B\",\n        callback=callback,\n        options={\n            \"maxiter\": max_iterations,\n            \"maxfun\": max_iterations * 2,\n            \"gtol\": gtol,\n            # ftol = 0 so termination is governed by gtol alone; the default\n            # ftol stops at ~1e-9 relative change, far short of the accuracy\n            # this project targets.\n            \"ftol\": 0.0,\n            \"maxcor\": 100,\n        },\n    )\n\n    return LBFGSResult(\n        params=unflatten(jnp.asarray(result.x)),\n        initial_loss=evaluations[\"first_loss\"],\n        final_loss=float(result.fun),\n        iterations=int(result.nit),\n        function_evaluations=int(evaluations[\"count\"]),\n        status=int(result.status),\n        message=str(result.message),\n        trace=trace,\n    )\n",
"src/training/weights.py": "\"\"\"The ONLY place loss weighting happens.\n\nCLAUDE.md: \"Residual functions return unweighted residuals. All loss weighting\nhappens in one place (src/training/weights.py).\" If you find yourself wanting\nto scale a residual anywhere else, the scaling belongs here instead.\n\nTwo schemes:\n\nfixed\n    The weights from the config, held constant. Start here.\n\ngrad_norm\n    Learning-rate annealing of the weights (Wang, Teng & Perdikaris 2021). Every\n    `update_every` steps:\n\n        w_hat_i = (sum_j ||grad_theta L_j||) / ||grad_theta L_i||\n        w_i     <- alpha * w_i + (1 - alpha) * w_hat_i\n\n    which equalises the gradient contribution of each term. The config weights\n    are the starting values.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jnp\n\n\ndef initial_weights(config):\n    \"\"\"Starting weights, ordered exactly as `config.loss_terms`.\"\"\"\n    return {term: jnp.asarray(config.weights.values[term], dtype=jnp.float64)\n            for term in config.loss_terms}\n\n\ndef total_loss(terms, weights):\n    \"\"\"Weighted sum. The single point where weights meet residuals.\"\"\"\n    missing = set(weights) ^ set(terms)\n    if missing:\n        raise KeyError(\n            f\"loss terms and weights disagree on {sorted(missing)}; \"\n            \"this means the config's active-term list and the loss builder \"\n            \"have drifted apart\"\n        )\n    return sum(weights[name] * terms[name] for name in sorted(terms))\n\n\ndef grad_norm_weights(term_grad_norms, previous, alpha):\n    \"\"\"One grad-norm annealing update.\n\n    `term_grad_norms` maps term name -> ||grad_theta L_term||_2 (the norm of the\n    gradient of the UNWEIGHTED term). A term whose gradient norm is zero (e.g. a\n    boundary condition already satisfied exactly) keeps its previous weight\n    rather than producing an infinity.\n    \"\"\"\n    total = sum(term_grad_norms.values())\n    updated = {}\n    for name, norm in term_grad_norms.items():\n        target = jnp.where(norm > 0.0, total / jnp.where(norm > 0.0, norm, 1.0),\n                           previous[name])\n        updated[name] = alpha * previous[name] + (1.0 - alpha) * target\n    return updated\n\n\ndef make_grad_norm_update(loss_terms_fn):\n    \"\"\"Build a jittable function computing per-term gradient norms.\"\"\"\n\n    def term_grad_norms(params, points):\n        norms = {}\n        for name in loss_terms_fn(params, points):\n            grad = jax.grad(lambda p, n=name: loss_terms_fn(p, points)[n])(params)\n            leaves = jax.tree.leaves(grad)\n            norms[name] = jnp.sqrt(sum(jnp.sum(leaf**2) for leaf in leaves))\n        return norms\n\n    return term_grad_norms\n"
}''')

for relative, text in SOURCES.items():
    path = pathlib.Path(relative)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

print(f"wrote {len(SOURCES)} source files")

In [ ]:
!pip install -q optax pyyaml

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
print("jax", jax.__version__, "| devices", jax.devices(),
      "| float64 active:", jnp.zeros(1).dtype == jnp.float64)

## 2. Verify the physics is correct

These check the residual operators against hand-derived forcings, the autodiff
against fourth-order finite differences, and the scaling round trip. **No
training.** If any of these fail, do not trust anything below.

In [ ]:
import numpy as np, jax.numpy as jnp
from src.physics import analytic, residuals
from src.physics.derivatives import batched_field_derivatives
from src.physics.conventions import re_h_to_re_dh, entrance_length_over_h, EntranceLengthCorrelation

# The analytic solution must satisfy the residual operator exactly.
Re, L = 10.0, 7.824
field = lambda xy: jnp.array([1.0 - xy[1]**2, 0.0, -2.0/Re*(xy[0] - L)])
pts = jnp.asarray(np.random.default_rng(0).uniform([0,-1],[L,1], size=(512,2)))
r_x, r_y, r_c = residuals.navier_stokes(batched_field_derivatives(field, pts), Re, "inertial")
print(f"analytic solution residual:  max|R| = {float(jnp.max(jnp.abs(jnp.stack([r_x,r_y,r_c])))):.2e}   (want ~0)")

# Reynolds conventions: Re_Dh = (8/3) Re_h exactly.
print(f"Re_Dh / Re_h = {re_h_to_re_dh(1.0)!r}   (want {8/3!r})")

# Flux of the reference profile is exactly 4/3.
nodes, weights = np.polynomial.legendre.leggauss(200)
print(f"flux of 1-y^2  = {float(analytic.u_nd(nodes) @ weights):.15f}   (want {4/3:.15f})")

## 3. Configure

`L*` is **derived**, not chosen: it comes from the entrance-length correlation
so the domain is always long enough for the Reynolds number.

Reduce `steps` / `max_iterations` for a quick look; raise them for accuracy.

In [ ]:
import yaml
from src.config import parse_config

CONFIG = yaml.safe_load(r'''
name: colab_caseA_re10
seed: 0
case: A

domain: {alpha: 1.5, outlet_margin: 5.0}

correlation:
  c0: 0.3155
  c1: 0.01105
  n: 1.6
  citation: >
    Durst, Ray, Unsal & Bayoumi (2005), J. Fluids Eng. 127:1154-1160,
    plane-channel fit C0=0.631, C1=0.0442, n=1.6 in full-height units;
    converted to hydraulic diameter D_h = 2H via c0 = C0/2, c1 = C1/4.

physics:
  reynolds: 10.0
  pressure_scaling: inertial
  formulation: velocity_pressure

model:
  arch: modified_mlp
  width: 128
  depth: 4
  fourier: {enabled: false}
  rwf: {enabled: false}

bc: {inlet: soft, outlet_velocity: soft, pressure_gauge: hard}

sampling:
  n_interior: 2048
  n_validation: 2048
  n_inlet: 128
  n_outlet: 128
  resample_every: 500
  seed: 1

weights:
  scheme: fixed
  values: {momentum_x: 1.0, momentum_y: 1.0, continuity: 1.0, inlet: 1.0, outlet_velocity: 1.0}

training:
  select: best_validation
  adam:  {steps: 2000, lr_init: 1.0e-3, decay_rate: 0.9, decay_steps: 1000}
  lbfgs: {enabled: true, max_iterations: 6000, gtol: 1.0e-14}

evaluation: {nx: 401, ny: 201}
''')

config = parse_config(CONFIG)
print(f"{config.name}: case {config.case}, Re_h = {config.physics.reynolds:g}")
print(f"  entrance length L_e/h = {config.domain.entrance_length_over_h:.3f}")
print(f"  derived domain L*     = {config.domain.length:.3f}")

## 4. Train

Watch **`train loss` against `held-out` and `rel L2 vs exact`**. The training
loss is measured on the points being optimised; the held-out loss is measured
on points the optimiser never sees. When they separate, the model is fitting
its collocation sample rather than the equations — and `select:
best_validation` keeps the best held-out iterate rather than the last one.

In [ ]:
from src.training.loop import train
from src.evaluation.accuracy import evaluate

result = train(config, verbose=True)
metrics = evaluate(config, result.model, result.params)

## 5. Results against the analytic solution

In [ ]:
print(f"test 1  u relative L2 : {metrics['u_relative_l2']:.4e}   target <= 1e-5 (stretch 1e-6)")
print(f"test 2  max |v*|      : {metrics['v_max_abs']:.4e}   target <= 1e-5")
print(f"test 3  mass drift D  : {metrics['flux_conservation_relative_error']:.4e}   target <= 1e-6")
if "p_relative_l2" in metrics:
    print(f"        p relative L2 : {metrics['p_relative_l2']:.4e}")
print()
print("DIAGNOSTICS (not accuracy claims)")
print(f"  final loss          : {result.diagnostics.get('loss_after_lbfgs'):.4e}")
print(f"  held-out loss best  : {result.diagnostics.get('validation_loss_best'):.4e}")
print(f"  parameters selected : {result.diagnostics.get('selected_from')}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.geometry.domain import Channel
from src.geometry.sampling import evaluation_grid
from src.evaluation.metrics import evaluate_fields

points, x, y = evaluation_grid(Channel(config.domain.length), 201, 101)
u, v, p = evaluate_fields(result.model.field, result.params, points)
u = u.reshape(201, 101)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i in np.linspace(0, 200, 5).astype(int):
    ax[0].plot(u[i], y, alpha=0.8)
ax[0].plot(analytic.u_nd(np.asarray(y)), y, "k--", lw=2, label="analytic 1 - y*²")
ax[0].set_xlabel("u*"); ax[0].set_ylabel("y*"); ax[0].legend(); ax[0].set_title("profiles")

err = np.abs(u - analytic.u_nd(np.asarray(y))[None, :])
im = ax[1].pcolormesh(x, y, err.T, shading="auto")
fig.colorbar(im, ax=ax[1]); ax[1].set_title("|u* - exact|"); ax[1].set_xlabel("x*")

rows = [r for r in result.history]
ax[2].semilogy([r["step"] for r in rows], [r["loss"] for r in rows], label="train loss")
ax[2].semilogy([r["step"] for r in rows], [r["validation_loss"] for r in rows], label="held-out loss")
ax[2].semilogy([r["step"] for r in rows], [r["relative_l2_vs_exact"] for r in rows], label="rel L2 vs exact")
ax[2].set_xlabel("Adam step"); ax[2].legend(); ax[2].set_title("loss is not accuracy")
plt.tight_layout(); plt.show()

## 6. Use the trained model

`solution` evaluates the trained network anywhere in the channel. It refuses to
evaluate outside the trained domain: a PINN is fitted, not solved, and outside
the region where its residual was enforced it is not solving anything.

In [ ]:
from src.predict import TrainedModel

solution = TrainedModel(config, result.params, result.model, run_directory=None)
print(solution.summary())

u_pt, v_pt = solution.velocity(3.0, 0.5)
print(f"  u*(3.0, 0.5) = {u_pt:+.9f}   exact {1 - 0.5**2:+.9f}")
print(f"  v*(3.0, 0.5) = {v_pt:+.9f}   exact +0.000000000")
print(f"  flux at x*=3 = {solution.flux(3.0):.12f}   exact {4/3:.12f}")

try:
    solution.velocity(config.domain.length * 2, 0.0)
except ValueError as error:
    print(f"\n  refused, correctly: {error}")

## 7. Save the trained model

Colab discards the VM when it disconnects. Mount Drive to keep the weights.

In [ ]:
import numpy as np
from jax.flatten_util import ravel_pytree

flat, _ = ravel_pytree(result.params)
np.savez_compressed("trained_params.npz", flat=np.asarray(flat, dtype=np.float64))
print(f"saved {flat.size} parameters to trained_params.npz")

# from google.colab import drive; drive.mount('/content/drive')
# !cp trained_params.npz /content/drive/MyDrive/